# Concentration Beyond Injectivity with Magnitude Modulated Spectral Kernels for Linear Attention -All models


In [ ]:
# =============================================================================
#  InLine-DS — Concentration Beyond Injectivity with Magnitude Modulated Spectral Kernels for Linear Attention
# =============================================================================
#  1. This code implements InLine-DS using Magnitude Modulated Spectral Kernels (MMSK).
#       The MMSK is complemented with dual-frequency, injectivity-preserving operator
#       with ViT-grade stabilization. PLOTS: attention proxy maps,
#       rollout, GMAR, flow graph, parallel coordinates, head redundancy,
#       locality heatmap, pruning, layer attribution, dropout sensitivity,
#       calibration, top-k, full JSON/CSV/TXT manifest.
#
# =============================================================================
#  Supports four InLine-DS backbones with a single switch (MODEL_TYPE):
#     * inline_deit_ds_*   (isotropic ViT/DeiT)
#     * inline_swin_ds_*   (hierarchical, windowed)
#     * inline_pvt_ds_*    (pyramid, spatial-reduction)
#     * inline_cswin_ds_*  (cross-shaped windows, qkv lives in the block)
# =============================================================================

import os
from google.colab import drive
drive.mount('/content/drive')

# Check the live custom_config.yaml values before training (informational only).
import yaml
_cfg_preview_path = '/content/drive/MyDrive/InLineDAttention/InLine/custom_config.yaml'
if os.path.exists(_cfg_preview_path):
    with open(_cfg_preview_path, 'r') as f:
        print(yaml.safe_load(f))

# To change DATASET or MODEL_TYPE, edit the variables in Section 3.

############### 1. Setup and Installation ################
import os
WORK_DIR = "/content/drive/MyDrive/InLineDAttention"
DRIVE_DATA_DIR = os.path.join(WORK_DIR, "data")
DRIVE_ARCHIVE_DIR = os.path.join(WORK_DIR, "data_archives")

LOCAL_WORK_DIR = "/content/InLineDAttention"
DATA_DIR = os.path.join(LOCAL_WORK_DIR, "data")

DRIVE_OUTPUT_DIR = os.path.join(WORK_DIR, "output")
LOCAL_OUTPUT_DIR = os.path.join(LOCAL_WORK_DIR, "output")

os.makedirs(WORK_DIR, exist_ok=True)
os.makedirs(DRIVE_DATA_DIR, exist_ok=True)
os.makedirs(DRIVE_ARCHIVE_DIR, exist_ok=True)
os.makedirs(LOCAL_WORK_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)
os.makedirs(LOCAL_OUTPUT_DIR, exist_ok=True)
os.chdir(WORK_DIR)

print(f"Working directory: {WORK_DIR}")
print(f"Drive data folder: {DRIVE_DATA_DIR}")
print(f"Drive archive folder: {DRIVE_ARCHIVE_DIR}")
print(f"Local training data directory: {DATA_DIR}")
print(f"Local logs/checkpoints folder: {LOCAL_OUTPUT_DIR}")
print(f"Drive logs/checkpoints folder: {DRIVE_OUTPUT_DIR}")

###########################################################################
import shutil
import re
import hashlib

repo_path = os.path.join(WORK_DIR, "InLine")
if os.path.exists(repo_path):
    print(f"Repo already exists at {repo_path} - skipping clone, edits preserved")
else:
    os.makedirs(os.path.dirname(repo_path), exist_ok=True)
    !git clone https://github.com/LeapLabTHU/InLine {repo_path}
    print(f"Cloned fresh to {repo_path}")

os.chdir(repo_path)
print(f"Working directory: {os.getcwd()}")

############################################################################
# Install dependencies
!pip install timm==0.4.12 yacs Pillow numpy einops -q
print("Dependencies installed!")

############### 1b. Copy InLine-DS model files + strip every softmax path ######
DS_SRC_DIR = os.path.join(WORK_DIR, "inlineds_models")
os.makedirs(DS_SRC_DIR, exist_ok=True)

# All four backbone files are OPTIONAL to upload; only the file for the backbone
# you select in Section 3 is required.
DS_FILES = [
    "inline_deit_ds.py",
    "inline_swin_ds.py",
    "inline_pvt_ds.py",
    "inline_cswin_ds.py",
]
models_dir = os.path.join(repo_path, "models")
os.makedirs(models_dir, exist_ok=True)

for fname in DS_FILES:
    src = os.path.join(DS_SRC_DIR, fname)
    dst = os.path.join(models_dir, fname)
    if os.path.exists(src):
        shutil.copy2(src, dst)
        with open(dst, "rb") as _fh:
            _raw = _fh.read()
        _nlines = _raw.count(b"\n")
        _digest = hashlib.md5(_raw).hexdigest()[:8]
        print(f"  copied {fname} -> models/  ({_nlines} lines, md5:{_digest})")
        _txt = _raw.decode("utf-8", "ignore")
        if "A_std = F.softmax" in _txt or "A_std=F.softmax" in _txt:
            print(f"      WARNING: {fname} contains 'A_std = F.softmax' in the kernel. "
                  f"This is a softmax inside the active attention path (not a residual "
                  f"gate) and will NOT be stripped. Section 7 will report softmax > 0.")
        if "reshape(b * c, 1, 3, 3)" in _txt or "reshape(b*c, 1, 3, 3)" in _txt:
            print(f"      WARNING: {fname} uses a fixed 3x3 local-residual reshape that "
                  f"crashes when a stage has >1 window (e.g. CSWin at 32/64px). Use the "
                  f"repeat_interleave / local_k=5 version instead.")
        if ", q_low, q_high = self._dual_stream_attn" in _txt or \
           re.search(r"denom\s*=\s*gate\s*\*\s*denom_low", _txt):
            print(f"      WARNING: {fname} has a stale _dual_stream_attn that does not "
                  f"squeeze the gate (denom = gate * denom_low). It crashes with "
                  f"'size of tensor a (num_heads) must match tensor b (batch)'. Use the "
                  f"version where _dual_stream_attn returns (out, denom, q_high) and uses "
                  f"gate.squeeze(-1).")
    else:
        print(f"  (skipping {fname} - not in inlineds_models/)")


_SOFTMAX_TOKENS = ("torch." + "softmax", "F." + "softmax", "nn." + "Softmax")


def _strip_use_residual_gate_block(src: str):
    lines = src.split("\n")
    out, i, changed = [], 0, False
    while i < len(lines):
        line = lines[i]
        _ls = line.strip()
        if _ls.startswith("if ") and "use_residual_gate" in _ls and "residual_gate_raw" in _ls:
            indent = len(line) - len(line.lstrip())
            out.append(" " * indent + "# [InLine-only] softmax residual gate removed.")
            changed = True
            i += 1
            while i < len(lines):
                nxt = lines[i]
                if nxt.strip() == "":
                    j = i + 1
                    while j < len(lines) and lines[j].strip() == "":
                        j += 1
                    if j < len(lines):
                        nindent = len(lines[j]) - len(lines[j].lstrip())
                        if nindent <= indent:
                            break
                    i += 1
                    continue
                nindent = len(nxt) - len(nxt.lstrip())
                if nindent <= indent:
                    break
                i += 1
            continue
        out.append(line)
        i += 1
    return "\n".join(out), changed


def _neutralize_softmax_residual_method(src: str):
    changed = False
    target = "        return torch.matmul(torch." + "softmax(logits, dim=-1), v)"
    repl = "        raise RuntimeError('softmax residual disabled (InLine-only).')"
    if target in src:
        src = src.replace(target, repl)
        changed = True
    return src, changed


def _which_attention_class_region(src: str):
    names = ["InLineDSAttention", "InLineDSSwinAttention", "PVTDSAttention"]
    for name in names:
        m = re.search(r"\nclass\s+" + name + r"\b", src)
        if not m:
            continue
        start = m.start()
        nxt = re.search(r"\nclass\s+\w+", src[m.end():])
        end = m.end() + nxt.start() if nxt else len(src)
        return start, end
    return None


def _patch_inline_only(py_path: str):
    if not os.path.isfile(py_path):
        return
    with open(py_path, "r") as fh:
        src = fh.read()

    src, c1 = _strip_use_residual_gate_block(src)
    src, c2 = _neutralize_softmax_residual_method(src)

    if c1 or c2:
        with open(py_path, "w") as fh:
            fh.write(src)
        print(f"  stripped softmax residual from {os.path.basename(py_path)}")
    else:
        print(f"  no softmax residual branch found in {os.path.basename(py_path)} (already clean)")

    region = _which_attention_class_region(src)
    if region is not None:
        start, end = region
        body = src[start:end]
        line0 = src[:start].count("\n") + 1
        offending = []
        for _ln_off, _line in enumerate(body.split("\n")):
            if any(tok in _line for tok in _SOFTMAX_TOKENS):
                offending.append((line0 + _ln_off, _line.strip()))
        if offending:
            print(f"  NOTE: {len(offending)} softmax token(s) remain in the active "
                  f"attention class of {os.path.basename(py_path)} "
                  f"(advisory - Section 7 verifies at runtime):")
            for _lno, _txt in offending[:6]:
                print(f"        L{_lno}: {_txt[:90]}")
            print("        If Section 7 reports 0 softmax calls, this path is unreachable "
                  "and safe; otherwise keep ATTN_TYPE all-'D' / set its guard flag off.")


def _patch_pvt_factory_attn_type(py_path: str):
    if not os.path.isfile(py_path):
        return
    with open(py_path, "r") as fh:
        src = fh.read()
    new = re.sub(
        r"depths=(\[[0-9, ]+\]),\s*sr_ratios=(\[[0-9, ]+\]),\s*attn_type='DDDD',\s*\*\*kwargs\)",
        r"depths=\1, sr_ratios=\2, attn_type=kwargs.pop('attn_type', 'DDDD'), **kwargs)",
        src,
    )
    if new != src:
        with open(py_path, "w") as fh:
            fh.write(new)
        print(f"  patched PVT factories to accept external attn_type in {os.path.basename(py_path)}")


for _fname in DS_FILES:
    _path = os.path.join(models_dir, _fname)
    if os.path.exists(_path):
        _patch_inline_only(_path)

_pvt_path = os.path.join(models_dir, "inline_pvt_ds.py")
if os.path.exists(_pvt_path):
    _patch_pvt_factory_attn_type(_pvt_path)

############### 1c. Build builder: PREFER uploaded build_ds.py ######
# ---------------------------------------------------------------------------
build_ds_code = '''# Auto-generated by the Colab notebook. Unified InLine^D-S model builder.
import importlib


def _family(model_type: str) -> str:
    mt = model_type.lower()
    if "cswin" in mt:
        return "cswin"
    if "swin" in mt:
        return "swin"
    if "pvt" in mt:
        return "pvt"
    return "deit"


def _sanitize_attn_type(at, num_stages=4):
    """Force an InLine-only attn_type: every stage is 'D'. No 'S' (softmax)."""
    if at is None:
        at = "D" * num_stages
    at = str(at)
    at = "".join("D" for _ in at) if at else "D" * num_stages
    if len(at) < num_stages:
        at = (at + "D" * num_stages)[:num_stages]
    return at[:num_stages]


def _get(config, *path, default=None):
    node = config
    for p in path:
        if not hasattr(node, p):
            return default
        node = getattr(node, p)
    return node


def build_model(config):
    model_type = config.MODEL.TYPE
    fam = _family(model_type)

    num_classes = int(config.MODEL.NUM_CLASSES)
    img_size = int(config.DATA.IMG_SIZE)
    drop_path = float(_get(config, "MODEL", "DROP_PATH_RATE", default=0.1))
    drop_rate = float(_get(config, "MODEL", "DROP_RATE", default=0.0))
    attn_type = _sanitize_attn_type(_get(config, "MODEL", "INLINE", "ATTN_TYPE", default="DDDD"))

    module_name = "models.inline_%s_ds" % fam
    M = importlib.import_module(module_name)
    if not hasattr(M, model_type):
        raise ValueError(
            "Model '%s' is not registered in %s. Available: %s"
            % (model_type, module_name,
               [a for a in dir(M) if a.startswith("inline_%s_ds" % fam)])
        )
    fn = getattr(M, model_type)

    common = dict(num_classes=num_classes, img_size=img_size,
                  drop_path_rate=drop_path, drop_rate=drop_rate)

    if fam == "deit":
        # DeiT resolves its own patch size; honour MODEL.SWIN.PATCH_SIZE if given.
        patch = _get(config, "MODEL", "SWIN", "PATCH_SIZE", default=None)
        kw = dict(common)
        if patch is not None:
            try:
                kw["patch_size"] = int(patch)
            except Exception:
                pass
        try:
            return fn(pretrained=False, **kw)
        except TypeError:
            kw.pop("patch_size", None)
            return fn(pretrained=False, **kw)

    # Pyramid / windowed backbones: keep attn_type all-D and pass
    # small-image geometry when the model factory supports it.
    if fam == "swin":
        window_size = int(_get(config, "MODEL", "SWIN", "WINDOW_SIZE", default=7))
        patch_size = int(_get(config, "MODEL", "SWIN", "PATCH_SIZE", default=4))
        try:
            return fn(pretrained=False, attn_type=attn_type,
                      patch_size=patch_size, window_size=window_size, **common)
        except TypeError:
            try:
                return fn(pretrained=False, attn_type=attn_type,
                          window_size=window_size, **common)
            except TypeError:
                return fn(pretrained=False, attn_type=attn_type, **common)

    if fam == "cswin":
        split = str(_get(config, "MODEL", "INLINE", "CSWIN_LA_SPLIT_SIZE",
                         default="7-4-2-1"))
        try:
            return fn(pretrained=False, attn_type=attn_type,
                      la_split_size=split, **common)
        except TypeError:
            try:
                return fn(pretrained=False, attn_type=attn_type,
                          split_size=split, **common)
            except TypeError:
                return fn(pretrained=False, attn_type=attn_type, **common)

    return fn(pretrained=False, attn_type=attn_type, **common)
'''

build_ds_path = os.path.join(models_dir, "build_ds.py")
_uploaded_build_ds = os.path.join(DS_SRC_DIR, "build_ds.py")

if os.path.exists(_uploaded_build_ds):
    # Code-One parity: use the validated uploaded builder verbatim.
    shutil.copy2(_uploaded_build_ds, build_ds_path)
    with open(build_ds_path, "rb") as _bfh:
        _braw = _bfh.read()
    print(f"  USING UPLOADED build_ds.py (Code-One parity) -> {build_ds_path}  "
          f"({_braw.count(bytes([10]))} lines, md5:{hashlib.md5(_braw).hexdigest()[:8]})")
    print("  (This restores the exact DeiT construction Code One trains; the timm "
          "fallback in the wrapper still covers any backbone it does not handle.)")
else:
    with open(build_ds_path, "w") as fh:
        fh.write(build_ds_code)
    print(f"  no uploaded build_ds.py found in {DS_SRC_DIR}")
    print(f"  wrote GENERATED unified builder -> {build_ds_path}")
    print("  TIP: for best inline_deit_ds_* accuracy, upload the same build_ds.py "
          "you used with 'Code One' to inlineds_models/ so this notebook matches it.")

# Patch models/__init__.py to use build_ds.build_model
init_path = os.path.join(models_dir, "__init__.py")
init_src = ""
if os.path.exists(init_path):
    with open(init_path, "r") as fh:
        init_src = fh.read()

DS_INIT_LINE = "from .build_ds import build_model  # InLine^D-S unified builder\n"
if DS_INIT_LINE not in init_src:
    if "from .build import build_model" in init_src:
        init_src = init_src.replace(
            "from .build import build_model",
            "# original: from .build import build_model  (kept for reference)\n"
            + DS_INIT_LINE.rstrip(),
        )
    else:
        init_src += "\n" + DS_INIT_LINE
    with open(init_path, "w") as fh:
        fh.write(init_src)
    print("  models/__init__.py patched to use build_ds.build_model")
else:
    print("  models/__init__.py already patched")

############### 2. Data Loader for Multi-Dataset Support ################
import re
import subprocess as _sp

# data/build.py is EXTENDED in place idempotently from a pristine snapshot.
_build_py = os.path.join(repo_path, "data", "build.py")
_build_orig = os.path.join(repo_path, "data", "build.py.inline_orig")
_EXT_MARK = "_resolve_tinyimagenet_root"


def _looks_pristine_build(txt):
    return ("def build_loader" in txt) and (_EXT_MARK not in txt) and (txt.count("\n") < 3000)


def _restore_build_from_git():
    for _ref in ("HEAD", ""):
        try:
            _sp.run(["git", "-C", repo_path, "checkout", _ref, "--", "data/build.py"],
                    check=False, capture_output=True)
        except Exception:
            pass
        try:
            if _looks_pristine_build(open(_build_py).read()):
                return True
        except Exception:
            pass
    return False


_have_pristine = os.path.exists(_build_orig) and _looks_pristine_build(open(_build_orig).read())
if not _have_pristine:
    if not _looks_pristine_build(open(_build_py).read()):
        if not _restore_build_from_git():
            raise RuntimeError(
                "Could not obtain a pristine data/build.py (it is already modified and "
                "`git checkout` did not restore it). Fix: delete the InLine repo folder on "
                "Drive (rm -rf <Drive>/InLineDAttention/InLine) and re-run Section 1 to "
                "re-clone, then run again."
            )
    shutil.copy2(_build_py, _build_orig)
    print("  saved pristine snapshot -> data/build.py.inline_orig")

with open(_build_orig, "r") as f:
    original_build = f.read()


extended_build_dataset = '''
def _tiny_has_usable_eval_split(root):
    if not os.path.isdir(os.path.join(root, 'train')):
        return False
    return (
        os.path.isdir(os.path.join(root, 'val')) or
        os.path.isdir(os.path.join(root, 'test')) or
        os.path.isdir(os.path.join(root, 'train'))
    )


def _resolve_tinyimagenet_root(data_path):
    tiny_aliases = [
        'tiny-imagenet-200', 'tiny-image-net-200',
        'tinyimagenet-200', 'tinyimagenet200', 'TinyImageNet-200'
    ]
    candidates = []
    for name in tiny_aliases:
        candidates.append(os.path.join(data_path, name))
        candidates.append(os.path.join(data_path, name, 'tiny-imagenet-200'))
        candidates.append(os.path.join(data_path, name, 'tiny-image-net-200'))
    candidates.append(data_path)

    for root in candidates:
        if _tiny_has_usable_eval_split(root):
            return root

    for parent in [data_path] + [os.path.join(data_path, name) for name in tiny_aliases]:
        if not os.path.isdir(parent):
            continue
        try:
            for child in os.listdir(parent):
                root = os.path.join(parent, child)
                if _tiny_has_usable_eval_split(root):
                    return root
        except Exception:
            pass
    return candidates[0]


def _tiny_split_has_class_dirs(split_dir):
    if not os.path.isdir(split_dir):
        return False
    try:
        return any(os.path.isdir(os.path.join(split_dir, d)) for d in os.listdir(split_dir))
    except Exception:
        return False


def _tinyimagenet_eval_split_name(tiny_root):
    val_dir = os.path.join(tiny_root, 'val')
    test_dir = os.path.join(tiny_root, 'test')
    if os.path.isdir(val_dir):
        if _tiny_split_has_class_dirs(val_dir) or (
            os.path.isdir(os.path.join(val_dir, 'images')) and
            os.path.exists(os.path.join(val_dir, 'val_annotations.txt'))
        ):
            return 'val'
    if _tiny_split_has_class_dirs(test_dir):
        return 'test'
    print('TinyImageNet val/ class folders were not found; using train/ as validation fallback.')
    return 'train'


def _prepare_tinyimagenet_val(tiny_root):
    val_dir = os.path.join(tiny_root, 'val')
    val_imgs = os.path.join(val_dir, 'images')
    val_anno = os.path.join(val_dir, 'val_annotations.txt')
    marker = os.path.join(tiny_root, '.imagefolder_ready')

    if os.path.exists(marker):
        return

    if os.path.isdir(val_dir):
        class_dirs = [d for d in os.listdir(val_dir)
                      if os.path.isdir(os.path.join(val_dir, d)) and d != 'images']
        if len(class_dirs) >= 200:
            try:
                with open(marker, 'w') as f:
                    f.write('TinyImageNet validation split prepared for ImageFolder.' + chr(10))
            except Exception:
                pass
            return

    if not (os.path.isdir(val_imgs) and os.path.exists(val_anno)):
        return

    with open(val_anno, 'r') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 2:
                continue
            img_name, cls = parts[0], parts[1]
            cls_dir = os.path.join(val_dir, cls)
            os.makedirs(cls_dir, exist_ok=True)
            src_path = os.path.join(val_imgs, img_name)
            dst_path = os.path.join(cls_dir, img_name)
            if os.path.exists(src_path) and not os.path.exists(dst_path):
                shutil.move(src_path, dst_path)

    if os.path.isdir(val_imgs) and not os.listdir(val_imgs):
        shutil.rmtree(val_imgs)
    try:
        with open(marker, 'w') as f:
            f.write('TinyImageNet validation split prepared for ImageFolder.' + chr(10))
    except Exception:
        pass


def _flatten_tinyimagenet_split(split_dir):
    if not os.path.isdir(split_dir):
        return
    marker = os.path.join(split_dir, '.imagefolder_flat')
    if os.path.exists(marker):
        return
    for cls in list(os.listdir(split_dir)):
        cls_dir = os.path.join(split_dir, cls)
        if not os.path.isdir(cls_dir):
            continue
        images_sub = os.path.join(cls_dir, 'images')
        if os.path.isdir(images_sub):
            tmp_dir = os.path.join(split_dir, '.__flatten_tmp__' + cls)
            if os.path.exists(tmp_dir):
                shutil.rmtree(tmp_dir, ignore_errors=True)
            try:
                os.rename(cls_dir, tmp_dir)
                os.rename(os.path.join(tmp_dir, 'images'), cls_dir)
            except OSError:
                os.makedirs(cls_dir, exist_ok=True)
                src_images = os.path.join(tmp_dir, 'images') if os.path.isdir(os.path.join(tmp_dir, 'images')) else images_sub
                if os.path.isdir(src_images):
                    for fn in os.listdir(src_images):
                        src_path = os.path.join(src_images, fn)
                        dst_path = os.path.join(cls_dir, fn)
                        if not os.path.exists(dst_path):
                            shutil.move(src_path, dst_path)
            finally:
                if os.path.isdir(tmp_dir):
                    shutil.rmtree(tmp_dir, ignore_errors=True)
    try:
        with open(marker, 'w') as f:
            f.write('TinyImageNet split flattened for ImageFolder.' + chr(10))
    except Exception:
        pass


def _food101_root_is_usable(root):
    images_dir = os.path.join(root, 'images')
    if not os.path.isdir(images_dir):
        return False
    try:
        return any(os.path.isdir(os.path.join(images_dir, d)) for d in os.listdir(images_dir))
    except Exception:
        return False


def _resolve_food101_root(data_path):
    aliases = ['food101', 'food-101', 'Food101', 'Food-101']
    candidates = [os.path.join(data_path, name) for name in aliases]
    candidates.append(data_path)
    if os.path.isdir(data_path):
        try:
            for name in os.listdir(data_path):
                if name.lower().replace('-', '').replace('_', '') == 'food101':
                    candidates.append(os.path.join(data_path, name))
        except Exception:
            pass
    seen = set()
    for root in candidates:
        if root in seen:
            continue
        seen.add(root)
        if _food101_root_is_usable(root):
            return root
    return None


class _Food101FolderDataset(__import__('torch').utils.data.Dataset):
    def __init__(self, root, split='train', transform=None):
        from PIL import Image
        self.Image = Image
        self.root = root
        self.split = split
        self.transform = transform
        self.images_dir = os.path.join(root, 'images')
        self.meta_dir = os.path.join(root, 'meta')
        classes_file = os.path.join(self.meta_dir, 'classes.txt')

        if os.path.exists(classes_file):
            with open(classes_file, 'r') as f:
                self.classes = [line.strip() for line in f if line.strip()]
        else:
            self.classes = sorted([
                d for d in os.listdir(self.images_dir)
                if os.path.isdir(os.path.join(self.images_dir, d))
            ])

        self.class_to_idx = {c: i for i, c in enumerate(self.classes)}
        split_file = os.path.join(self.meta_dir, split + '.txt')
        self.samples = []

        if os.path.exists(split_file):
            with open(split_file, 'r') as f:
                rels = [line.strip() for line in f if line.strip()]
            for rel in rels:
                cls = rel.split('/')[0]
                img_path = os.path.join(self.images_dir, rel + '.jpg')
                if cls in self.class_to_idx and os.path.exists(img_path):
                    self.samples.append((img_path, self.class_to_idx[cls]))
        else:
            for cls in self.classes:
                cls_dir = os.path.join(self.images_dir, cls)
                if not os.path.isdir(cls_dir):
                    continue
                for name in os.listdir(cls_dir):
                    if name.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp', '.webp')):
                        self.samples.append((os.path.join(cls_dir, name), self.class_to_idx[cls]))

        if len(self.samples) == 0:
            raise FileNotFoundError(
                f"No Food-101 images found in {root} for split '{split}'. "
                "Expected images/<class>/*.jpg and preferably meta/train.txt/meta/test.txt."
            )
        self.targets = [y for _, y in self.samples]

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        path, target = self.samples[index]
        img = self.Image.open(path).convert('RGB')
        if self.transform is not None:
            img = self.transform(img)
        return img, target


def build_dataset(is_train, config):
    transform = build_transform(is_train, config)
    dataset_name = config.DATA.DATASET.lower()

    if dataset_name == 'cifar10':
        from torchvision.datasets import CIFAR10
        dataset = CIFAR10(root=config.DATA.DATA_PATH, train=is_train,
                          download=False, transform=transform)
        nb_classes = 10

    elif dataset_name == 'cifar100':
        from torchvision.datasets import CIFAR100
        dataset = CIFAR100(root=config.DATA.DATA_PATH, train=is_train,
                           download=False, transform=transform)
        nb_classes = 100

    elif dataset_name == 'svhn':
        from torchvision.datasets import SVHN
        split = 'train' if is_train else 'test'
        dataset = SVHN(root=config.DATA.DATA_PATH, split=split,
                       download=False, transform=transform)
        nb_classes = 10

    elif dataset_name == 'tinyimagenet':
        from torchvision.datasets import ImageFolder
        tiny_root = _resolve_tinyimagenet_root(config.DATA.DATA_PATH)
        _prepare_tinyimagenet_val(tiny_root)
        split_name = 'train' if is_train else _tinyimagenet_eval_split_name(tiny_root)
        split_path = os.path.join(tiny_root, split_name)
        if not os.path.isdir(split_path):
            raise FileNotFoundError(f"TinyImageNet split not found: {split_path}")
        _flatten_tinyimagenet_split(split_path)
        dataset = ImageFolder(root=split_path, transform=transform)
        nb_classes = 200

    elif dataset_name == 'food101':
        split = 'train' if is_train else 'test'
        food_root = _resolve_food101_root(config.DATA.DATA_PATH)
        if food_root is None:
            raise FileNotFoundError(
                f"Food-101 was not found under {config.DATA.DATA_PATH}. "
                "Expected food101/ or food-101/ containing images/."
            )
        dataset = _Food101FolderDataset(food_root, split=split, transform=transform)
        nb_classes = 101

    elif dataset_name == 'imagenet':
        prefix = 'train' if is_train else 'val'
        if config.DATA.ZIP_MODE:
            ann_file = prefix + "_map.txt"
            prefix   = prefix + ".zip@/"
            dataset  = CachedImageFolder(
                config.DATA.DATA_PATH, ann_file, prefix, transform,
                cache_mode=config.DATA.CACHE_MODE if is_train else 'part')
        else:
            root    = os.path.join(config.DATA.DATA_PATH, prefix)
            dataset = datasets.ImageFolder(root, transform=transform)
        nb_classes = 1000

    else:
        raise NotImplementedError(
            f"Supported: imagenet, cifar10, cifar100, svhn, tinyimagenet, food101. "
            f"Got: {dataset_name}")

    return dataset, nb_classes
'''

extended_transform = '''
def build_transform(is_train, config):
    dataset_name = config.DATA.DATASET.lower()
    resize_im    = config.DATA.IMG_SIZE > 32

    if dataset_name in ['cifar10', 'cifar100']:
        mean = (0.4914, 0.4822, 0.4465)
        std  = (0.2470, 0.2435, 0.2616)
        if is_train:
            transform = transforms.Compose([
                transforms.RandomCrop(config.DATA.IMG_SIZE, padding=4)
                if config.DATA.IMG_SIZE <= 64 else
                transforms.RandomResizedCrop(config.DATA.IMG_SIZE, scale=(0.6, 1.0)),
                transforms.RandomHorizontalFlip(),
                transforms.ToTensor(),
                transforms.Normalize(mean, std),
            ])
        else:
            if config.DATA.IMG_SIZE == 32:
                transform = transforms.Compose([
                    transforms.ToTensor(),
                    transforms.Normalize(mean, std),
                ])
            else:
                transform = transforms.Compose([
                    transforms.Resize((config.DATA.IMG_SIZE, config.DATA.IMG_SIZE)),
                    transforms.ToTensor(),
                    transforms.Normalize(mean, std),
                ])
        return transform

    if dataset_name == 'svhn':
        mean = (0.4377, 0.4438, 0.4728)
        std  = (0.1980, 0.2010, 0.1970)
        if is_train:
            transform = transforms.Compose([
                transforms.RandomCrop(config.DATA.IMG_SIZE, padding=4)
                if config.DATA.IMG_SIZE <= 64 else
                transforms.Resize((config.DATA.IMG_SIZE, config.DATA.IMG_SIZE)),
                transforms.ToTensor(),
                transforms.Normalize(mean, std),
            ])
        else:
            transform = transforms.Compose([
                transforms.Resize((config.DATA.IMG_SIZE, config.DATA.IMG_SIZE))
                if config.DATA.IMG_SIZE != 32 else transforms.Lambda(lambda x: x),
                transforms.ToTensor(),
                transforms.Normalize(mean, std),
            ])
        return transform

    if dataset_name == 'food101':
        if is_train:
            transform = transforms.Compose([
                transforms.RandomResizedCrop(
                    config.DATA.IMG_SIZE,
                    scale=(0.6, 1.0),
                    ratio=(0.75, 1.3333)
                ),
                transforms.RandomHorizontalFlip(),
                transforms.ColorJitter(
                    brightness=0.4, contrast=0.4,
                    saturation=0.4, hue=0.1),
                transforms.ToTensor(),
                transforms.Normalize(IMAGENET_DEFAULT_MEAN,
                                     IMAGENET_DEFAULT_STD),
            ])
        else:
            transform = transforms.Compose([
                transforms.Resize(
                    int((256 / 224) * config.DATA.IMG_SIZE)),
                transforms.CenterCrop(config.DATA.IMG_SIZE),
                transforms.ToTensor(),
                transforms.Normalize(IMAGENET_DEFAULT_MEAN,
                                     IMAGENET_DEFAULT_STD),
            ])
        return transform

    if is_train:
        transform = create_transform(
            input_size    = config.DATA.IMG_SIZE,
            is_training   = True,
            color_jitter  = config.AUG.COLOR_JITTER
                            if config.AUG.COLOR_JITTER > 0 else None,
            auto_augment  = config.AUG.AUTO_AUGMENT
                            if config.AUG.AUTO_AUGMENT != 'none' else None,
            re_prob       = config.AUG.REPROB,
            re_mode       = config.AUG.REMODE,
            re_count      = config.AUG.RECOUNT,
            interpolation = config.DATA.INTERPOLATION,
        )
        if not resize_im:
            transform.transforms[0] = transforms.RandomCrop(
                config.DATA.IMG_SIZE, padding=4)
        return transform

    t = []
    if resize_im:
        if config.TEST.CROP:
            size = int((256 / 224) * config.DATA.IMG_SIZE)
            t.append(transforms.Resize(
                (size, size),
                interpolation=_pil_interp(config.DATA.INTERPOLATION)))
            t.append(transforms.CenterCrop(config.DATA.IMG_SIZE))
        else:
            t.append(transforms.Resize(
                (config.DATA.IMG_SIZE, config.DATA.IMG_SIZE),
                interpolation=_pil_interp(config.DATA.INTERPOLATION)))
    t.append(transforms.ToTensor())
    t.append(transforms.Normalize(IMAGENET_DEFAULT_MEAN,
                                   IMAGENET_DEFAULT_STD))
    return transforms.Compose(t)
'''

modified_build = re.sub(
    r'def build_dataset\(is_train, config\):.*?(?=\ndef |\nclass |$)',
    extended_build_dataset.strip(),
    original_build,
    flags=re.DOTALL,
)
modified_build = re.sub(
    r'def build_transform\(is_train, config\):.*?(?=\ndef |\nclass |$)',
    extended_transform.strip(),
    modified_build,
    flags=re.DOTALL,
)

if 'import shutil' not in modified_build:
    modified_build = modified_build.replace('import os', 'import os\nimport shutil', 1) if 'import os' in modified_build else 'import shutil\n' + modified_build

with open("data/build.py", "w") as f:
    f.write(modified_build)

print("data/build.py extended with multi-dataset support")

############### 3. Select Dataset & Model ################
DATASET    = 'cifar100'   # Options include: cifar10, cifar100, tinyimagenet, svhn
# Any of the backbones (tiny/small/base variants as registered):
#   inline_deit_ds_tiny  inline_deit_ds_small  inline_deit_ds_base
#   inline_swin_ds_tiny  inline_swin_ds_small  inline_swin_ds_base
#   inline_pvt_ds_tiny   inline_pvt_ds_small   inline_pvt_ds_medium  inline_pvt_ds_large
#   inline_cswin_ds_tiny inline_cswin_ds_small inline_cswin_ds_base
MODEL_TYPE = 'inline_cswin_ds_tiny'


def _model_family(mt):
    mt = mt.lower()
    if 'cswin' in mt:
        return 'cswin'
    if 'swin' in mt:
        return 'swin'
    if 'pvt' in mt:
        return 'pvt'
    return 'deit'


def _model_module_name(mt):
    return 'models.inline_%s_ds' % _model_family(mt)


MODEL_FAMILY = _model_family(MODEL_TYPE)
MODEL_MODULE = _model_module_name(MODEL_TYPE)

_required_file = os.path.join(DS_SRC_DIR, 'inline_%s_ds.py' % MODEL_FAMILY)
if not os.path.exists(os.path.join(models_dir, 'inline_%s_ds.py' % MODEL_FAMILY)):
    raise FileNotFoundError(
        "\n\nMissing model file for the selected backbone '%s'.\n"
        "    Upload  inline_%s_ds.py  to your Drive at:\n      %s\n"
        % (MODEL_TYPE, MODEL_FAMILY, _required_file)
    )

print(f"Selected MODEL_TYPE : {MODEL_TYPE}")
print(f"Backbone family     : {MODEL_FAMILY}")
print(f"Model module        : {MODEL_MODULE}")

############### 3a. Stage Dataset for Fast Local Training ################
print(f"\nPreparing {DATASET} for local Colab training...")
print(f"Drive archive folder: {DRIVE_ARCHIVE_DIR}")
print(f"Local training data : {DATA_DIR}")

import tarfile
import zipfile
import subprocess


def _norm_name(name):
    return name.lower().replace('-', '').replace('_', '').replace(' ', '')


def _find_archive(dataset_name):
    archive_exts = ['.tar.gz', '.tgz', '.tar', '.zip']
    if dataset_name == 'food101':
        bases = ['food101', 'food-101', 'Food101', 'Food-101']
    elif dataset_name == 'tinyimagenet':
        bases = [
            'tiny-imagenet-200', 'tiny-image-net-200',
            'tinyimagenet-200', 'tinyimagenet200', 'TinyImageNet-200'
        ]
    else:
        return None

    search_dirs = [DRIVE_ARCHIVE_DIR, DRIVE_DATA_DIR]
    for folder in search_dirs:
        if not os.path.isdir(folder):
            continue
        for base in bases:
            for ext in archive_exts:
                path = os.path.join(folder, base + ext)
                if os.path.exists(path):
                    return path
        for name in os.listdir(folder):
            low = name.lower()
            if not any(low.endswith(ext) for ext in archive_exts):
                continue
            stem = low
            for ext in archive_exts:
                if stem.endswith(ext):
                    stem = stem[:-len(ext)]
                    break
            if _norm_name(stem) in [_norm_name(b) for b in bases]:
                return os.path.join(folder, name)
    return None


def _extract_archive(archive_path, extract_dir):
    os.makedirs(extract_dir, exist_ok=True)
    print(f"Extracting archive to local Colab SSD: {archive_path}")
    if archive_path.lower().endswith('.zip'):
        with zipfile.ZipFile(archive_path, 'r') as zf:
            zf.extractall(extract_dir)
    elif archive_path.lower().endswith(('.tar.gz', '.tgz', '.tar')):
        with tarfile.open(archive_path, 'r:*') as tf:
            tf.extractall(extract_dir)
    else:
        raise ValueError(f"Unsupported archive format: {archive_path}")


def _create_archive_from_drive_folder(dataset_name):
    if dataset_name == 'food101':
        aliases = ['food101', 'food-101', 'Food101', 'Food-101']
        archive_name = 'food101.tar.gz'
    elif dataset_name == 'tinyimagenet':
        aliases = [
            'tiny-imagenet-200', 'tiny-image-net-200',
            'tinyimagenet-200', 'tinyimagenet200', 'TinyImageNet-200'
        ]
        archive_name = 'tiny-imagenet-200.tar.gz'
    else:
        return None

    src_folder = None
    for alias in aliases:
        candidate = os.path.join(DRIVE_DATA_DIR, alias)
        if os.path.isdir(candidate):
            src_folder = candidate
            break
    if src_folder is None:
        return None

    os.makedirs(DRIVE_ARCHIVE_DIR, exist_ok=True)
    archive_path = os.path.join(DRIVE_ARCHIVE_DIR, archive_name)
    if os.path.exists(archive_path):
        return archive_path

    print(f"No archive found, but found extracted Drive folder: {src_folder}")
    print(f"Creating one-time compressed archive on Drive: {archive_path}")
    with tarfile.open(archive_path, 'w:gz') as tf:
        tf.add(src_folder, arcname=os.path.basename(src_folder))
    return archive_path


def _resolve_food101_root_local(base_dir):
    aliases = ['food101', 'food-101', 'Food101', 'Food-101']
    candidates = [os.path.join(base_dir, name) for name in aliases]
    candidates.append(base_dir)
    if os.path.isdir(base_dir):
        try:
            for name in os.listdir(base_dir):
                if _norm_name(name) == 'food101':
                    candidates.append(os.path.join(base_dir, name))
        except Exception:
            pass
    seen = set()
    for root in candidates:
        if root in seen:
            continue
        seen.add(root)
        images_dir = os.path.join(root, 'images')
        if os.path.isdir(images_dir):
            try:
                if any(os.path.isdir(os.path.join(images_dir, d)) for d in os.listdir(images_dir)):
                    return root
            except Exception:
                pass
    return None


def _resolve_tiny_root(base_dir):
    aliases = [
        'tiny-imagenet-200', 'tiny-image-net-200',
        'tinyimagenet-200', 'tinyimagenet200', 'TinyImageNet-200'
    ]
    candidates = []
    for alias in aliases:
        candidates.append(os.path.join(base_dir, alias))
        candidates.append(os.path.join(base_dir, alias, 'tiny-imagenet-200'))
        candidates.append(os.path.join(base_dir, alias, 'tiny-image-net-200'))
    candidates.append(base_dir)
    for root in candidates:
        if os.path.isdir(os.path.join(root, 'train')) and (
            os.path.isdir(os.path.join(root, 'val')) or os.path.isdir(os.path.join(root, 'test'))
        ):
            return root
    return None


def _prepare_tiny_val_for_imagefolder(tiny_dir):
    val_dir = os.path.join(tiny_dir, 'val')
    val_imgs = os.path.join(val_dir, 'images')
    val_anno = os.path.join(val_dir, 'val_annotations.txt')
    marker = os.path.join(tiny_dir, '.imagefolder_ready')
    if os.path.exists(marker):
        return
    if os.path.isdir(val_dir):
        class_dirs = [d for d in os.listdir(val_dir)
                      if os.path.isdir(os.path.join(val_dir, d)) and d != 'images']
        if len(class_dirs) >= 200:
            with open(marker, 'w') as f:
                f.write('TinyImageNet validation split prepared for ImageFolder.' + chr(10))
            return
    if os.path.isdir(val_imgs) and os.path.exists(val_anno):
        with open(val_anno, 'r') as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) < 2:
                    continue
                img_name, cls = parts[0], parts[1]
                cls_dir = os.path.join(val_dir, cls)
                os.makedirs(cls_dir, exist_ok=True)
                src_path = os.path.join(val_imgs, img_name)
                dst_path = os.path.join(cls_dir, img_name)
                if os.path.exists(src_path) and not os.path.exists(dst_path):
                    shutil.move(src_path, dst_path)
        if os.path.isdir(val_imgs) and not os.listdir(val_imgs):
            shutil.rmtree(val_imgs)
        with open(marker, 'w') as f:
            f.write('TinyImageNet validation split prepared for ImageFolder.' + chr(10))


def _torchvision_dataset_available(dataset_name, root):
    dataset_name = dataset_name.lower()
    if dataset_name == 'cifar10':
        folder = os.path.join(root, 'cifar-10-batches-py')
        return os.path.isdir(folder) and os.path.exists(os.path.join(folder, 'data_batch_1'))
    if dataset_name == 'cifar100':
        folder = os.path.join(root, 'cifar-100-python')
        return os.path.isdir(folder) and os.path.exists(os.path.join(folder, 'train'))
    if dataset_name == 'svhn':
        return (
            os.path.exists(os.path.join(root, 'train_32x32.mat')) and
            os.path.exists(os.path.join(root, 'test_32x32.mat'))
        )
    return False


def _copy_path_to_local(src_path, dst_path):
    if os.path.isdir(src_path):
        if os.path.exists(dst_path):
            print(f"  ✓ Local dataset folder already exists: {dst_path}")
        else:
            print("  Copying existing Drive dataset folder to local Colab SSD:")
            print(f"    from: {src_path}")
            print(f"    to  : {dst_path}")
            shutil.copytree(src_path, dst_path, dirs_exist_ok=True)
    elif os.path.isfile(src_path):
        os.makedirs(os.path.dirname(dst_path), exist_ok=True)
        if os.path.exists(dst_path):
            print(f"  ✓ Local dataset file already exists: {dst_path}")
        else:
            print("  Copying existing Drive dataset file to local Colab SSD:")
            print(f"    from: {src_path}")
            print(f"    to  : {dst_path}")
            shutil.copy2(src_path, dst_path)


def _stage_torchvision_dataset_from_drive(dataset_name):
    """Reuse CIFAR/SVHN from Drive when present; download only if Drive has no usable copy."""
    dataset_name = dataset_name.lower()
    if _torchvision_dataset_available(dataset_name, DATA_DIR):
        print(f"  ✓ {dataset_name} already exists in local session storage; skipping download.")
        return True

    specs = {
        'cifar10': {
            'aliases': ['cifar10', 'cifar-10', 'CIFAR10'],
            'folders': ['cifar-10-batches-py'],
            'files': [],
            'archives': ['cifar-10-python.tar.gz'],
        },
        'cifar100': {
            'aliases': ['cifar100', 'cifar-100', 'CIFAR100'],
            'folders': ['cifar-100-python'],
            'files': [],
            'archives': ['cifar-100-python.tar.gz'],
        },
        'svhn': {
            'aliases': ['svhn', 'SVHN'],
            'folders': [],
            'files': ['train_32x32.mat', 'test_32x32.mat'],
            'archives': [],
        },
    }
    spec = specs[dataset_name]
    search_roots = [DRIVE_DATA_DIR, DRIVE_ARCHIVE_DIR]
    search_roots.extend(os.path.join(DRIVE_DATA_DIR, alias) for alias in spec['aliases'])
    search_roots.extend(os.path.join(DRIVE_ARCHIVE_DIR, alias) for alias in spec['aliases'])

    for root in search_roots:
        if _torchvision_dataset_available(dataset_name, root):
            print(f"  ✓ Found existing {dataset_name} dataset on Drive; no fresh download needed.")
            for folder in spec['folders']:
                src = os.path.join(root, folder)
                if os.path.isdir(src):
                    _copy_path_to_local(src, os.path.join(DATA_DIR, folder))
            for filename in spec['files']:
                src = os.path.join(root, filename)
                if os.path.isfile(src):
                    _copy_path_to_local(src, os.path.join(DATA_DIR, filename))
            return _torchvision_dataset_available(dataset_name, DATA_DIR)

    for root in search_roots:
        if not os.path.isdir(root):
            continue
        for archive_name in spec['archives']:
            archive_path = os.path.join(root, archive_name)
            if os.path.exists(archive_path):
                print(f"  ✓ Found existing {dataset_name} archive on Drive; extracting locally without a fresh download.")
                _extract_archive(archive_path, DATA_DIR)
                return _torchvision_dataset_available(dataset_name, DATA_DIR)

    print(f"  No usable {dataset_name} copy found on Drive; torchvision will download it once.")
    return False


def _stage_archive_dataset(dataset_name):
    if dataset_name == 'food101':
        existing = _resolve_food101_root_local(DATA_DIR)
        if existing is not None:
            print(f"Food-101 already exists locally: {existing}")
            return existing
    elif dataset_name == 'tinyimagenet':
        existing = _resolve_tiny_root(DATA_DIR)
        if existing is not None:
            print(f"TinyImageNet-200 already exists locally: {existing}")
            _prepare_tiny_val_for_imagefolder(existing)
            return existing
    else:
        return None

    archive_path = _find_archive(dataset_name)
    if archive_path is None:
        archive_path = _create_archive_from_drive_folder(dataset_name)

    if archive_path is None:
        raise FileNotFoundError(
            f"No archive found for {dataset_name}. Upload a .zip/.tar/.tar.gz archive to:\n"
            f"  {DRIVE_ARCHIVE_DIR}\n"
            f"or place the extracted dataset folder in:\n"
            f"  {DRIVE_DATA_DIR}\n"
            "Accepted names include food101, food-101, tiny-imagenet-200, tiny-image-net-200."
        )

    _extract_archive(archive_path, DATA_DIR)

    if dataset_name == 'food101':
        root = _resolve_food101_root_local(DATA_DIR)
        if root is None:
            raise FileNotFoundError(
                f"Food-101 archive was extracted, but no usable images/ folder was found under {DATA_DIR}."
            )
        print(f"Food-101 ready locally: {root}")
        return root

    if dataset_name == 'tinyimagenet':
        root = _resolve_tiny_root(DATA_DIR)
        if root is None:
            raise FileNotFoundError(
                f"TinyImageNet-200 archive was extracted, but no usable train/val or train/test folder was found under {DATA_DIR}."
            )
        _prepare_tiny_val_for_imagefolder(root)
        print(f"TinyImageNet-200 ready locally: {root}")
        return root


if DATASET == 'cifar10':
    from torchvision.datasets import CIFAR10
    staged_from_drive = _stage_torchvision_dataset_from_drive('cifar10')
    CIFAR10(DATA_DIR, train=True,  download=not staged_from_drive)
    CIFAR10(DATA_DIR, train=False, download=not staged_from_drive)
    print("CIFAR-10 ready!")

elif DATASET == 'cifar100':
    from torchvision.datasets import CIFAR100
    staged_from_drive = _stage_torchvision_dataset_from_drive('cifar100')
    CIFAR100(DATA_DIR, train=True,  download=not staged_from_drive)
    CIFAR100(DATA_DIR, train=False, download=not staged_from_drive)
    print("CIFAR-100 ready!")

elif DATASET == 'svhn':
    from torchvision.datasets import SVHN
    staged_from_drive = _stage_torchvision_dataset_from_drive('svhn')
    SVHN(DATA_DIR, split='train', download=not staged_from_drive)
    SVHN(DATA_DIR, split='test',  download=not staged_from_drive)
    print("SVHN ready!")

elif DATASET == 'tinyimagenet':
    _stage_archive_dataset('tinyimagenet')
    print("TinyImageNet-200 ready!")

elif DATASET == 'food101':
    _stage_archive_dataset('food101')
    print("Food-101 ready!")

else:
    raise ValueError(f"Unknown dataset: {DATASET}")


_FAMILY_FILE = {
    'deit':  'inline_deit_ds.py',
    'swin':  'inline_swin_ds.py',
    'pvt':   'inline_pvt_ds.py',
    'cswin': 'inline_cswin_ds.py',
}
############### 4. Write YAML Config (always overwritten with correct values) ################
dataset_cfg = {
    'cifar10':      {'classes': 10,  'img_size': 32,  'patch': 4,
                     'epochs': 100, 'batch': 128},
    'cifar100':     {'classes': 100, 'img_size': 32,  'patch': 4,
                     'epochs': 100, 'batch': 128},
    'svhn':         {'classes': 10,  'img_size': 32,  'patch': 4,
                     'epochs': 100, 'batch': 128},
    'tinyimagenet': {'classes': 200, 'img_size': 64,  'patch': 16,
                     'epochs': 100, 'batch': 128},
    'food101':      {'classes': 101, 'img_size':224, 'patch': 16,
                     'epochs': 100, 'batch': 128},
}
cfg = dict(dataset_cfg[DATASET])

_is_ds_model = '_ds_' in MODEL_TYPE
_attn_type   = 'DDDD' if _is_ds_model else 'IIII'

if MODEL_FAMILY in ('swin', 'cswin') and DATASET in ('cifar10', 'cifar100', 'svhn', 'tinyimagenet'):
    cfg['patch'] = 4

_patch_grid = max(1, int(cfg['img_size']) // max(1, int(cfg['patch'])))
_stage_grids = [max(1, _patch_grid // (2 ** i)) for i in range(4)]

_default_swin_window = 7
if MODEL_FAMILY == 'swin':
    _valid_windows = [d for d in range(min(_default_swin_window, _patch_grid), 0, -1)
                      if _patch_grid % d == 0]
    _swin_window_size = _valid_windows[0] if _valid_windows else min(_default_swin_window, _patch_grid)
else:
    _swin_window_size = _default_swin_window

if MODEL_FAMILY == 'cswin':
    _cswin_split_size = '-'.join(str(max(1, min(7, g))) for g in _stage_grids)
else:
    _cswin_split_size = '56-28-14-7'

_HIER_SMALL_DATA_FIX = MODEL_FAMILY in ('swin', 'cswin') and DATASET in ('cifar100', 'tinyimagenet')
_BASE_LR       = 0.004 if _HIER_SMALL_DATA_FIX else 0.005
_WARMUP_LR     = 0.00005 if _HIER_SMALL_DATA_FIX else 0.000004
_MIN_LR        = 0.0004 if _HIER_SMALL_DATA_FIX else 0.00004
_WEIGHT_DECAY  = 0.05 if _HIER_SMALL_DATA_FIX else 0.1
_WARMUP_EPOCHS = 10 if _HIER_SMALL_DATA_FIX else 3
_MIXUP         = 0.1
_CUTMIX        = 0.1
_AUTO_AUGMENT  = 'rand-m5-mstd0.5-inc1' #Options: 'rand-m9-mstd0.5-inc1'
_REPROB        = 0.0

config_yaml = f"""DATA:
  IMG_SIZE: {cfg['img_size']}
  BATCH_SIZE: {cfg['batch']}
  DATASET: '{DATASET}'
  DATA_PATH: '{DATA_DIR}'
  NUM_WORKERS: 4
  PIN_MEMORY: true
  INTERPOLATION: 'bicubic'

TRAIN:
  WEIGHT_DECAY: {_WEIGHT_DECAY:.8f}
  BASE_LR: {_BASE_LR:.8f}
  WARMUP_LR: {_WARMUP_LR:.8f}
  MIN_LR: {_MIN_LR:.8f}
  CLIP_GRAD: 1.0
  EPOCHS: {cfg['epochs']}
  WARMUP_EPOCHS: {_WARMUP_EPOCHS}
  AUTO_RESUME: false
  USE_CHECKPOINT: false
  LR_SCHEDULER:
    NAME: 'cosine'
    DECAY_EPOCHS: 30
    DECAY_RATE: 0.1
  OPTIMIZER:
    NAME: 'adamw'
    EPS: 1.0e-8
    BETAS: [0.9, 0.999]
    MOMENTUM: 0.9

AUG:
  MIXUP: {_MIXUP}
  CUTMIX: {_CUTMIX}
  MIXUP_PROB: 1.0
  MIXUP_SWITCH_PROB: 0.5
  MIXUP_MODE: 'batch'
  COLOR_JITTER: 0.4
  AUTO_AUGMENT: '{_AUTO_AUGMENT}'
  REPROB: {_REPROB}
  REMODE: 'pixel'
  RECOUNT: 1
  CUTMIX_MINMAX:

MODEL:
  TYPE: '{MODEL_TYPE}'
  NAME: '{MODEL_TYPE}'
  RESUME: ''
  NUM_CLASSES: {cfg['classes']}
  DROP_RATE: 0.0
  DROP_PATH_RATE: 0.1
  LABEL_SMOOTHING: 0.1
  SWIN:
    PATCH_SIZE: {cfg['patch']}
    WINDOW_SIZE: {_swin_window_size}
  INLINE:
    ATTN_TYPE: '{_attn_type}'
    PVT_LA_SR_RATIOS: 1111
    CSWIN_LA_SPLIT_SIZE: '{_cswin_split_size}'

TEST:
  CROP: true

AMP: false
OUTPUT: '{LOCAL_OUTPUT_DIR}'
SAVE_FREQ: 10
PRINT_FREQ: 10
SEED: 0
TAG: '{DATASET}_{MODEL_TYPE}'
EVAL_MODE: false
THROUGHPUT_MODE: false
LOCAL_RANK: 0
"""

config_path = os.path.join(repo_path, 'custom_config.yaml')
with open(config_path, 'w') as f:
    f.write(config_yaml)

import yaml as _yaml
with open(config_path) as _f:
    _v = _yaml.safe_load(_f)

for _num_key in ['WEIGHT_DECAY', 'BASE_LR', 'WARMUP_LR', 'MIN_LR', 'CLIP_GRAD']:
    _v['TRAIN'][_num_key] = float(_v['TRAIN'][_num_key])
try:
    _v['TRAIN']['OPTIMIZER']['EPS'] = float(_v['TRAIN']['OPTIMIZER']['EPS'])
except Exception:
    pass

_SCALE = 128 / 512

assert abs(_v['TRAIN']['MIN_LR']  - _MIN_LR) < 1e-12, f"MIN_LR wrong: {_v['TRAIN']['MIN_LR']}"
assert abs(_v['TRAIN']['BASE_LR'] - _BASE_LR) < 1e-12, f"BASE_LR wrong: {_v['TRAIN']['BASE_LR']}"
assert _v['AUG']['MIXUP'] == _MIXUP,                f"MIXUP wrong: {_v['AUG']['MIXUP']}"
assert _v['AUG']['CUTMIX'] == _CUTMIX,              f"CUTMIX wrong: {_v['AUG']['CUTMIX']}"
assert _v['AUG']['REPROB'] == _REPROB,              f"REPROB wrong: {_v['AUG']['REPROB']}"
assert abs(_v['TRAIN']['WEIGHT_DECAY'] - _WEIGHT_DECAY) < 1e-12, f"WEIGHT_DECAY wrong: {_v['TRAIN']['WEIGHT_DECAY']}"
assert _v['TRAIN']['CLIP_GRAD'] == 1.0,             f"CLIP_GRAD wrong: {_v['TRAIN']['CLIP_GRAD']}"
assert _v['TRAIN']['WARMUP_EPOCHS'] == _WARMUP_EPOCHS, f"WARMUP_EPOCHS wrong: {_v['TRAIN']['WARMUP_EPOCHS']}"
assert _v['TRAIN']['EPOCHS'] == cfg['epochs'],      f"EPOCHS wrong: {_v['TRAIN']['EPOCHS']}"
assert _v['MODEL']['DROP_PATH_RATE'] == 0.1,       f"DROP_PATH wrong: {_v['MODEL']['DROP_PATH_RATE']}"
assert _v['MODEL']['LABEL_SMOOTHING'] == 0.1,       f"LABEL_SMOOTHING wrong: {_v['MODEL']['LABEL_SMOOTHING']}"
assert _v['AMP'] is False,                           f"AMP wrong: {_v['AMP']}"
assert 'S' not in str(_v['MODEL']['INLINE']['ATTN_TYPE']).upper(), \
    f"ATTN_TYPE must not contain 'S' (softmax path): {_v['MODEL']['INLINE']['ATTN_TYPE']}"

print(f"✓ Config written: {config_path}")
print(f"  MODEL_TYPE  : {MODEL_TYPE}  (family={MODEL_FAMILY})")
print(f"  ATTN_TYPE   : {_attn_type}  (softmax-free: no 'S' path)")
print(f"  NUM_CLASSES : {cfg['classes']}")
print(f"  PATCH_SIZE  : {cfg['patch']}  ({cfg['img_size']//cfg['patch']}×{cfg['img_size']//cfg['patch']} grid)")
print(f"  SWIN WINDOW : {_swin_window_size}")
print(f"  CSWIN SPLIT : {_cswin_split_size}")
print(f"  BASE_LR     : {_v['TRAIN']['BASE_LR']} in YAML → {_v['TRAIN']['BASE_LR']*_SCALE:.2e} effective (after /4 scaling)")
print(f"  MIN_LR      : {_v['TRAIN']['MIN_LR']} in YAML → {_v['TRAIN']['MIN_LR']*_SCALE:.2e} effective")
print(f"  WEIGHT_DECAY: {_v['TRAIN']['WEIGHT_DECAY']}")
print(f"  WARMUP_EPOCHS: {_v['TRAIN']['WARMUP_EPOCHS']}")
print(f"  CLIP_GRAD   : {_v['TRAIN']['CLIP_GRAD']}")
print(f"  EPOCHS      : {_v['TRAIN']['EPOCHS']}")
print(f"  MIXUP/CUTMIX: {_v['AUG']['MIXUP']} / {_v['AUG']['CUTMIX']}")
print(f"  AUTO_AUGMENT: {_v['AUG']['AUTO_AUGMENT']}")
print(f"  REPROB      : {_v['AUG']['REPROB']}")
print(f"  AMP          : {_v['AMP']}")

############### 5. Write main_colab.py Wrapper ################
_FAMILY_BUILD_KWARGS = {
    'deit':  "",
    'swin':  f"attn_type='DDDD', patch_size={cfg['patch']}, window_size={_swin_window_size}, ",
    'cswin': f"attn_type='DDDD', la_split_size='{_cswin_split_size}', ",
    'pvt':   "attn_type='DDDD', la_sr_ratios='1111', ",
}
_fallback_kwargs_str = _FAMILY_BUILD_KWARGS[MODEL_FAMILY]

wrapper_code = f'''#!/usr/bin/env python
"""Single-GPU training wrapper for InLine^D-S on Google Colab.

Imports the model module selected in section 3 ({MODEL_MODULE}) so the timm
registry contains {MODEL_TYPE!r}.  build_model is taken from build_ds; if that
fails for any reason we fall back to timm.create_model with the right
family-specific kwargs.  Attention stays purely InLine (attn_type='DDDD'); no
softmax path is ever requested.
"""
import os, sys, traceback
os.environ["RANK"]        = "0"
os.environ["WORLD_SIZE"]  = "1"
os.environ["LOCAL_RANK"]  = "0"
os.environ["MASTER_ADDR"] = "localhost"
os.environ["MASTER_PORT"] = "12355"
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))

try:
    import torch
    import torch.nn as nn
    import main as main_module

    import {MODEL_MODULE}   # noqa: F401

    from models import build_model as _orig_build_model

    def _patch_nan_hook(model):
        """Zero out NaN/Inf gradients before each optimizer step."""
        def hook(grad):
            if grad is not None:
                grad = torch.where(torch.isnan(grad) | torch.isinf(grad),
                                   torch.zeros_like(grad), grad)
            return grad
        for p in model.parameters():
            if p.requires_grad:
                p.register_hook(hook)

    def _fallback_build(config):
        """Build the model directly from the timm registry if build_ds fails."""
        import timm
        _family = {MODEL_FAMILY!r}
        _kw = dict(
            num_classes=config.MODEL.NUM_CLASSES,
            img_size=config.DATA.IMG_SIZE,
            drop_path_rate=config.MODEL.DROP_PATH_RATE,
        )
        if _family == 'swin':
            _kw.update(attn_type='DDDD', patch_size={cfg['patch']}, window_size={_swin_window_size})
        elif _family == 'cswin':
            _kw.update(attn_type='DDDD', la_split_size={_cswin_split_size!r})
        elif _family == 'pvt':
            _kw.update(attn_type='DDDD', la_sr_ratios='1111')
        try:
            return timm.create_model(config.MODEL.TYPE, **_kw)
        except TypeError:
            if _family == 'swin':
                _kw.pop('patch_size', None); _kw.pop('window_size', None)
                return timm.create_model(config.MODEL.TYPE, **_kw)
            if _family == 'cswin' and 'la_split_size' in _kw:
                _kw['split_size'] = _kw.pop('la_split_size')
                return timm.create_model(config.MODEL.TYPE, **_kw)
            raise

    def _patched_build_model(config):
        """Build model (build_ds first, timm fallback) and fix the head."""
        try:
            model = _orig_build_model(config)
        except Exception as _build_err:
            print(f"[BUILD] build_ds.build_model failed ({{type(_build_err).__name__}}: "
                  f"{{_build_err}}); falling back to timm.create_model", flush=True)
            model = _fallback_build(config)

        num_classes = config.MODEL.NUM_CLASSES
        if hasattr(model, "head") and isinstance(model.head, nn.Linear):
            if model.head.out_features != num_classes:
                in_features = model.head.in_features
                model.head  = nn.Linear(in_features, num_classes)
                print(f"[HEAD-FIX] Replaced classification head: "
                      f"{{in_features}} → {{num_classes}} classes", flush=True)
        _patch_nan_hook(model)
        return model

    main_module.build_model = _patched_build_model
    main_module._patch_nan_hook = _patch_nan_hook

    if __name__ == "__main__":
        main_module.main()

except Exception as e:
    print(f"\\n❌ FATAL ERROR in main_colab.py:", file=sys.stderr)
    print(f"   {{type(e).__name__}}: {{e}}", file=sys.stderr)
    traceback.print_exc(file=sys.stderr)
    sys.exit(1)
'''

wrapper_path = os.path.join(repo_path, 'main_colab.py')
with open(wrapper_path, 'w') as f:
    f.write(wrapper_code)
print(f"✓ main_colab.py written (imports {MODEL_MODULE}, family={MODEL_FAMILY})")

############### 6. Patch utils.py for PyTorch 2.x compatibility ################
utils_path = os.path.join(repo_path, 'utils.py')
with open(utils_path, 'r') as f:
    utils_code = f.read()

old_load = "checkpoint = torch.load(config.MODEL.RESUME, map_location='cpu')"
new_load = "checkpoint = torch.load(config.MODEL.RESUME, map_location='cpu', weights_only=False)"

if old_load in utils_code:
    utils_code = utils_code.replace(old_load, new_load)
    with open(utils_path, 'w') as f:
        f.write(utils_code)
    print("✓ utils.py patched: torch.load weights_only=False")
elif "weights_only=False" in utils_code:
    print("✓ utils.py already patched")
else:
    print("⚠ torch.load line not found in utils.py — check manually")

############### 7. Smoke-Test: Import & Config Check ################
import sys
import argparse
sys.path.insert(0, repo_path)
os.environ.update({
    "RANK": "0", "WORLD_SIZE": "1", "LOCAL_RANK": "0",
    "MASTER_ADDR": "localhost", "MASTER_PORT": "12355",
})

print("\nRunning smoke-test imports...")
try:
    import importlib
    from config import get_config
    importlib.import_module(MODEL_MODULE)

    args = argparse.Namespace(
        cfg              = config_path,
        opts             = [
            'MODEL.SWIN.PATCH_SIZE', str(cfg['patch']),
            'MODEL.NUM_CLASSES',     str(cfg['classes']),
        ],
        local_rank       = 0,
        rank             = 0,
        world_size       = 1,
        dist_url         = 'env://',
        amp              = True,
        output           = LOCAL_OUTPUT_DIR,
        tag              = f'{DATASET}_{MODEL_TYPE}',
        resume           = '',
        eval             = False,
        throughput       = False,
        seed             = 0,
        batch_size       = None,
        lr               = None,
        min_lr           = None,
        weight_decay     = None,
        data_path        = None,
        zip              = False,
        cache_mode       = None,
        accumulation_steps = 1,
        num_classes      = None,
        use_checkpoint   = False,
    )

    config = get_config(args)
    print(f"✓ Config loaded")
    print(f"  Dataset    : {config.DATA.DATASET}")
    print(f"  Model      : {config.MODEL.TYPE}  (family={MODEL_FAMILY})")
    print(f"  Classes    : {config.MODEL.NUM_CLASSES}")
    print(f"  Image size : {config.DATA.IMG_SIZE}")
    print(f"  Mixup      : {config.AUG.MIXUP}")
    print(f"  ATTN_TYPE  : {config.MODEL.INLINE.ATTN_TYPE}")

    import torch.nn as nn
    from models import build_model as _raw_build_model

    def _smoke_fallback_build(config):
        import timm
        _kw = dict(num_classes=config.MODEL.NUM_CLASSES,
                   img_size=config.DATA.IMG_SIZE,
                   drop_path_rate=config.MODEL.DROP_PATH_RATE)
        if MODEL_FAMILY == 'swin':
            _kw['attn_type'] = 'DDDD'; _kw['patch_size'] = cfg['patch']; _kw['window_size'] = _swin_window_size
        elif MODEL_FAMILY == 'cswin':
            _kw['attn_type'] = 'DDDD'; _kw['la_split_size'] = _cswin_split_size
        elif MODEL_FAMILY == 'pvt':
            _kw['attn_type'] = 'DDDD'; _kw['la_sr_ratios'] = '1111'
        try:
            return timm.create_model(config.MODEL.TYPE, **_kw)
        except TypeError:
            if MODEL_FAMILY == 'swin':
                _kw.pop('patch_size', None); _kw.pop('window_size', None)
                return timm.create_model(config.MODEL.TYPE, **_kw)
            if MODEL_FAMILY == 'cswin' and 'la_split_size' in _kw:
                _kw['split_size'] = _kw.pop('la_split_size')
                return timm.create_model(config.MODEL.TYPE, **_kw)
            raise

    def _smoke_build_model(config):
        try:
            model = _raw_build_model(config)
        except Exception as _e:
            print(f"  ⚠ build_ds failed ({type(_e).__name__}: {_e}); using timm fallback")
            model = _smoke_fallback_build(config)
        num_classes = config.MODEL.NUM_CLASSES
        if hasattr(model, "head") and isinstance(model.head, nn.Linear):
            if model.head.out_features != num_classes:
                model.head = nn.Linear(model.head.in_features, num_classes)
        return model
    model = _smoke_build_model(config)

    n_params = sum(p.numel() for p in model.parameters()) / 1e6
    print(f"✓ Model built: {type(model).__name__}  ({n_params:.1f}M params)")

    assert model.head.out_features == cfg['classes'], \
        f"Head mismatch: {model.head.out_features} != {cfg['classes']}"
    print(f"✓ Head: {model.head.in_features} → {model.head.out_features}")

    import torch
    model.eval()
    dummy = torch.randn(2, 3, cfg['img_size'], cfg['img_size'])
    with torch.no_grad():
        out = model(dummy)
    assert out.shape == (2, cfg['classes']), f"Output shape wrong: {out.shape}"
    print(f"✓ Forward pass: input {tuple(dummy.shape)} → output {tuple(out.shape)}")

    _softmax_hit = {'n': 0}
    _orig_sm, _orig_F_sm = torch.softmax, torch.nn.functional.softmax
    def _trap_softmax(*a, **k):
        _softmax_hit['n'] += 1
        return _orig_sm(*a, **k)
    def _trap_F_softmax(*a, **k):
        _softmax_hit['n'] += 1
        return _orig_F_sm(*a, **k)
    torch.softmax = _trap_softmax
    torch.nn.functional.softmax = _trap_F_softmax
    try:
        with torch.no_grad():
            _ = model(dummy)
    finally:
        torch.softmax = _orig_sm
        torch.nn.functional.softmax = _orig_F_sm
    if _softmax_hit['n'] == 0:
        print("✓ Softmax-free: no softmax call fired during a forward pass")
    else:
        print(f"⚠ softmax fired {_softmax_hit['n']}× — check ATTN_TYPE and the DS patches")

    if MODEL_FAMILY == 'deit':
        print(f"  Token grid : {cfg['img_size']//cfg['patch']}×{cfg['img_size']//cfg['patch']} "
              f"(+1 CLS) for patch size {cfg['patch']}")
    else:
        print(f"  ({MODEL_FAMILY} is hierarchical/pyramidal; patch grid varies per stage)")
    print("\n🎉 All checks passed — ready to train!")

except Exception as e:
    import traceback
    print(f"✗ Smoke-test failed: {type(e).__name__}: {e}")
    traceback.print_exc()

############### 8. Train ################
import subprocess
import threading

output_dir = LOCAL_OUTPUT_DIR
drive_output_dir = DRIVE_OUTPUT_DIR
os.makedirs(output_dir, exist_ok=True)
os.makedirs(drive_output_dir, exist_ok=True)

def _sync_training_outputs_to_drive(reason='epoch'):
    """Copy locally written logs/checkpoints to the Google Drive output path."""
    local_subdir = os.path.join(output_dir, MODEL_TYPE, f'{DATASET}_{MODEL_TYPE}')
    drive_subdir = os.path.join(drive_output_dir, MODEL_TYPE, f'{DATASET}_{MODEL_TYPE}')
    if not os.path.exists(local_subdir):
        return
    os.makedirs(drive_subdir, exist_ok=True)
    try:
        shutil.copytree(local_subdir, drive_subdir, dirs_exist_ok=True)
        print(f"✓ Synced {reason} logs/checkpoints to Drive: {drive_subdir}", flush=True)
    except Exception as _sync_e:
        print(f"⚠ Could not sync {reason} outputs to Drive: {_sync_e}", flush=True)

cmd = [
    sys.executable, '-u', 'main_colab.py',
    '--cfg',    config_path,
    '--output', output_dir,
    '--tag',    f'{DATASET}_{MODEL_TYPE}',
    '--local-rank', '0',
    '--opts',
        'MODEL.NUM_CLASSES',     str(cfg['classes']),
        'MODEL.SWIN.PATCH_SIZE', str(cfg['patch']),
]

print(f"\n{'='*62}")
print(f"Training  :  {MODEL_TYPE}  (family={MODEL_FAMILY})")
print(f"Dataset   :  {DATASET}  ({cfg['classes']} classes, {cfg['img_size']}px)")
print(f"Patch size:  {cfg['patch']}  → {cfg['img_size']//cfg['patch']}×{cfg['img_size']//cfg['patch']} token grid")
print(f"Epochs    :  {cfg['epochs']}")
print(f"Config    :  {config_path}")
print(f"{'='*62}\n")
sys.stdout.flush()

_all_train_lines = []
stderr_lines = []

import time as _time
import json as _json
import csv as _csv

_epoch_time_records = []
_current_epoch_id = None
_current_epoch_start = None
_train_wall_start = _time.perf_counter()

def _finish_epoch_timer(stop_time=None):
    global _current_epoch_id, _current_epoch_start, _epoch_time_records
    if stop_time is None:
        stop_time = _time.perf_counter()
    if _current_epoch_id is not None and _current_epoch_start is not None:
        _duration = max(0.0, float(stop_time - _current_epoch_start))
        _epoch_time_records.append({
            'epoch': int(_current_epoch_id),
            'wall_seconds': _duration,
            'wall_minutes': _duration / 60.0,
            'cumulative_seconds': float(stop_time - _train_wall_start),
            'cumulative_minutes': float(stop_time - _train_wall_start) / 60.0,
        })
    _current_epoch_id = None
    _current_epoch_start = None

def _stream_stderr(pipe):
    for line in iter(pipe.readline, ''):
        print(line, end='', flush=True)
        stderr_lines.append(line)
    pipe.close()

try:
    process = subprocess.Popen(
        cmd,
        cwd   = repo_path,
        stdout = subprocess.PIPE,
        stderr = subprocess.PIPE,
        text   = True,
        bufsize = 1,
        env    = {**os.environ, 'PYTHONUNBUFFERED': '1'},
    )

    stderr_thread = threading.Thread(
        target=_stream_stderr, args=(process.stderr,), daemon=True)
    stderr_thread.start()

    for line in iter(process.stdout.readline, ''):
        print(line, end='', flush=True)
        _all_train_lines.append(line)

        _epoch_match = re.search(r'EPOCH\s+(\d+)\s+training', line, flags=re.I)
        if _epoch_match:
            _now = _time.perf_counter()
            _finish_epoch_timer(stop_time=_now)
            _current_epoch_id = int(_epoch_match.group(1))
            _current_epoch_start = _now

        if 'Max accuracy:' in line:
            _sync_training_outputs_to_drive(reason=f"epoch {_current_epoch_id}")

    process.stdout.close()

    stderr_thread.join(timeout=10)
    return_code = process.wait(timeout=3600 * 4)

    _finish_epoch_timer()
    try:
        _epoch_time_dir = os.path.join(output_dir, MODEL_TYPE, f'{DATASET}_{MODEL_TYPE}')
        os.makedirs(_epoch_time_dir, exist_ok=True)
        _epoch_time_json = os.path.join(_epoch_time_dir, 'epoch_wall_times.json')
        _epoch_time_csv  = os.path.join(_epoch_time_dir, 'epoch_wall_times.csv')
        with open(_epoch_time_json, 'w') as _f:
            _json.dump(_epoch_time_records, _f, indent=2)
        with open(_epoch_time_csv, 'w', newline='') as _f:
            _writer = _csv.DictWriter(
                _f,
                fieldnames=['epoch', 'wall_seconds', 'wall_minutes',
                            'cumulative_seconds', 'cumulative_minutes']
            )
            _writer.writeheader()
            _writer.writerows(_epoch_time_records)
        print(f"✓ Epoch timing saved: {_epoch_time_csv}")
        _sync_training_outputs_to_drive(reason="epoch timing")
    except Exception as _e:
        print(f"⚠ Epoch timing could not be saved: {_e}")

    _sync_training_outputs_to_drive(reason="final training")
    if return_code == 0:
        print("\n✓ Training completed successfully!")
    else:
        print(f"\n✗ Training failed (exit code {return_code})")
        if stderr_lines:
            print("\n--- Last 50 lines of STDERR ---")
            print(''.join(stderr_lines[-50:]))
        print("\n💡 Checklist:")
        print("  1. Smoke-test (section 7) must pass before training")
        print(f"  2. {_FAMILY_FILE[MODEL_FAMILY]} (and optionally build_ds.py) must be in: {DS_SRC_DIR}")
        print(f"  3. DATA_PATH exists: {DATA_DIR}")
        print("  4. Reduce BATCH_SIZE if OOM (edit dataset_cfg above)")
        raise subprocess.CalledProcessError(return_code, cmd)

except subprocess.TimeoutExpired:
    process.kill()
    print("\n✗ Training timed out (4 h limit)")
    raise

############### 9. Evaluate Latest Checkpoint ################
import glob

output_subdir = os.path.join(output_dir, MODEL_TYPE, f'{DATASET}_{MODEL_TYPE}')
checkpoints   = sorted(glob.glob(os.path.join(output_subdir, 'ckpt_epoch_*.pth')))

if not checkpoints:
    print("No checkpoints found — run training first.")
else:
    latest_ckpt = checkpoints[-1]
    print(f"Evaluating: {latest_ckpt}\n")

    eval_cmd = [
        sys.executable, '-u', 'main_colab.py',
        '--cfg',    config_path,
        '--eval',
        '--resume', latest_ckpt,
        '--output', output_dir,
        '--tag',    f'{DATASET}_{MODEL_TYPE}',
        '--opts',
            'MODEL.NUM_CLASSES',     str(cfg['classes']),
            'MODEL.SWIN.PATCH_SIZE', str(cfg['patch']),
    ]

    stderr_lines = []

    eval_process = subprocess.Popen(
        eval_cmd,
        cwd    = repo_path,
        stdout = subprocess.PIPE,
        stderr = subprocess.PIPE,
        text   = True,
        bufsize = 1,
        env    = {**os.environ, 'PYTHONUNBUFFERED': '1'},
    )

    et = threading.Thread(
        target=_stream_stderr, args=(eval_process.stderr,), daemon=True)
    et.start()

    for line in iter(eval_process.stdout.readline, ''):
        print(line, end='', flush=True)
    eval_process.stdout.close()

    et.join(timeout=10)
    rc = eval_process.wait(timeout=600)
    _sync_training_outputs_to_drive(reason="evaluation")
    print("\n✓ Evaluation done!" if rc == 0
          else f"\n✗ Evaluation failed (exit code {rc})")

############### 10. Comprehensive Visualisations ################
# Works for DeiT, Swin, PVT, CSWin.
# It does NOT change training, hyperparameters, optimizer, data, or checkpoints.

print("\n" + "="*72)
print("Section 10: Comprehensive Visualisations")
print("="*72)

import os, re, math, glob, json, csv, time, warnings, itertools
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

SKD_SLA_TAG = ' (InLine-DS)'

def _skd_title(text):
    text = '' if text is None else str(text)
    return text if 'InLine-DS' in text else text + SKD_SLA_TAG

try:
    from matplotlib.axes import Axes as _MplAxes
    from matplotlib.figure import Figure as _MplFigure
    if not getattr(_MplAxes.set_title, '_skd_sla_patched', False):
        _orig_axes_set_title = _MplAxes.set_title
        def _skd_axes_set_title(self, label, *args, **kwargs):
            return _orig_axes_set_title(self, _skd_title(label), *args, **kwargs)
        _skd_axes_set_title._skd_sla_patched = True
        _MplAxes.set_title = _skd_axes_set_title
    if not getattr(_MplFigure.suptitle, '_skd_sla_patched', False):
        _orig_fig_suptitle = _MplFigure.suptitle
        def _skd_fig_suptitle(self, t, *args, **kwargs):
            return _orig_fig_suptitle(self, _skd_title(t), *args, **kwargs)
        _skd_fig_suptitle._skd_sla_patched = True
        _MplFigure.suptitle = _skd_fig_suptitle
except Exception as _e:
    print(f'  ⚠ InLine-DS title patch could not be applied: {_e}')

import matplotlib.cm as cm
import matplotlib.colors as mcolors
from matplotlib.patches import FancyArrowPatch
from matplotlib.collections import LineCollection
import matplotlib.gridspec as gridspec
warnings.filterwarnings('ignore')

try:
    import seaborn as sns
except ImportError:
    import subprocess as _sp
    _sp.run([sys.executable, '-m', 'pip', 'install', 'seaborn', '-q'], check=False)
    import seaborn as sns

try:
    import pandas as pd
except ImportError:
    import subprocess as _sp
    _sp.run([sys.executable, '-m', 'pip', 'install', 'pandas', '-q'], check=False)
    import pandas as pd

try:
    from sklearn.metrics import (
        roc_curve, auc, precision_recall_curve, average_precision_score,
        confusion_matrix, precision_recall_fscore_support, f1_score,
        accuracy_score, top_k_accuracy_score
    )
    from sklearn.preprocessing import label_binarize, MinMaxScaler
    from sklearn.decomposition import PCA
except ImportError:
    import subprocess as _sp
    _sp.run([sys.executable, '-m', 'pip', 'install', 'scikit-learn', '-q'], check=False)
    from sklearn.metrics import (
        roc_curve, auc, precision_recall_curve, average_precision_score,
        confusion_matrix, precision_recall_fscore_support, f1_score,
        accuracy_score, top_k_accuracy_score
    )
    from sklearn.preprocessing import label_binarize, MinMaxScaler
    from sklearn.decomposition import PCA

try:
    import networkx as nx
except ImportError:
    import subprocess as _sp
    _sp.run([sys.executable, '-m', 'pip', 'install', 'networkx', '-q'], check=False)
    import networkx as nx

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms, datasets

IS_DEIT  = (MODEL_FAMILY == 'deit')
IS_SWIN  = (MODEL_FAMILY == 'swin')
IS_PVT   = (MODEL_FAMILY == 'pvt')
IS_CSWIN = (MODEL_FAMILY == 'cswin')

VIZ_DIR = os.path.join(WORK_DIR, 'output', 'visualisations', f'{DATASET}_{MODEL_TYPE}')
os.makedirs(VIZ_DIR, exist_ok=True)
VIZ_MANIFEST = []

CIFAR10_CLASSES = [
    'airplane','automobile','bird','cat','deer','dog','frog','horse','ship','truck'
]
CIFAR100_CLASSES = [
    'apple','aquarium_fish','baby','bear','beaver','bed','bee','beetle',
    'bicycle','bottle','bowl','boy','bridge','bus','butterfly','camel',
    'can','castle','caterpillar','cattle','chair','chimpanzee','clock',
    'cloud','cockroach','couch','crab','crocodile','cup','dinosaur',
    'dolphin','elephant','flatfish','forest','fox','girl','hamster',
    'house','kangaroo','keyboard','lamp','lawn_mower','leopard','lion',
    'lizard','lobster','man','maple_tree','motorcycle','mountain',
    'mouse','mushroom','oak_tree','orange','orchid','otter','palm_tree',
    'pear','pickup_truck','pine_tree','plain','plate','poppy','porcupine',
    'possum','rabbit','raccoon','ray','road','rocket','rose','sea',
    'seal','shark','shrew','skunk','skyscraper','snail','snake','spider',
    'squirrel','streetcar','sunflower','sweet_pepper','table','tank',
    'telephone','television','tiger','tractor','train','trout','tulip',
    'turtle','wardrobe','whale','willow_tree','wolf','woman','worm'
]
NUM_CLASSES = cfg['classes']
if DATASET == 'cifar10':
    CLASS_NAMES = CIFAR10_CLASSES[:NUM_CLASSES]
elif DATASET == 'cifar100':
    CLASS_NAMES = CIFAR100_CLASSES[:NUM_CLASSES]
elif DATASET == 'svhn':
    CLASS_NAMES = [str(i) for i in range(10)]
else:
    CLASS_NAMES = [str(i) for i in range(NUM_CLASSES)]


def _register_viz(fname, title, category, description):
    VIZ_MANIFEST.append({
        'file': fname,
        'title': title,
        'category': category,
        'description': description,
        'path': os.path.join(VIZ_DIR, fname),
        'created_utc': time.strftime('%Y-%m-%d %H:%M:%S', time.gmtime()),
    })


def _savefig(fname, title=None, category='general', description='', tight=True):
    path = os.path.join(VIZ_DIR, fname)
    if tight:
        plt.tight_layout()
    plt.savefig(path, dpi=140, bbox_inches='tight')
    plt.close('all')
    _register_viz(fname, _skd_title(title or fname), category, description)
    print(f"  ✓ saved → {path}")
    return path


def _safe_div(a, b, eps=1e-12):
    return a / (b + eps)


def _to_numpy(x):
    if isinstance(x, torch.Tensor):
        return x.detach().cpu().numpy()
    return np.asarray(x)


def _unnormalize_img(img_tensor, mean, std):
    x = img_tensor.detach().cpu().clone()
    mean_t = torch.tensor(mean).view(3,1,1)
    std_t  = torch.tensor(std).view(3,1,1)
    x = x * std_t + mean_t
    return x.clamp(0, 1).permute(1,2,0).numpy()


def _select_class_indices(labels, probs, max_classes=12):
    if labels is None or probs is None:
        return []
    support = np.bincount(labels, minlength=NUM_CLASSES)
    conf = np.nan_to_num(probs.mean(axis=0), nan=0.0)
    score = support / max(1, support.max()) + conf / max(1e-12, conf.max())
    ids = np.argsort(score)[::-1][:min(max_classes, NUM_CLASSES)]
    return [int(i) for i in ids]

print("\n[10a] Parsing training log …")

_log_lines = _all_train_lines if '_all_train_lines' in dir() and _all_train_lines else []
output_subdir = os.path.join(output_dir, MODEL_TYPE, f'{DATASET}_{MODEL_TYPE}')
_log_file = os.path.join(output_subdir, 'log_rank0.txt')
if not _log_lines and os.path.exists(_log_file):
    with open(_log_file) as _f:
        _log_lines = _f.readlines()

train_epochs, train_losses, train_gnorms, train_lrs = [], [], [], []
test_epochs, test_acc1, test_acc5, test_losses = [], [], [], []
_cur_epoch = 0
_epoch_losses, _epoch_gnorms, _epoch_lrs = [], [], []
_test_loss_buf = []

for _line in _log_lines:
    _em = re.search(r'EPOCH\s+(\d+)\s+training', _line)
    if _em:
        _ep = int(_em.group(1))
        if _epoch_losses:
            train_epochs.append(_cur_epoch)
            train_losses.append(float(np.mean(_epoch_losses)))
            _gnorm_valid = [g for g in _epoch_gnorms if np.isfinite(g)]
            train_gnorms.append(float(np.mean(_gnorm_valid)) if _gnorm_valid else float('nan'))
            train_lrs.append(float(np.mean(_epoch_lrs)) if _epoch_lrs else float('nan'))
            _epoch_losses, _epoch_gnorms, _epoch_lrs = [], [], []
        _cur_epoch = _ep
        continue

    _tm = re.search(r'Train:.*loss\s+[\d.]+\s+\(([\d.]+)\).*grad_norm\s+[\d.inf]+\s+\(([\d.inf]+)\)', _line)
    if _tm:
        try:
            _epoch_losses.append(float(_tm.group(1)))
            _gn = _tm.group(2)
            _epoch_gnorms.append(float(_gn) if _gn != 'inf' else float('nan'))
        except ValueError:
            pass

    _lrm = re.search(r'lr\s+([0-9.eE+-]+)', _line)
    if _lrm:
        try:
            _epoch_lrs.append(float(_lrm.group(1)))
        except ValueError:
            pass

    _tl = re.search(r'Test:.*Loss\s+[\d.]+\s+\(([\d.]+)\).*Acc@1', _line)
    if _tl:
        try:
            _test_loss_buf.append(float(_tl.group(1)))
        except ValueError:
            pass

    _acc = re.search(r'\*\s+Acc@1\s+([\d.]+)\s+Acc@5\s+([\d.]+)', _line)
    if _acc:
        test_epochs.append(_cur_epoch)
        test_acc1.append(float(_acc.group(1)))
        test_acc5.append(float(_acc.group(2)))
        if _test_loss_buf:
            test_losses.append(float(np.mean(_test_loss_buf)))
            _test_loss_buf = []

if _epoch_losses and (not train_epochs or _cur_epoch != train_epochs[-1]):
    train_epochs.append(_cur_epoch)
    train_losses.append(float(np.mean(_epoch_losses)))
    _gnorm_valid = [g for g in _epoch_gnorms if np.isfinite(g)]
    train_gnorms.append(float(np.mean(_gnorm_valid)) if _gnorm_valid else float('nan'))
    train_lrs.append(float(np.mean(_epoch_lrs)) if _epoch_lrs else float('nan'))

print(f"  Parsed {len(train_epochs)} train epochs, {len(test_epochs)} eval epochs")

def _load_epoch_wall_times():
    records = []
    if '_epoch_time_records' in globals() and isinstance(_epoch_time_records, list):
        records.extend([dict(r) for r in _epoch_time_records if isinstance(r, dict)])
    candidate_files = [
        os.path.join(output_subdir, 'epoch_wall_times.csv'),
        os.path.join(output_subdir, 'epoch_wall_times.json'),
        os.path.join(output_dir, 'epoch_wall_times.csv'),
        os.path.join(output_dir, 'epoch_wall_times.json'),
    ]
    if not records:
        for _path in candidate_files:
            if not os.path.exists(_path):
                continue
            try:
                if _path.endswith('.csv'):
                    _df = pd.read_csv(_path)
                    records = _df.to_dict('records')
                else:
                    with open(_path, 'r') as _f:
                        records = json.load(_f)
                if records:
                    print(f"  ✓ loaded epoch timing: {_path}")
                    break
            except Exception as _e:
                print(f"  ⚠ could not read epoch timing {_path}: {_e}")
    clean = []
    for r in records:
        try:
            ep = int(r.get('epoch', len(clean)))
            sec = float(r.get('wall_seconds', r.get('seconds', np.nan)))
            if np.isfinite(sec) and sec >= 0:
                clean.append({
                    'epoch': ep,
                    'wall_seconds': sec,
                    'wall_minutes': sec / 60.0,
                    'cumulative_seconds': float(r.get('cumulative_seconds', np.nan)),
                    'cumulative_minutes': float(r.get('cumulative_minutes', np.nan)),
                })
        except Exception:
            pass
    clean = sorted(clean, key=lambda z: z['epoch'])
    if clean:
        cum = 0.0
        for r in clean:
            if not np.isfinite(r.get('cumulative_seconds', np.nan)):
                cum += r['wall_seconds']
                r['cumulative_seconds'] = cum
                r['cumulative_minutes'] = cum / 60.0
    return clean

_epoch_wall_time_records = _load_epoch_wall_times()
print(f"  Epoch timing records: {len(_epoch_wall_time_records)}")

print("\n[10b] Loading best checkpoint …")

sys.path.insert(0, repo_path)
os.environ.update({"RANK":"0", "WORLD_SIZE":"1", "LOCAL_RANK":"0", "MASTER_ADDR":"localhost", "MASTER_PORT":"12356"})

_ckpt_path = os.path.join(output_subdir, 'max_acc.pth')
if not os.path.exists(_ckpt_path):
    _ckpts = sorted(glob.glob(os.path.join(output_subdir, 'ckpt_epoch_*.pth')))
    _ckpt_path = _ckpts[-1] if _ckpts else None

_model_loaded = False
_viz_model = None
_device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


def _viz_fallback_build(config):
    import timm
    _kw = dict(num_classes=config.MODEL.NUM_CLASSES,
               img_size=config.DATA.IMG_SIZE,
               drop_path_rate=config.MODEL.DROP_PATH_RATE)
    if IS_SWIN:
        _kw['attn_type'] = 'DDDD'; _kw['patch_size'] = cfg['patch']; _kw['window_size'] = _swin_window_size
    elif IS_CSWIN:
        _kw['attn_type'] = 'DDDD'; _kw['la_split_size'] = _cswin_split_size
    elif IS_PVT:
        _kw['attn_type'] = 'DDDD'; _kw['la_sr_ratios'] = '1111'
    try:
        return timm.create_model(config.MODEL.TYPE, **_kw)
    except TypeError:
        if IS_SWIN:
            _kw.pop('patch_size', None); _kw.pop('window_size', None)
            return timm.create_model(config.MODEL.TYPE, **_kw)
        if IS_CSWIN and 'la_split_size' in _kw:
            _kw['split_size'] = _kw.pop('la_split_size')
            return timm.create_model(config.MODEL.TYPE, **_kw)
        raise


if _ckpt_path and os.path.exists(_ckpt_path):
    try:
        import importlib
        from config import get_config
        import argparse as _ap
        _args = _ap.Namespace(
            cfg=config_path,
            opts=['MODEL.SWIN.PATCH_SIZE', str(cfg['patch']), 'MODEL.NUM_CLASSES', str(cfg['classes'])],
            local_rank=0, rank=0, world_size=1, dist_url='env://', amp=True,
            output=output_dir, tag=f'{DATASET}_{MODEL_TYPE}', resume='', eval=False,
            throughput=False, seed=0, batch_size=None, lr=None, min_lr=None,
            weight_decay=None, data_path=None, zip=False, cache_mode=None,
            accumulation_steps=1, num_classes=None, use_checkpoint=False,
        )
        _config = get_config(_args)
        importlib.import_module(MODEL_MODULE)
        from models import build_model as _bm
        try:
            _viz_model = _bm(_config)
        except Exception as _be:
            print(f"  ⚠ build_ds failed ({type(_be).__name__}: {_be}); using timm fallback")
            _viz_model = _viz_fallback_build(_config)
        if hasattr(_viz_model, 'head') and isinstance(_viz_model.head, nn.Linear):
            if _viz_model.head.out_features != NUM_CLASSES:
                _viz_model.head = nn.Linear(_viz_model.head.in_features, NUM_CLASSES)
        _ckpt = torch.load(_ckpt_path, map_location='cpu', weights_only=False)
        _state = _ckpt.get('model', _ckpt)
        _missing, _unexpected = _viz_model.load_state_dict(_state, strict=False)
        _viz_model = _viz_model.to(_device).eval()
        _model_loaded = True
        print(f"  ✓ Checkpoint loaded: {_ckpt_path}")
        if _missing:
            print(f"  ⚠ Missing keys ignored: {len(_missing)}")
        if _unexpected:
            print(f"  ⚠ Unexpected keys ignored: {len(_unexpected)}")
    except Exception as _e:
        print(f"  ⚠ Could not load checkpoint: {_e}")
else:
    print("  ⚠ No checkpoint found — model-dependent plots will be skipped")

print("\n[10c] Building evaluation dataset …")

if DATASET in ['cifar10', 'cifar100']:
    _mean = (0.4914, 0.4822, 0.4465)
    _std  = (0.2470, 0.2435, 0.2616)
elif DATASET == 'svhn':
    _mean = (0.4377, 0.4438, 0.4728)
    _std  = (0.1980, 0.2010, 0.1970)
else:
    _mean = (0.485, 0.456, 0.406)
    _std  = (0.229, 0.224, 0.225)

def _build_viz_dataset():
    if DATASET == 'cifar10':
        tf = transforms.Compose([transforms.ToTensor(), transforms.Normalize(_mean, _std)])
        return datasets.CIFAR10(DATA_DIR, train=False, download=False, transform=tf)
    if DATASET == 'cifar100':
        tf = transforms.Compose([transforms.ToTensor(), transforms.Normalize(_mean, _std)])
        return datasets.CIFAR100(DATA_DIR, train=False, download=False, transform=tf)
    if DATASET == 'svhn':
        tf = transforms.Compose([transforms.ToTensor(), transforms.Normalize(_mean, _std)])
        return datasets.SVHN(DATA_DIR, split='test', download=False, transform=tf)
    if DATASET == 'food101':
        tf = transforms.Compose([
            transforms.Resize(int((256/224) * cfg['img_size'])), transforms.CenterCrop(cfg['img_size']),
            transforms.ToTensor(), transforms.Normalize(_mean, _std)
        ])

        class _Food101VizDataset(torch.utils.data.Dataset):
            def __init__(self, root, split='test', transform=None):
                from PIL import Image
                self.Image = Image
                self.root = root
                self.transform = transform
                self.images_dir = os.path.join(root, 'images')
                self.meta_dir = os.path.join(root, 'meta')
                classes_file = os.path.join(self.meta_dir, 'classes.txt')
                if os.path.exists(classes_file):
                    with open(classes_file, 'r') as f:
                        self.classes = [line.strip() for line in f if line.strip()]
                else:
                    self.classes = sorted([d for d in os.listdir(self.images_dir) if os.path.isdir(os.path.join(self.images_dir, d))])
                self.class_to_idx = {c: i for i, c in enumerate(self.classes)}
                split_file = os.path.join(self.meta_dir, split + '.txt')
                self.samples = []
                if os.path.exists(split_file):
                    with open(split_file, 'r') as f:
                        rels = [line.strip() for line in f if line.strip()]
                    for rel in rels:
                        cls = rel.split('/')[0]
                        img_path = os.path.join(self.images_dir, rel + '.jpg')
                        if cls in self.class_to_idx and os.path.exists(img_path):
                            self.samples.append((img_path, self.class_to_idx[cls]))
                else:
                    for cls in self.classes:
                        cls_dir = os.path.join(self.images_dir, cls)
                        for name in os.listdir(cls_dir):
                            if name.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp', '.webp')):
                                self.samples.append((os.path.join(cls_dir, name), self.class_to_idx[cls]))
                self.targets = [y for _, y in self.samples]
            def __len__(self):
                return len(self.samples)
            def __getitem__(self, index):
                path, target = self.samples[index]
                img = self.Image.open(path).convert('RGB')
                if self.transform is not None:
                    img = self.transform(img)
                return img, target

        def _resolve_food101_for_viz(base_dir):
            aliases = ['food101', 'food-101', 'Food101', 'Food-101']
            candidates = [os.path.join(base_dir, name) for name in aliases]
            candidates.append(base_dir)
            for root in candidates:
                if os.path.isdir(os.path.join(root, 'images')):
                    return root
            return None

        food_root = _resolve_food101_for_viz(DATA_DIR)
        if food_root is None:
            raise FileNotFoundError(f"Food-101 not found locally under {DATA_DIR}")
        ds = _Food101VizDataset(food_root, split='test', transform=tf)
        globals()['CLASS_NAMES'] = getattr(ds, 'classes', [str(i) for i in range(NUM_CLASSES)])
        return ds
    if DATASET == 'tinyimagenet':
        tf = transforms.Compose([transforms.Resize((cfg['img_size'], cfg['img_size'])), transforms.ToTensor(), transforms.Normalize(_mean, _std)])
        tiny_root = _resolve_tiny_root(DATA_DIR) if '_resolve_tiny_root' in globals() else os.path.join(DATA_DIR, 'tiny-imagenet-200')
        val_dir = os.path.join(tiny_root, 'val')
        test_dir = os.path.join(tiny_root, 'test')
        train_dir = os.path.join(tiny_root, 'train')
        def _has_class_dirs(root):
            return os.path.isdir(root) and any(os.path.isdir(os.path.join(root, d)) for d in os.listdir(root))
        root = val_dir if _has_class_dirs(val_dir) else (test_dir if _has_class_dirs(test_dir) else train_dir)
        ds = datasets.ImageFolder(root=root, transform=tf)
        globals()['CLASS_NAMES'] = getattr(ds, 'classes', [str(i) for i in range(NUM_CLASSES)])
        return ds
    raise ValueError(f"Visualization dataset builder does not support DATASET={DATASET}")

try:
    _val_ds = _build_viz_dataset()
    _val_loader = torch.utils.data.DataLoader(_val_ds, batch_size=128, shuffle=False, num_workers=2, pin_memory=True)
    print(f"  ✓ Eval samples: {len(_val_ds)}")
except Exception as _e:
    _val_ds, _val_loader = None, None
    print(f"  ⚠ Could not build eval dataset: {_e}")

print("\n[10d] Running inference and feature extraction …")

_all_preds, _all_labels, _all_probs, _all_logits, _all_features = None, None, None, None, None
_sample_imgs = None

if _model_loaded and _val_loader is not None:
    _preds_l, _labels_l, _probs_l, _logits_l, _features_l, _images_l = [], [], [], [], [], []
    _viz_img_bank_l, _viz_lbl_bank_l, _viz_pred_bank_l, _viz_conf_bank_l = [], [], [], []
    _MAX_VIZ_IMAGE_BANK = 512
    _viz_bank_count = 0
    with torch.no_grad():
        for _imgs, _lbls in _val_loader:
            _imgs = _imgs.to(_device, non_blocking=True)
            _logits = _viz_model(_imgs)
            # Output-only multiclass probability normalization for plots/metrics.
            # This is not an attention operation and does not change the softmax-free
            # InLine attention used inside the model.
            _z = _logits - _logits.max(dim=1, keepdim=True).values
            _probs = torch.exp(_z)
            _probs = _probs / _probs.sum(dim=1, keepdim=True).clamp_min(1e-12)
            _preds = _logits.argmax(dim=1)
            try:
                _feat = _viz_model.forward_features(_imgs)
                if isinstance(_feat, tuple):
                    _feat = _feat[0]
                if _feat.dim() > 2:
                    _feat = _feat.flatten(1)
            except Exception:
                _feat = _logits
            _preds_l.append(_preds.cpu())
            _labels_l.append(_lbls.cpu())
            _probs_l.append(_probs.cpu())
            _logits_l.append(_logits.cpu())
            _features_l.append(_feat.detach().cpu())
            if len(_images_l) < 4:
                _images_l.append(_imgs.detach().cpu())
            if _viz_bank_count < _MAX_VIZ_IMAGE_BANK:
                _take = min(int(_imgs.size(0)), _MAX_VIZ_IMAGE_BANK - _viz_bank_count)
                if _take > 0:
                    _viz_img_bank_l.append(_imgs[:_take].detach().cpu())
                    _viz_lbl_bank_l.append(_lbls[:_take].detach().cpu())
                    _viz_pred_bank_l.append(_preds[:_take].detach().cpu())
                    _viz_conf_bank_l.append(_probs[:_take].max(dim=1).values.detach().cpu())
                    _viz_bank_count += _take
    _all_preds = torch.cat(_preds_l).numpy()
    _all_labels = torch.cat(_labels_l).numpy()
    _all_probs = torch.cat(_probs_l).numpy()
    _all_logits = torch.cat(_logits_l).numpy()
    _all_features = torch.cat(_features_l).numpy()
    _sample_imgs = torch.cat(_images_l)[:32] if _images_l else None
    _viz_img_bank = torch.cat(_viz_img_bank_l) if _viz_img_bank_l else None
    _viz_lbl_bank = torch.cat(_viz_lbl_bank_l).numpy() if _viz_lbl_bank_l else None
    _viz_pred_bank = torch.cat(_viz_pred_bank_l).numpy() if _viz_pred_bank_l else None
    _viz_conf_bank = torch.cat(_viz_conf_bank_l).numpy() if _viz_conf_bank_l else None
    _val_acc = float((_all_preds == _all_labels).mean() * 100.0)
    print(f"  ✓ Inference done. Validation accuracy: {_val_acc:.2f}%")
else:
    _viz_img_bank = _viz_lbl_bank = _viz_pred_bank = _viz_conf_bank = None
    print("  ⚠ Skipping inference-dependent plots because model/dataset is unavailable")

# ════════════════════════════════════════════════════════════════════════
# Architecture-agnostic InLine-DS attention adapter (validated ~1e-6).
# Single source of truth for reconstructing explicit attention rows from any
# of the four backbones for visualisation only. Purely InLine (no softmax).
# ════════════════════════════════════════════════════════════════════════

def _is_ds_attn(m):
    return all(hasattr(m, a) for a in ('kernel_q', 'kernel_k', 'tfm_q', 'tfm_k', 'num_heads'))


def _ds_attn_modules(model):
    if model is None:
        return []
    return [(n, m) for n, m in model.named_modules() if _is_ds_attn(m)]


def _arch_of(m):
    if hasattr(m, 'im2cswin'):
        return 'cswin'
    if hasattr(m, 'q') and hasattr(m, 'kv'):
        return 'pvt'
    if hasattr(m, 'qkv'):
        return 'deit_swin'
    return 'unknown'


def _is_swin_attn(m):
    return hasattr(m, 'local_static')


def _rows_from_phi(phi_q, phi_k, normalization='subtractive', eps=1e-6):
    sim = torch.matmul(phi_q.float(), phi_k.float().transpose(-2, -1))
    if normalization == 'divisive':
        return sim / sim.sum(-1, keepdim=True).clamp_min(eps)
    if normalization == 'subtractive':
        N = sim.shape[-1]
        return sim - sim.mean(-1, keepdim=True) + 1.0 / float(N)
    raise ValueError(f"Unknown normalization: {normalization}")


def _scaled_feature_map(x, head_dim):
    return F.elu(x.float() * (float(head_dim) ** -0.5)) + 1.0


def _ds_rows_from_qk(m, q, k, normalization='subtractive', feature_map='mmsk'):
    """Reconstruct explicit attention rows [B,H,Nq,Nk] for a DS attention module."""
    if feature_map == 'scaled':
        hd = getattr(m, 'head_dim', q.shape[-1])
        phi_q = _scaled_feature_map(q, hd)
        phi_k = _scaled_feature_map(k, hd)
        return _rows_from_phi(phi_q, phi_k, normalization)
    if feature_map != 'mmsk':
        raise ValueError(f"Unknown feature_map: {feature_map}")

    q_low, q_high, gate_q = m.tfm_q(q)
    k_low, k_high, gate_k = m.tfm_k(k)
    pql, pkl = m.kernel_q(q_low), m.kernel_k(k_low)
    pqh, pkh = m.kernel_q(q_high), m.kernel_k(k_high)
    rl = _rows_from_phi(pql, pkl, normalization)
    rh = _rows_from_phi(pqh, pkh, normalization)

    def _stable_gate(gq, gk):
        gq = gq.clamp(1e-4, 1 - 1e-4)
        gk = gk.clamp(1e-4, 1 - 1e-4)
        return torch.sigmoid(0.5 * (torch.logit(gq) + torch.logit(gk)))

    gk = gate_k.mean(dim=2, keepdim=True) if gate_k.shape[2] != gate_q.shape[2] else gate_k
    gate = _stable_gate(gate_q, gk)
    return gate * rl + (1 - gate) * rh


def _ema_center(m, k):
    if hasattr(m, 'k_ema'):
        return k - m.k_ema.to(device=k.device, dtype=k.dtype)
    return k


def _extract_qkv(m, inputs):
    """Recreate (q, k, v) [B,H,N,D] exactly as the DS forward path prepares them."""
    a = _arch_of(m)
    if a == 'deit_swin':
        x = inputs[0]
        B, N, C = x.shape
        H = m.num_heads
        D = C // H
        qkv = m.qkv(x.float()).reshape(B, N, 3, H, D).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        if hasattr(m, '_apply_rpb_to_keys'):
            k = m._apply_rpb_to_keys(k, N)
        k = _ema_center(m, k)
        return q, k, v
    if a == 'pvt':
        x = inputs[0]; H = inputs[1]; W = inputs[2]
        B, N, C = x.shape
        nh = m.num_heads; D = m.head_dim
        xf = x.float()
        q = m.q(xf).reshape(B, N, nh, D).permute(0, 2, 1, 3)
        if m.sr_ratio > 1:
            x_ = xf.permute(0, 2, 1).reshape(B, C, H, W)
            x_ = m.sr(x_).reshape(B, C, -1).permute(0, 2, 1)
            x_ = m.norm(x_)
        else:
            x_ = xf
        kv = m.kv(x_).reshape(B, -1, 2, nh, D).permute(2, 0, 3, 1, 4)
        k, v = kv[0], kv[1]
        k = _ema_center(m, k)
        return q, k, v
    if a == 'cswin':
        qkv = inputs[0]
        q, k, v = qkv[0], qkv[1], qkv[2]
        qw = m.im2cswin(q); kw = m.im2cswin(k)
        vw, lepe = m.get_lepe(v, m.get_v)
        b, n, c = qw.shape
        nh = m.num_heads; D = m.head_dim
        qm = qw.reshape(b, n, nh, D).permute(0, 2, 1, 3).float()
        km = kw.reshape(b, n, nh, D).permute(0, 2, 1, 3).float()
        vm = vw.reshape(b, n, nh, D).permute(0, 2, 1, 3).float()
        km = _ema_center(m, km)
        return qm, km, vm
    raise ValueError(f"Unsupported arch: {a}")


def _capture_ds_inputs(model, imgs):
    """Forward once and capture the inputs each DS attention module receives."""
    captured = {}
    handles = []
    def _mk(nm):
        def _hook(mod, inp, out):
            captured[nm] = inp
        return _hook
    mods = _ds_attn_modules(model)
    for nm, mod in mods:
        handles.append(mod.register_forward_hook(_mk(nm)))
    was_training = model.training
    model.eval()
    try:
        with torch.no_grad():
            model(imgs.to(_device))
    finally:
        for h in handles:
            h.remove()
        model.train(was_training)
    return mods, captured


def _rows_to_entropy_prob(rows, eps=1e-12):
    r = rows.float()
    if bool((r < 0).any()):
        r = torch.clamp(r, min=0.0)
    mass = r.sum(dim=-1, keepdim=True)
    N = r.shape[-1]
    uniform = torch.full_like(r, 1.0 / float(N))
    return torch.where(mass > eps, r / mass.clamp_min(eps), uniform)


def _normalized_entropy_from_rows(rows):
    p = _rows_to_entropy_prob(rows)
    N = p.shape[-1]
    return -(p * torch.log(p.clamp_min(1e-12))).sum(dim=-1) / math.log(max(N, 2))


def _factor_grid(n_tokens, preferred_h=None, preferred_w=None):
    n_tokens = int(max(1, n_tokens))
    if preferred_h is not None and preferred_w is not None:
        preferred_h, preferred_w = int(preferred_h), int(preferred_w)
        if preferred_h > 0 and preferred_w > 0:
            ratio = preferred_h / max(1, preferred_w)
            best = (1, n_tokens, float('inf'))
            for h in range(1, int(math.sqrt(n_tokens)) + 1):
                if n_tokens % h == 0:
                    w = n_tokens // h
                    score = abs((h / max(1, w)) - ratio)
                    if score < best[2]:
                        best = (h, w, score)
                    score2 = abs((w / max(1, h)) - ratio)
                    if score2 < best[2]:
                        best = (w, h, score2)
            return best[0], best[1]
    g = int(round(math.sqrt(n_tokens)))
    if g * g == n_tokens:
        return g, g
    h = int(math.floor(math.sqrt(n_tokens)))
    while h > 1 and n_tokens % h != 0:
        h -= 1
    return h, int(math.ceil(n_tokens / max(1, h)))


def _grid_qk_of(m, inputs, Nq, Nk):
    a = _arch_of(m)
    if a == 'deit_swin' and not _is_swin_attn(m):
        prefix_q = int(getattr(m, 'prefix_tokens', 1))
        prefix_k = prefix_q
        qh, qw = _factor_grid(Nq - prefix_q)
        kh, kw = _factor_grid(Nk - prefix_k, qh, qw)
        return prefix_q, prefix_k, qh, qw, kh, kw
    if a == 'deit_swin' and _is_swin_attn(m):
        ws = getattr(m, 'window_size', (int(math.isqrt(max(1, Nq))),) * 2)
        if isinstance(ws, int):
            ws = (ws, ws)
        qh, qw = int(ws[0]), int(ws[1])
        kh, kw = _factor_grid(Nk, qh, qw)
        return 0, 0, qh, qw, kh, kw
    if a == 'pvt':
        H, W = int(inputs[1]), int(inputs[2])
        kh, kw = _factor_grid(Nk, H, W)
        return 0, 0, H, W, kh, kw
    if a == 'cswin':
        qh, qw = int(getattr(m, 'H_sp', 1)), int(getattr(m, 'W_sp', max(1, Nq)))
        kh, kw = _factor_grid(Nk, qh, qw)
        return 0, 0, qh, qw, kh, kw
    qh, qw = _factor_grid(Nq)
    kh, kw = _factor_grid(Nk, qh, qw)
    return 0, 0, qh, qw, kh, kw


def _build_qk_mask(Nq, Nk, prefix_q, prefix_k, qh, qw, kh, kw, kind, seed=0, device='cpu'):
    Nq, Nk = int(Nq), int(Nk)
    prefix_q, prefix_k = int(prefix_q), int(prefix_k)
    q_img = Nq - prefix_q
    k_img = Nk - prefix_k
    mask = torch.zeros(Nq, Nk, dtype=torch.bool)
    if kind not in ('local3x3', 'random'):
        return mask.to(device)
    if q_img <= 0 or k_img <= 0 or qh * qw != q_img or kh * kw != k_img:
        return mask.to(device)
    rng = torch.Generator().manual_seed(int(seed))
    key_pool = torch.arange(prefix_k, Nk, dtype=torch.long)
    for qi in range(prefix_q, Nq):
        qpos = qi - prefix_q
        qr, qc = divmod(qpos, qw)
        kr = min(kh - 1, max(0, int((qr + 0.5) * kh / max(1, qh))))
        kc = min(kw - 1, max(0, int((qc + 0.5) * kw / max(1, qw))))
        local = []
        for rr in range(max(0, kr - 1), min(kh, kr + 2)):
            for cc in range(max(0, kc - 1), min(kw, kc + 2)):
                local.append(prefix_k + rr * kw + cc)
        if kind == 'local3x3':
            chosen = local
        else:
            cnt = min(len(local), len(key_pool))
            chosen = key_pool[torch.randperm(len(key_pool), generator=rng)[:cnt]].tolist()
        if chosen:
            mask[qi, torch.tensor(chosen, dtype=torch.long)] = True
    return mask.to(device)


def _renorm_after_mask(rows, mask, eps=1e-6):
    if mask is None or not bool(mask.any()):
        return rows
    m = mask.view(1, 1, *mask.shape).to(rows.device)
    masked = rows.float().masked_fill(m, 0.0)
    denom = masked.sum(-1, keepdim=True)
    valid = (~m).float()
    uni = valid / valid.sum(-1, keepdim=True).clamp_min(1.0)
    return torch.where(denom.abs() > eps, masked / denom.clamp(min=eps), uni).to(rows.dtype)


def _ds_explicit_forward(m, inputs, kind=None, seed=0):
    a = _arch_of(m)
    q, k, v = _extract_qkv(m, inputs)
    rows = _ds_rows_from_qk(m, q, k, 'subtractive', 'mmsk')
    Nq, Nk = rows.shape[-2], rows.shape[-1]
    if kind in ('local3x3', 'random'):
        prefix_q, prefix_k, qh, qw, kh, kw = _grid_qk_of(m, inputs, Nq, Nk)
        mask = _build_qk_mask(Nq, Nk, prefix_q, prefix_k, qh, qw, kh, kw, kind, seed, rows.device)
        rows = _renorm_after_mask(rows, mask)
    sc = 1.0 / math.sqrt(max(1, Nk)) if (a == 'cswin' or (a == 'deit_swin' and _is_swin_attn(m))) else 1.0
    out = torch.matmul(rows.float(), v.float()) * sc
    _, q_high, _ = m.tfm_q(q)

    if a == 'deit_swin':
        x = inputs[0].float(); B, N, C = x.shape
        if hasattr(m, '_local_attn_residual'):
            out = out + m._local_attn_residual(x, v)
        if getattr(m, 'use_svg', False) and getattr(m, 'w_svg', None) is not None:
            e = q_high.pow(2).mean(-1, keepdim=True)
            if _is_swin_attn(m):
                arg = (m.w_svg.float().clamp(-2, 2) * e.clamp(max=50) + m.b_svg.float().clamp(-8, 8)).clamp(-12, 12)
            else:
                arg = m.w_svg.float() * e + m.b_svg.float()
            out = out * (1.0 + 0.25 * torch.sigmoid(arg))
        out = out.transpose(1, 2).reshape(B, N, C)
        out = torch.nan_to_num(out)
        out = m.proj(out)
        if hasattr(m, '_local_token_mix'):
            out = m._local_token_mix(out)
        return m.proj_drop(out).to(dtype=inputs[0].dtype)

    if a == 'pvt':
        x = inputs[0].float(); H, W = inputs[1], inputs[2]; B, N, C = x.shape
        v_local = m.kv(x)[..., C:].reshape(B, N, m.num_heads, m.head_dim).permute(0, 2, 1, 3)
        if hasattr(m, '_local_attn_residual'):
            out = out + m._local_attn_residual(x, v_local, H, W)
        if getattr(m, 'use_svg', False) and getattr(m, 'w_svg', None) is not None:
            e = q_high.pow(2).mean(-1, keepdim=True)
            out = out * (1.0 + 0.25 * torch.sigmoid(m.w_svg.float() * e + m.b_svg.float()))
        out = out.transpose(1, 2).reshape(B, N, C)
        out = m.proj(out)
        if hasattr(m, '_local_token_mix'):
            out = m._local_token_mix(out, H, W)
        return m.proj_drop(out).to(dtype=inputs[0].dtype)

    if a == 'cswin':
        from models.inline_cswin_ds import windows2img
        qkv = inputs[0]; x_mean = inputs[1]
        q0, k0, v0 = qkv[0], qkv[1], qkv[2]
        in_dtype = q0.dtype; Hh = W = m.resolution; B, L, C = q0.shape
        vw, lepe = m.get_lepe(v0, m.get_v)
        b, n, c = m.im2cswin(q0).shape
        if hasattr(m, '_local_attn_residual'):
            out = out + m._local_attn_residual(x_mean.float(), v)
        if getattr(m, 'use_svg', False) and getattr(m, 'w_svg', None) is not None:
            e = q_high.pow(2).mean(-1, keepdim=True).clamp(max=50.0)
            out = out * (1.0 + 0.25 * torch.sigmoid((m.w_svg.float().clamp(-2, 2) * e + m.b_svg.float().clamp(-8, 8)).clamp(-12, 12)))
        out = out.transpose(1, 2).reshape(b, n, c)
        out = out + m.lepe_gamma.float().clamp(0, 1) * lepe.float()
        out = torch.nan_to_num(out).clamp(-1e4, 1e4)
        out = windows2img(out, m.H_sp, m.W_sp, Hh, W).view(B, -1, C)
        return out.to(dtype=in_dtype)

    raise ValueError(f"Unsupported arch: {a}")


def _eval_with_attention_mask(model, loader, kind=None, max_batches=4, seed=0):
    import types
    if model is None or loader is None:
        return float('nan')
    mods = _ds_attn_modules(model)
    if not mods:
        return float('nan')
    originals = []
    for _nm, mod in mods:
        originals.append((mod, mod.forward))
        def _pf(self, *a, _k=kind, _s=seed, **kw):
            return _ds_explicit_forward(self, a, kind=_k, seed=_s)
        mod.forward = types.MethodType(_pf, mod)
    correct, total = 0, 0
    was_training = model.training
    model.eval()
    try:
        with torch.no_grad():
            for bi, (imgs, lbls) in enumerate(loader):
                if bi >= max_batches:
                    break
                imgs = imgs.to(_device, non_blocking=True)
                lbls = lbls.to(_device, non_blocking=True)
                logits = model(imgs)
                pred = logits.argmax(1)
                correct += int((pred == lbls).sum().item())
                total += int(lbls.numel())
    finally:
        for mod, f in originals:
            mod.forward = f
        model.train(was_training)
    return 100.0 * correct / max(1, total)


def _collect_generic_attention_rows(model, imgs, normalization='subtractive', feature_map='mmsk'):
    if model is None or imgs is None:
        return [], []
    mods, captured = _capture_ds_inputs(model, imgs)
    rows_list, names = [], []
    for nm, mod in mods:
        if nm not in captured:
            continue
        try:
            q, k, _v = _extract_qkv(mod, captured[nm])
            rows = _ds_rows_from_qk(mod, q, k, normalization, feature_map)
            rows_list.append(rows.detach().cpu())
            names.append(nm)
        except Exception as _e:
            print(f"  ⚠ Could not reconstruct rows for {nm}: {_e}")
    return rows_list, names


def _square_rows_only(rows_list):
    return [r for r in rows_list if r.dim() == 4 and r.shape[-1] == r.shape[-2]]

def _grid_size_from_tokens(n_tokens, prefix=1):
    patch_tokens = max(1, n_tokens - prefix)
    return int(round(math.sqrt(patch_tokens)))


def _patch_map_from_cls(vec, prefix=1):
    vec = np.asarray(vec)
    if vec.ndim > 1:
        vec = vec.reshape(-1)
    if len(vec) > prefix:
        vec = vec[prefix:]
    g = int(round(math.sqrt(len(vec))))
    if g * g != len(vec):
        g = int(math.floor(math.sqrt(len(vec))))
        vec = vec[:g*g]
    return vec.reshape(g, g)


def _evaluate_subset(model, loader, max_batches=3):
    if model is None or loader is None:
        return float('nan')
    correct, total = 0, 0
    was_training = model.training
    model.eval()
    with torch.no_grad():
        for bi, (imgs, lbls) in enumerate(loader):
            if bi >= max_batches:
                break
            imgs = imgs.to(_device, non_blocking=True)
            lbls = lbls.to(_device, non_blocking=True)
            logits = model(imgs)
            pred = logits.argmax(1)
            correct += int((pred == lbls).sum().item())
            total += int(lbls.numel())
    model.train(was_training)
    return correct / max(1, total)

# ════════════════════════════════════════════════════════════════════════
# Viz 1: Loss / Accuracy / LR / Gradient curves  (model-agnostic)
# ════════════════════════════════════════════════════════════════════════
print("\n[Viz 1] Loss / accuracy / optimization curves …")
if train_epochs or test_epochs:
    fig, axes = plt.subplots(2, 2, figsize=(16, 10))
    ax = axes[0,0]
    if train_epochs:
        ax.plot(train_epochs, train_losses, marker='o', lw=1.5, label='Train loss')
    if test_losses and test_epochs:
        ax.plot(test_epochs[:len(test_losses)], test_losses, marker='s', lw=1.5, label='Validation loss')
    ax.set_title('Loss Curves'); ax.set_xlabel('Epoch'); ax.set_ylabel('Loss'); ax.legend(); ax.grid(alpha=0.3)
    ax = axes[0,1]
    if test_epochs:
        ax.plot(test_epochs, test_acc1, marker='o', lw=1.5, label='Acc@1')
        ax.plot(test_epochs, test_acc5, marker='s', lw=1.5, label='Acc@5')
    ax.set_title('Validation Accuracy Curves'); ax.set_xlabel('Epoch'); ax.set_ylabel('Accuracy (%)'); ax.legend(); ax.grid(alpha=0.3)
    ax = axes[1,0]
    if train_epochs and any(np.isfinite(train_gnorms)):
        ax.plot(train_epochs, train_gnorms, marker='o', lw=1.5)
    ax.set_title('Gradient Norm Curve'); ax.set_xlabel('Epoch'); ax.set_ylabel('Grad norm'); ax.grid(alpha=0.3)
    ax = axes[1,1]
    if train_epochs and any(np.isfinite(train_lrs)):
        ax.plot(train_epochs, train_lrs, marker='o', lw=1.5)
    ax.set_title('Learning Rate Curve'); ax.set_xlabel('Epoch'); ax.set_ylabel('LR'); ax.grid(alpha=0.3)
    _savefig('01_loss_accuracy_optimization_curves.png', 'Loss/Accuracy/LR/Gradient Curves', 'training', 'Training loss, validation loss/accuracy, gradient norm, and LR curves parsed from logs.')
else:
    print('  ⚠ No train/eval log curves found')

print("\n[Viz 1b] Epoch training-time curves …")
if _epoch_wall_time_records:
    try:
        _te = np.array([r['epoch'] for r in _epoch_wall_time_records], dtype=int)
        _ts = np.array([r['wall_seconds'] for r in _epoch_wall_time_records], dtype=float)
        _tc = np.array([r['cumulative_seconds'] for r in _epoch_wall_time_records], dtype=float)
        fig, axes = plt.subplots(1, 3, figsize=(18, 5))
        axes[0].plot(_te, _ts / 60.0, marker='o', lw=1.8)
        axes[0].set_title('Wall-Clock Time per Epoch'); axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Minutes'); axes[0].grid(alpha=0.3)
        axes[1].plot(_te, _tc / 3600.0, marker='s', lw=1.8)
        axes[1].set_title('Cumulative Training Time'); axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Hours'); axes[1].grid(alpha=0.3)
        if test_epochs and test_acc1:
            _acc_x = np.asarray(test_epochs, dtype=int); _acc_y = np.asarray(test_acc1, dtype=float)
            _time_lookup = {int(e): float(c) / 3600.0 for e, c in zip(_te, _tc)}
            _acc_t = np.array([_time_lookup.get(int(e), np.nan) for e in _acc_x])
            _m = np.isfinite(_acc_t)
            if _m.any():
                axes[2].plot(_acc_t[_m], _acc_y[_m], marker='D', lw=1.8); axes[2].set_xlabel('Cumulative Training Time (hours)')
            else:
                axes[2].plot(_acc_x, _acc_y, marker='D', lw=1.8); axes[2].set_xlabel('Epoch')
            axes[2].set_ylabel('Validation Acc@1 (%)'); axes[2].set_title('Accuracy vs Training Time')
        else:
            axes[2].hist(_ts / 60.0, bins=min(20, max(5, len(_ts)//2)))
            axes[2].set_xlabel('Epoch Duration (minutes)'); axes[2].set_ylabel('Count'); axes[2].set_title('Epoch-Time Distribution')
        axes[2].grid(alpha=0.3)
        _savefig('01b_epoch_training_time_curves.png', 'Epoch Training-Time Curves', 'paper_runtime', 'Per-epoch wall time, cumulative training time, and accuracy-versus-time diagnostic.')
        _time_csv = os.path.join(VIZ_DIR, '01b_epoch_wall_times.csv')
        pd.DataFrame(_epoch_wall_time_records).to_csv(_time_csv, index=False)
        _register_viz('01b_epoch_wall_times.csv', 'Epoch Wall-Time CSV', 'paper_runtime', 'Raw per-epoch wall-clock timing records collected during training.')
        print(f"  ✓ epoch timing CSV → {_time_csv}")
    except Exception as _e:
        print(f"  ⚠ Epoch-time visualisation skipped: {_e}")
else:
    print('  ⚠ No epoch timing records found. Re-run training with this code to collect exact epoch times.')

# ════════════════════════════════════════════════════════════════════════
# Viz 2–10: classification / calibration / representation (model-agnostic)
# ════════════════════════════════════════════════════════════════════════

def _adaptive_conf_thresholds(conf, n=50):
    conf = np.asarray(conf, dtype=float)
    conf = conf[np.isfinite(conf)]
    if conf.size == 0:
        return np.linspace(0.0, 1.0, n)
    lo, hi = float(conf.min()), float(conf.max())
    if hi <= lo:
        return np.array([max(0.0, min(1.0, lo))])
    q = np.quantile(conf, np.linspace(0.0, 1.0, n))
    lin = np.linspace(lo, hi, n)
    vals = np.unique(np.clip(np.r_[0.0, q, lin, hi], 0.0, 1.0))
    return vals


def _adaptive_conf_bins(conf, n_bins=10):
    conf = np.asarray(conf, dtype=float)
    conf = conf[np.isfinite(conf)]
    if conf.size == 0:
        return np.linspace(0.0, 1.0, n_bins + 1)
    lo, hi = float(conf.min()), float(conf.max())
    if hi <= lo:
        pad = max(1e-3, abs(lo) * 0.05)
    else:
        pad = max(1e-3, (hi - lo) * 0.05)
    start = max(0.0, lo - pad)
    end = min(1.0, hi + pad)
    if end <= start:
        end = min(1.0, start + 1e-2)
    return np.linspace(start, end, n_bins + 1)


def _padded_limits(vals, lo_bound=0.0, hi_bound=1.0, frac=0.06):
    vals = np.asarray(vals, dtype=float)
    vals = vals[np.isfinite(vals)]
    if vals.size == 0:
        return lo_bound, hi_bound
    lo, hi = float(vals.min()), float(vals.max())
    pad = max((hi - lo) * frac, 1e-4)
    return max(lo_bound, lo - pad), min(hi_bound, hi + pad)

if _all_probs is not None:
    print("\n[Viz 2] ROC Curve & AUC …")
    _Y = label_binarize(_all_labels, classes=np.arange(NUM_CLASSES))
    fig, ax = plt.subplots(figsize=(9, 7))
    try:
        fpr_micro, tpr_micro, _ = roc_curve(_Y.ravel(), _all_probs.ravel())
        auc_micro = auc(fpr_micro, tpr_micro)
        ax.plot(fpr_micro, tpr_micro, lw=2, label=f'Micro-average AUC={auc_micro:.3f}')
        for ci in _select_class_indices(_all_labels, _all_probs, max_classes=10):
            if _Y[:, ci].sum() == 0:
                continue
            fpr, tpr, _ = roc_curve(_Y[:, ci], _all_probs[:, ci])
            ax.plot(fpr, tpr, lw=1, alpha=0.75, label=f'{CLASS_NAMES[ci]} AUC={auc(fpr,tpr):.2f}')
        ax.plot([0,1], [0,1], ls='--', lw=1, color='gray')
        ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate'); ax.set_title('ROC Curve & AUC')
        ax.legend(fontsize=7, loc='lower right'); ax.grid(alpha=0.3)
        _savefig('02_roc_curve_auc.png', 'ROC Curve & AUC', 'classification', 'Micro-average ROC and selected one-vs-rest class ROC curves.')
    except Exception as _e:
        plt.close('all'); print(f'  ⚠ ROC plot skipped: {_e}')

    print("\n[Viz 3] Precision-Recall curves …")
    fig, ax = plt.subplots(figsize=(9, 7))
    try:
        p_micro, r_micro, _ = precision_recall_curve(_Y.ravel(), _all_probs.ravel())
        ap_micro = average_precision_score(_Y, _all_probs, average='micro')
        ax.plot(r_micro, p_micro, lw=2, label=f'Micro-average AP={ap_micro:.3f}')
        for ci in _select_class_indices(_all_labels, _all_probs, max_classes=10):
            if _Y[:, ci].sum() == 0:
                continue
            p, r, _ = precision_recall_curve(_Y[:, ci], _all_probs[:, ci])
            ap = average_precision_score(_Y[:, ci], _all_probs[:, ci])
            ax.plot(r, p, lw=1, alpha=0.75, label=f'{CLASS_NAMES[ci]} AP={ap:.2f}')
        ax.set_xlabel('Recall'); ax.set_ylabel('Precision'); ax.set_title('Precision-Recall Curves')
        ax.legend(fontsize=7, loc='lower left'); ax.grid(alpha=0.3)
        _savefig('03_precision_recall_curves.png', 'Precision-Recall Curves', 'classification', 'Micro-average and selected class precision-recall curves.')
    except Exception as _e:
        plt.close('all'); print(f'  ⚠ PR plot skipped: {_e}')

    print("\n[Viz 4] F1-score bars and confidence-threshold F1 curve …")
    prec, rec, f1, support = precision_recall_fscore_support(_all_labels, _all_preds, labels=np.arange(NUM_CLASSES), zero_division=0)
    order = np.argsort(f1)
    fig, axes = plt.subplots(1, 2, figsize=(18, 6))
    axes[0].bar(np.arange(NUM_CLASSES), f1[order])
    axes[0].set_title('Per-Class F1 Score, Sorted'); axes[0].set_xlabel('Class rank'); axes[0].set_ylabel('F1')
    xt = np.linspace(0, NUM_CLASSES-1, min(NUM_CLASSES, 12), dtype=int)
    axes[0].set_xticks(xt); axes[0].set_xticklabels([CLASS_NAMES[order[i]] for i in xt], rotation=45, ha='right', fontsize=7)
    conf = _all_probs.max(axis=1)
    thresholds = _adaptive_conf_thresholds(conf, n=50)
    f1_curve, cov_curve = [], []
    for th in thresholds:
        mask = conf >= th
        cov_curve.append(mask.mean())
        f1_curve.append(f1_score(_all_labels[mask], _all_preds[mask], average='macro', zero_division=0) if mask.any() else np.nan)
    axes[1].plot(thresholds, f1_curve, marker='o', lw=1.5, label='Macro-F1 among retained')
    axes[1].plot(thresholds, cov_curve, marker='s', lw=1.5, label='Coverage')
    axes[1].set_title('F1 / Coverage vs Confidence Threshold'); axes[1].set_xlabel('Confidence threshold')
    axes[1].set_ylim(0, 1.05); axes[1].set_xlim(float(thresholds.min()), float(thresholds.max()))
    axes[1].legend(); axes[1].grid(alpha=0.3)
    _savefig('04_f1_bar_and_threshold_curve.png', 'F1-Score Bar Charts/Curves', 'classification', 'Per-class F1 bar chart and macro-F1/coverage confidence-threshold curve.')

    print("\n[Viz 5] Per-class precision / recall / F1 heatmap …")
    metrics = np.vstack([prec, rec, f1]).T
    fig, ax = plt.subplots(figsize=(7, max(8, NUM_CLASSES * 0.16)))
    sns.heatmap(metrics, cmap='viridis', xticklabels=['Precision','Recall','F1'], yticklabels=CLASS_NAMES, ax=ax, vmin=0, vmax=1)
    ax.set_title('Per-Class Precision / Recall / F1 Heatmap')
    _savefig('05_per_class_precision_recall_f1_heatmap.png', 'Per-class Precision/Recall/F1 Heatmap', 'classification', 'Heatmap of precision, recall, and F1 for each class.')

    print("\n[Viz 6] Confidence threshold vs coverage and selective accuracy …")
    acc_curve, err_curve, coverage_curve = [], [], []
    for th in thresholds:
        mask = conf >= th
        coverage = mask.mean()
        acc = (_all_preds[mask] == _all_labels[mask]).mean() if mask.any() else np.nan
        coverage_curve.append(coverage); acc_curve.append(acc); err_curve.append(1 - acc if np.isfinite(acc) else np.nan)
    fig, ax = plt.subplots(figsize=(9, 6))
    ax.plot(thresholds, acc_curve, marker='o', label='Selective accuracy')
    ax.plot(thresholds, coverage_curve, marker='^', label='Coverage')
    ax.set_xlabel('Confidence threshold'); ax.set_ylabel('Rate'); ax.set_title('Confidence Threshold vs Coverage and Selective Accuracy')
    ax.set_ylim(0, 1.05); ax.set_xlim(float(thresholds.min()), float(thresholds.max()))
    ax.legend(); ax.grid(alpha=0.3)
    _savefig('06_confidence_threshold_vs_coverage_selective_accuracy.png', 'Confidence Threshold vs Coverage and Selective Accuracy', 'calibration', 'Coverage and selective accuracy as low-confidence predictions are rejected.')

    print("\n[Viz 6b] Risk-Coverage Curve …")
    try:
        order_conf = np.argsort(-conf)
        corr_sorted = (_all_preds[order_conf] == _all_labels[order_conf]).astype(float)
        coverages = np.arange(1, len(corr_sorted) + 1) / max(1, len(corr_sorted))
        selective_acc = np.cumsum(corr_sorted) / np.arange(1, len(corr_sorted) + 1)
        risk = 1.0 - selective_acc
        fig, ax = plt.subplots(figsize=(8, 6))
        ax.plot(coverages, risk, lw=2, label='Risk = 1 - selective accuracy')
        ax.set_xlabel('Coverage'); ax.set_ylabel('Risk'); ax.set_title('Risk-Coverage Curve')
        ax.legend(); ax.grid(alpha=0.3)
        _savefig('06b_risk_coverage_curve.png', 'Risk-Coverage Curve', 'calibration', 'Selective classification risk as coverage increases from most-confident to all samples.')
    except Exception as _e:
        plt.close('all'); print(f'  ⚠ Risk-coverage plot skipped: {_e}')

    print("\n[Viz 7] Top-1 vs Top-2 probability margin and entropy plots …")
    sorted_probs = np.sort(_all_probs, axis=1)
    top1_probs = sorted_probs[:, -1]; top2_probs = sorted_probs[:, -2]
    margins = top1_probs - top2_probs
    entropy = -np.sum(_all_probs * np.log(_all_probs + 1e-12), axis=1) / math.log(NUM_CLASSES)
    correct = (_all_preds == _all_labels)
    fig, axes = plt.subplots(2, 2, figsize=(14, 11)); axes = axes.ravel()
    axes[0].hist(margins[correct], bins=40, alpha=0.65, label='Correct'); axes[0].hist(margins[~correct], bins=40, alpha=0.65, label='Wrong')
    axes[0].set_title('Prediction Margin Distribution'); axes[0].set_xlabel('Top-1 - Top-2 probability'); axes[0].legend()
    axes[1].hist(entropy[correct], bins=40, alpha=0.65, label='Correct'); axes[1].hist(entropy[~correct], bins=40, alpha=0.65, label='Wrong')
    axes[1].set_title('Predictive Entropy Distribution'); axes[1].set_xlabel('Normalized entropy'); axes[1].legend()
    axes[2].scatter(margins, entropy, c=correct.astype(int), s=8, alpha=0.45, label='Samples')
    axes[2].set_title('Margin vs Entropy'); axes[2].set_xlabel('Prediction margin'); axes[2].set_ylabel('Normalized entropy'); axes[2].legend()
    axes[3].scatter(top2_probs[correct], top1_probs[correct], s=8, alpha=0.45, label='Correct')
    axes[3].scatter(top2_probs[~correct], top1_probs[~correct], s=8, alpha=0.45, label='Wrong')
    _xlo, _xhi = _padded_limits(top2_probs)
    _ylo, _yhi = _padded_limits(top1_probs)
    _dlo, _dhi = min(_xlo, _ylo), max(_xhi, _yhi)
    axes[3].plot([_dlo, _dhi], [_dlo, _dhi], '--', lw=1, label='Top-1 = Top-2')
    axes[3].set_xlim(_xlo, _xhi); axes[3].set_ylim(_ylo, _yhi)
    axes[3].set_title('Top-1 vs Top-2 Probability Margin'); axes[3].set_xlabel('Top-2 probability'); axes[3].set_ylabel('Top-1 probability'); axes[3].legend()
    for _ax in axes: _ax.grid(alpha=0.3)
    _savefig('07_top1_top2_probability_margin_entropy.png', 'Top-1 vs Top-2 Probability Margin and Predictive Entropy Plots', 'calibration', 'Correct/wrong distributions of prediction margin, entropy, and top-1/top-2 probability separation.')

    print("\n[Viz 8] Class support, difficulty, and confidence scatter plots …")
    per_class_acc = np.zeros(NUM_CLASSES); per_class_conf = np.zeros(NUM_CLASSES); per_class_entropy = np.zeros(NUM_CLASSES)
    for ci in range(NUM_CLASSES):
        mask = _all_labels == ci
        if mask.any():
            per_class_acc[ci] = (_all_preds[mask] == _all_labels[mask]).mean()
            per_class_conf[ci] = _all_probs[mask].max(axis=1).mean()
            per_class_entropy[ci] = entropy[mask].mean()
    difficulty = 1 - per_class_acc
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    axes[0].scatter(support, difficulty, s=25, alpha=0.75); axes[0].set_xlabel('Class support'); axes[0].set_ylabel('Difficulty = 1 - accuracy'); axes[0].set_title('Support vs Difficulty')
    axes[1].scatter(per_class_conf, difficulty, s=25, alpha=0.75); axes[1].set_xlabel('Mean confidence'); axes[1].set_ylabel('Difficulty'); axes[1].set_title('Confidence vs Difficulty')
    axes[2].scatter(per_class_entropy, difficulty, s=25, alpha=0.75); axes[2].set_xlabel('Mean normalized entropy'); axes[2].set_ylabel('Difficulty'); axes[2].set_title('Entropy vs Difficulty')
    for ax in axes: ax.grid(alpha=0.3)
    _savefig('08_class_support_difficulty_confidence_scatter.png', 'Class Support/Difficulty/Confidence Scatter', 'classification', 'Per-class support, difficulty, confidence, and entropy scatter plots.')

    print("\n[Viz 9] Confusion matrix and cumulative error Pareto chart …")
    cmx = confusion_matrix(_all_labels, _all_preds, labels=np.arange(NUM_CLASSES))
    cmx_norm = _safe_div(cmx, cmx.sum(axis=1, keepdims=True))
    fig, ax = plt.subplots(figsize=(12, 10))
    show_n = min(NUM_CLASSES, 50)
    sns.heatmap(cmx_norm[:show_n, :show_n], cmap='Reds', ax=ax, cbar=True, xticklabels=CLASS_NAMES[:show_n], yticklabels=CLASS_NAMES[:show_n])
    ax.set_title(f'Normalized Confusion Matrix, first {show_n} classes'); ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    plt.xticks(rotation=90, fontsize=6); plt.yticks(rotation=0, fontsize=6)
    _savefig('09a_confusion_matrix_normalized.png', 'Normalized Confusion Matrix', 'classification', 'Normalized confusion matrix for the first classes.')
    errors_by_class = support - np.diag(cmx)
    e_order = np.argsort(errors_by_class)[::-1]
    cum_errors = np.cumsum(errors_by_class[e_order]) / max(1, errors_by_class.sum())
    fig, ax1 = plt.subplots(figsize=(14, 6))
    ax1.bar(np.arange(NUM_CLASSES), errors_by_class[e_order])
    ax1.set_ylabel('Errors'); ax1.set_xlabel('Class rank by errors'); ax1.set_title('Cumulative Error Pareto Chart')
    ax2 = ax1.twinx(); ax2.plot(np.arange(NUM_CLASSES), cum_errors, marker='o', lw=1.5)
    ax2.set_ylabel('Cumulative error fraction'); ax2.set_ylim(0, 1.05)
    xt = np.linspace(0, NUM_CLASSES-1, min(NUM_CLASSES, 15), dtype=int)
    ax1.set_xticks(xt); ax1.set_xticklabels([CLASS_NAMES[e_order[i]] for i in xt], rotation=45, ha='right', fontsize=7)
    _savefig('09b_cumulative_error_pareto.png', 'Cumulative Error Pareto Chart', 'classification', 'Classes ranked by number of errors with cumulative error share.')

    print("\n[Viz 10] Penultimate feature PCA projections …")
    try:
        max_points = min(3000, len(_all_features))
        idx = np.linspace(0, len(_all_features)-1, max_points, dtype=int)
        pca = PCA(n_components=2, random_state=0)
        z = pca.fit_transform(_all_features[idx])
        fig, ax = plt.subplots(figsize=(10, 8))
        sc = ax.scatter(z[:,0], z[:,1], c=_all_labels[idx], s=8, alpha=0.7, cmap='tab20')
        ax.set_title(f'Penultimate Feature PCA by Class Subset — {max_points} samples')
        ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)'); ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
        fig.colorbar(sc, ax=ax, fraction=0.03, pad=0.02)
        _savefig('10_penultimate_feature_pca_by_class_subset.png', 'Penultimate Feature PCA by Class Subset', 'representation', 'PCA projection of penultimate features colored by true class.')
    except Exception as _e:
        print(f'  ⚠ PCA plot skipped: {_e}')

    print("\n[Viz 10b] Failed and correct prediction image grids …")
    try:
        if _viz_img_bank is not None and _viz_lbl_bank is not None and _viz_pred_bank is not None:
            _bank_correct = (_viz_lbl_bank == _viz_pred_bank)
            _bank_conf = np.asarray(_viz_conf_bank)
            def _plot_prediction_grid(_indices, _fname, _title, _desc):
                _indices = list(_indices)[:16]
                if not _indices:
                    print(f'  ⚠ {_title} skipped: no matching samples in bounded image bank'); return
                _cols = 4; _rows = int(math.ceil(len(_indices) / _cols))
                fig, axes = plt.subplots(_rows, _cols, figsize=(3.6*_cols, 3.9*_rows))
                axes = np.asarray(axes).reshape(-1)
                for _ax in axes: _ax.axis('off')
                for _ax, _idx in zip(axes, _indices):
                    _img = _unnormalize_img(_viz_img_bank[_idx], _mean, _std)
                    _true = int(_viz_lbl_bank[_idx]); _pred = int(_viz_pred_bank[_idx]); _cf = float(_bank_conf[_idx])
                    _ax.imshow(_img); _ax.set_title(f'Pred: {CLASS_NAMES[_pred]} ({_cf:.2f})\nTrue: {CLASS_NAMES[_true]}'); _ax.axis('off')
                fig.suptitle(_title); _savefig(_fname, _title, 'qualitative', _desc)
            _failed_rank = np.where(~_bank_correct)[0]
            _failed_rank = _failed_rank[np.argsort(-_bank_conf[_failed_rank])] if len(_failed_rank) else []
            _correct_rank = np.where(_bank_correct)[0]
            _correct_rank = _correct_rank[np.argsort(-_bank_conf[_correct_rank])] if len(_correct_rank) else []
            _plot_prediction_grid(_failed_rank, '10b_failed_prediction_images_confidence_true_classes.png',
                                  'Failed Prediction Images with Confidence Scores and Correct Classes',
                                  'Highest-confidence failed predictions, showing predicted class, confidence, and true class.')
            _plot_prediction_grid(_correct_rank, '10c_correct_prediction_images_confidence.png',
                                  'Correct Prediction Images with Confidence Scores',
                                  'Highest-confidence correct predictions, showing predicted class, confidence, and true class.')
        else:
            print('  ⚠ Prediction image grids skipped: no bounded image bank available')
    except Exception as _e:
        print(f'  ⚠ Prediction image grids skipped: {_e}')

# ════════════════════════════════════════════════════════════════════════
# InLine-DS PAPER DIAGNOSTICS (P1–P4)  — architecture-agnostic
# ════════════════════════════════════════════════════════════════════════
print("\n" + "="*72)
print("InLine^D-S paper diagnostics (P1–P4) + extra plots")
print("="*72)

CONFUSION_L2_THRESHOLD = globals().get('CONFUSION_L2_THRESHOLD', 1e-3)

_diag_imgs = None
if _model_loaded and _sample_imgs is not None and _sample_imgs.numel() > 0:
    _diag_imgs = _sample_imgs[:16]
elif _model_loaded and _val_loader is not None:
    try:
        _diag_imgs = next(iter(_val_loader))[0][:16]
    except Exception:
        _diag_imgs = None


def _layer_depth_index(names):
    return list(range(len(names)))

########################## P1 ##################################################
print("\n[P1] Attention entropy by depth (InLine-DS MMSK vs scaled feature map) …")
try:
    if _diag_imgs is None or not _ds_attn_modules(_viz_model):
        print("  ⚠ P1 skipped: no model / no InLine-DS attention modules.")
    else:
        rows_mmsk, names_mmsk = _collect_generic_attention_rows(
            _viz_model, _diag_imgs, normalization='subtractive', feature_map='mmsk')
        rows_scaled, names_scaled = _collect_generic_attention_rows(
            _viz_model, _diag_imgs, normalization='divisive', feature_map='scaled')

        def _per_layer_mean_entropy(rows_list):
            vals = []
            for r in rows_list:
                try:
                    ent = _normalized_entropy_from_rows(r)
                    vals.append(float(ent.mean().item()))
                except Exception:
                    vals.append(float('nan'))
            return vals

        ent_mmsk = _per_layer_mean_entropy(rows_mmsk)
        ent_scaled = _per_layer_mean_entropy(rows_scaled)
        if ent_mmsk:
            xs = _layer_depth_index(names_mmsk)
            fig, ax = plt.subplots(figsize=(9, 5))
            ax.plot(xs, ent_mmsk, 'o-', lw=2, label='InLine-DS (MMSK, subtractive)')
            if len(ent_scaled) == len(xs):
                ax.plot(xs, ent_scaled, 's--', lw=2, label='Scaled ELU + divisive (baseline)')
            ax.set_xlabel('Attention layer (depth →)')
            ax.set_ylabel('Mean normalized entropy  (0 = concentrated, 1 = uniform)')
            ax.set_ylim(0, 1.02)
            ax.grid(alpha=0.3); ax.legend()
            ax.set_title('P1: Attention Entropy by Depth')
            _savefig('P1_attention_entropy_by_depth.png',
                     'P1: Attention Entropy by Depth', 'paper_diagnostics',
                     'Per-layer mean normalized attention entropy for the InLine-DS MMSK '
                     'subtractive rows versus the scaled-feature-map baseline.')
            try:
                _p1_csv = os.path.join(VIZ_DIR, 'P1_attention_entropy_by_depth.csv')
                pd.DataFrame({
                    'layer_index': xs,
                    'layer_name': names_mmsk,
                    'entropy_mmsk_subtractive': ent_mmsk,
                    'entropy_scaled_subtractive': ent_scaled[:len(xs)] if len(ent_scaled) >= len(xs) else [np.nan] * len(xs),
                }).to_csv(_p1_csv, index=False)
                _register_viz('P1_attention_entropy_by_depth.csv',
                              'P1 Attention Entropy CSV', 'paper_diagnostics',
                              'Raw per-layer normalized entropy values for MMSK and scaled feature-map rows.')
            except Exception as _csv_e:
                print(f"  ⚠ P1 CSV skipped: {_csv_e}")
        else:
            print("  ⚠ P1 produced no layers.")
except Exception as _e:
    print(f"  ⚠ P1 failed: {_e}")

########################## P2 ##################################################
print("\n[P2] Row injectivity / confusion count by depth "
      "(MMSK+subtractive vs scaled+divisive) ...")
try:
    if _diag_imgs is None or not _ds_attn_modules(_viz_model):
        print("  ⚠ P2 skipped: no model / no InLine-DS attention modules.")
    else:
        def _confusion_counts(normalization, feature_map='mmsk',
                              tol=1e-3, max_q=256):
            """Per-head row-collision statistics for one (feature_map, normalization)
            configuration.

            For each attention layer we reconstruct the explicit rows [B,H,Nq,Nk],
            then measure query-row collisions PER HEAD (not head-averaged, since
            injectivity is a per-head property) on the first diagnostic image.
            We report, per layer:
              * confused_query_pairs  : worst-head count of query-row pairs with
                                        pairwise L2 < tol
              * confused_pair_percent : that count as a % of all query pairs
              * rows_with_any_collision (worst head)
              * min_pairwise_l2       : smallest distance between any two distinct
                                        query rows, averaged over heads (a
                                        threshold-free "how injective" margin;
                                        larger = more injective)
              * min_pairwise_l2_worst : smallest distance over ALL heads (the
                                        single closest colliding pair anywhere)
            """
            rows_list, names = _collect_generic_attention_rows(
                _viz_model, _diag_imgs,
                normalization=normalization, feature_map=feature_map)
            out = []
            for li, (nm, r) in enumerate(zip(names, rows_list)):
                if r.dim() != 4:
                    continue
                heads = r[0].float()               # [H, Nq, Nk]  (image 0, all heads)
                H, Nq, Nk = heads.shape
                if Nq <= 1:
                    continue
                if Nq > max_q:
                    idx = torch.linspace(0, Nq - 1, max_q).long()
                    heads = heads[:, idx, :]
                    Nq = int(heads.shape[1])

                total_pairs = int(Nq * (Nq - 1) // 2)
                eye_big = torch.eye(Nq, device=heads.device) * 1e9

                per_head_pairs = []
                per_head_rowcol = []
                per_head_min = []
                for h in range(H):
                    rr = heads[h]                                    # [Nq, Nk]
                    d = torch.cdist(rr.unsqueeze(0), rr.unsqueeze(0)).squeeze(0)
                    upper = torch.triu(torch.ones_like(d, dtype=torch.bool), diagonal=1)
                    per_head_pairs.append(int(((d < tol) & upper).sum().item()))
                    dmask = d + eye_big                              # ignore self-distance
                    per_head_rowcol.append(int((dmask.min(dim=1).values < tol).sum().item()))
                    per_head_min.append(float(dmask.min().item()))

                worst_h = int(np.argmax(per_head_pairs))             # head with most collisions
                pair_count = int(per_head_pairs[worst_h])
                row_has_collision = int(per_head_rowcol[worst_h])
                min_l2_mean = float(np.mean(per_head_min))           # avg-over-heads margin
                min_l2_worst = float(np.min(per_head_min))           # closest pair anywhere

                out.append({
                    'layer_index': li, 'layer_name': nm,
                    'feature_map': feature_map, 'normalization': normalization,
                    'num_heads': int(H), 'num_queries_used': int(Nq),
                    'num_key_tokens': int(heads.shape[2]),
                    'worst_head': worst_h,
                    'confused_query_pairs_l2_lt_tol': pair_count,
                    'total_query_pairs': total_pairs,
                    'confused_pair_percent': 100.0 * pair_count / max(1, total_pairs),
                    'rows_with_any_collision': row_has_collision,
                    'rows_with_any_collision_percent': 100.0 * row_has_collision / max(1, Nq),
                    'min_pairwise_l2_mean_over_heads': min_l2_mean,
                    'min_pairwise_l2_worst_head': min_l2_worst,
                    'tol_l2': float(tol),
                })
            return out

        # The paper's claim is: MMSK + subtractive (injective) vs the scaled
        # feature map + divisive (the non-injective baseline). Divisive must use
        # a (near-)homogeneous map for its scale-invariance failure to appear;
        # reusing MMSK would hide it.
        rows_sub = _confusion_counts('subtractive', feature_map='mmsk',
                                     tol=CONFUSION_L2_THRESHOLD)
        rows_div = _confusion_counts('divisive', feature_map='scaled',
                                     tol=CONFUSION_L2_THRESHOLD)
        df_p2 = pd.DataFrame(rows_sub + rows_div)

        if not df_p2.empty:
            fig, axes = plt.subplots(1, 3, figsize=(19, 5))

            _series = [
                ('subtractive', 'mmsk',   'o-',  'MMSK + subtractive (ours)'),
                ('divisive',    'scaled', 's--', 'Scaled map + divisive (baseline)'),
            ]

            # Panel 0: raw confused-pair count (worst head) by depth
            for norm, fmap, style, label in _series:
                dfn = df_p2[(df_p2['normalization'] == norm) &
                            (df_p2['feature_map'] == fmap)]
                if not dfn.empty:
                    axes[0].plot(dfn['layer_index'],
                                 dfn['confused_query_pairs_l2_lt_tol'],
                                 style, lw=2, label=label)
            axes[0].set_xlabel('Attention layer (depth →)')
            axes[0].set_ylabel('Confused query-row pairs (worst head)')
            axes[0].set_title('P2: Confusion Count')
            axes[0].grid(alpha=0.3); axes[0].legend()

            # Panel 1: normalized confusion rate (%) by depth
            for norm, fmap, style, label in _series:
                dfn = df_p2[(df_p2['normalization'] == norm) &
                            (df_p2['feature_map'] == fmap)]
                if not dfn.empty:
                    axes[1].plot(dfn['layer_index'],
                                 dfn['confused_pair_percent'],
                                 style, lw=2, label=label)
            axes[1].set_xlabel('Attention layer (depth →)')
            axes[1].set_ylabel(f'% of query-row pairs with L2 < {CONFUSION_L2_THRESHOLD:g}')
            axes[1].set_title('P2: Normalized Confusion Rate')
            axes[1].grid(alpha=0.3); axes[1].legend()

            # Panel 2: threshold-free injectivity margin — min pairwise L2 by depth.
            # Lower curve = rows crowd together = less injective. Log scale so the
            # divisive collapse toward ~0 is visible against the subtractive floor.
            for norm, fmap, style, label in _series:
                dfn = df_p2[(df_p2['normalization'] == norm) &
                            (df_p2['feature_map'] == fmap)]
                if not dfn.empty:
                    axes[2].plot(dfn['layer_index'],
                                 dfn['min_pairwise_l2_worst_head'].clip(lower=1e-12),
                                 style, lw=2, label=label)
            axes[2].axhline(CONFUSION_L2_THRESHOLD, color='gray', ls=':', lw=1,
                            label=f'collision tol = {CONFUSION_L2_THRESHOLD:g}')
            axes[2].set_yscale('log')
            axes[2].set_xlabel('Attention layer (depth →)')
            axes[2].set_ylabel('Min pairwise row L2 (worst head, log)')
            axes[2].set_title('P2: Injectivity Margin (higher = more injective)')
            axes[2].grid(alpha=0.3, which='both'); axes[2].legend()

            fig.suptitle('P2: Row Confusion / Injectivity')
            _savefig('P2_row_confusion_injectivity.png',
                     'P2: Row Confusion / Injectivity', 'paper_diagnostics',
                     'Per-head query-row collisions by depth for MMSK+subtractive versus '
                     'the scaled-feature-map+divisive baseline, plus a threshold-free '
                     'minimum-pairwise-distance injectivity margin. Collisions are measured '
                     'per head (worst head reported), not head-averaged.')
            try:
                _p2_csv = os.path.join(VIZ_DIR, 'P2_row_confusion_injectivity.csv')
                df_p2.to_csv(_p2_csv, index=False)
                _register_viz('P2_row_confusion_injectivity.csv',
                              'P2 Row Confusion CSV', 'paper_diagnostics',
                              'Per-layer per-head confusion counts, rates, and minimum '
                              'pairwise row distances for both configurations.')
            except Exception as _csv_e:
                print(f"  ⚠ P2 CSV skipped: {_csv_e}")

            # Console summary
            try:
                _sub_pairs = int(df_p2[(df_p2.normalization == 'subtractive')]
                                 ['confused_query_pairs_l2_lt_tol'].sum())
                _div_pairs = int(df_p2[(df_p2.normalization == 'divisive')]
                                 ['confused_query_pairs_l2_lt_tol'].sum())
                _sub_marg = float(df_p2[(df_p2.normalization == 'subtractive')]
                                  ['min_pairwise_l2_worst_head'].min())
                _div_marg = float(df_p2[(df_p2.normalization == 'divisive')]
                                  ['min_pairwise_l2_worst_head'].min())
                print(f"  subtractive(MMSK): total confused pairs={_sub_pairs}, "
                      f"min margin={_sub_marg:.2e}")
                print(f"  divisive(scaled) : total confused pairs={_div_pairs}, "
                      f"min margin={_div_marg:.2e}")
            except Exception:
                pass
        else:
            print("  ⚠ P2 produced no reconstructable attention layers.")
except Exception as _e:
    print(f"  ⚠ P2 failed: {_e}")
########################## P3 ##################################################
print("\n[P3] Local vs random key masking (locality probe) …")
try:
    if not _model_loaded or _val_loader is None or not _ds_attn_modules(_viz_model):
        print("  ⚠ P3 skipped: no model / loader / InLine-DS attention modules.")
    else:
        _P3_BATCHES = 4
        acc_base = _eval_with_attention_mask(_viz_model, _val_loader, kind=None, max_batches=_P3_BATCHES)
        acc_local = _eval_with_attention_mask(_viz_model, _val_loader, kind='local3x3', max_batches=_P3_BATCHES)
        acc_rand = _eval_with_attention_mask(_viz_model, _val_loader, kind='random', max_batches=_P3_BATCHES, seed=0)
        if not math.isnan(acc_base):
            drop_local = acc_base - acc_local
            drop_rand = acc_base - acc_rand
            df_p3 = pd.DataFrame([
                {'condition': 'baseline', 'top1_accuracy': acc_base, 'accuracy_drop_pp': 0.0, 'batches': _P3_BATCHES},
                {'condition': 'mask_local_3x3', 'top1_accuracy': acc_local, 'accuracy_drop_pp': drop_local, 'batches': _P3_BATCHES},
                {'condition': 'mask_random_equal_count', 'top1_accuracy': acc_rand, 'accuracy_drop_pp': drop_rand, 'batches': _P3_BATCHES},
            ])
            fig, axes = plt.subplots(1, 2, figsize=(12, 5))
            axes[0].bar(['baseline', 'local 3×3', 'random'], [acc_base, acc_local, acc_rand])
            axes[0].set_ylabel('Top-1 accuracy (%)'); axes[0].set_title('Accuracy under query-key masking')
            for _i, _v in enumerate([acc_base, acc_local, acc_rand]):
                axes[0].text(_i, _v, f'{_v:.1f}', ha='center', va='bottom')
            axes[1].bar(['local 3×3', 'random'], [drop_local, drop_rand])
            axes[1].set_ylabel('Accuracy drop vs baseline (percentage points)'); axes[1].set_title('Locality: equal-count key masking')
            for _i, _v in enumerate([drop_local, drop_rand]):
                axes[1].text(_i, _v, f'{_v:.2f}', ha='center', va='bottom')
            fig.suptitle('P3: Local vs Random Masking')
            _savefig('P3_local_vs_random_masking.png',
                     'P3: Local vs Random Masking', 'paper_diagnostics',
                     'Top-1 accuracy when each query has its local 3×3 key neighborhood '
                     'masked versus the same number of random keys masked. The mask is '
                     'rectangular when the backbone uses spatial-reduction attention.')
            try:
                _p3_csv = os.path.join(VIZ_DIR, 'P3_local_vs_random_masking.csv')
                df_p3.to_csv(_p3_csv, index=False)
                _register_viz('P3_local_vs_random_masking.csv',
                              'P3 Local vs Random Masking CSV', 'paper_diagnostics',
                              'Raw baseline, local-masked, and random-masked accuracies and drops.')
            except Exception as _csv_e:
                print(f"  ⚠ P3 CSV skipped: {_csv_e}")
            print(f"  baseline={acc_base:.2f}%  local={acc_local:.2f}%  random={acc_rand:.2f}%  "
                  f"(drop local={drop_local:.2f}pp, random={drop_rand:.2f}pp)")
        else:
            print("  ⚠ P3 baseline accuracy is NaN; masking probe unavailable.")
except Exception as _e:
    print(f"  ⚠ P3 failed: {_e}")

########################## P4 ##################################################
print("\n[P4] Learned kernel parameters (per-head τ and η/floor) …")
try:
    _mods = _ds_attn_modules(_viz_model)
    if not _mods:
        print("  ⚠ P4 skipped: no InLine-DS attention modules.")
    else:
        import torch.nn.functional as _Fp

        def _positive_param(_ker, names, default_nan=True):
            for _name in names:
                if hasattr(_ker, _name):
                    _p = getattr(_ker, _name).detach().float()
                    if _name.startswith('log_') or _name in ('beta', 'raw_eta', 'eta_raw', 'floor_raw'):
                        return _Fp.softplus(_p).reshape(-1).cpu().numpy()
                    return _p.reshape(-1).cpu().numpy()
            return None if default_nan else np.array([])

        def _param_by_role(_ker, _kind):
            """Fallback used when the expected attribute names are absent: scan the
            kernel's own parameters/buffers for a τ-like (temperature) or η-like
            (positive floor) per-head tensor. This lets P4 populate for whichever
            InLine-DS backbone is being trained (deit/swin/pvt/cswin), even if its
            kernels store these under a slightly different name."""
            _want = ('tau', 'temp') if _kind == 'tau' else ('eta', 'floor', 'beta')
            try:
                _named = list(_ker.named_parameters()) + list(_ker.named_buffers())
            except Exception:
                return None
            for _pname, _pt in _named:
                _low = _pname.lower()
                if _pt is None:
                    continue
                if any(_w in _low for _w in _want):
                    _v = _pt.detach().float()
                    if 'log' in _low or 'raw' in _low:
                        _v = _Fp.softplus(_v)
                    return _v.reshape(-1).cpu().numpy()
            return None

        def _row_idx(_arr, _h):
            # Per-head value when length matches head count; broadcast a shared
            # scalar (length 1) across all heads; NaN if unavailable.
            if _arr is None or len(_arr) == 0:
                return np.nan
            return float(_arr[_h % len(_arr)])

        rows_p4 = []
        for li, (_nm, _m) in enumerate(_mods):
            for side, _ker in [('query', getattr(_m, 'kernel_q', None)), ('key', getattr(_m, 'kernel_k', None))]:
                if _ker is None:
                    continue
                tau = _positive_param(_ker, ['log_tau', 'tau_raw', 'tau'])
                if tau is None:
                    tau = _param_by_role(_ker, 'tau')
                eta = _positive_param(_ker, ['log_eta', 'eta_raw', 'raw_eta', 'eta', 'floor', 'floor_eta', 'beta'])
                if eta is None:
                    eta = _param_by_role(_ker, 'eta')
                if eta is not None and hasattr(_ker, 'beta') and not any(hasattr(_ker, n) for n in ['log_eta', 'eta_raw', 'raw_eta', 'eta', 'floor', 'floor_eta']):
                    eta = eta + float(getattr(_ker, '_PHI_EPS', 1e-4))
                # Resolve head count from the params themselves, then the kernel,
                # then the parent attention module (correct for every backbone).
                _len_tau = len(tau) if tau is not None else 0
                _len_eta = len(eta) if eta is not None else 0
                n_heads = max(_len_tau, _len_eta,
                              int(getattr(_ker, 'num_heads', 0) or 0),
                              int(getattr(_m, 'num_heads', 0) or 0), 1)
                for h in range(n_heads):
                    rows_p4.append({
                        'layer_index': li, 'layer_name': _nm, 'side': side, 'head': h,
                        'tau': _row_idx(tau, h),
                        'eta_floor': _row_idx(eta, h),
                    })
        df_p4 = pd.DataFrame(rows_p4)
        if not df_p4.empty:
            fig, axes = plt.subplots(1, 3, figsize=(17, 5.5))
            for side in ['query', 'key']:
                d = df_p4[df_p4['side'] == side]
                axes[0].hist(d['tau'].dropna(), bins=20, alpha=0.55, label=side)
                axes[1].hist(d['eta_floor'].dropna(), bins=20, alpha=0.55, label=side)
                layer_mean = d.groupby('layer_index')[['tau', 'eta_floor']].mean(numeric_only=True).reset_index()
                axes[2].plot(layer_mean['layer_index'], layer_mean['tau'], marker='o', lw=1.8, label=f'{side} τ')
                axes[2].plot(layer_mean['layer_index'], layer_mean['eta_floor'], marker='s', lw=1.8, linestyle='--', label=f'{side} η')
            axes[0].set_xlabel('τ = softplus(log_tau)'); axes[0].set_ylabel('Head count'); axes[0].set_title('Temperature τ distribution'); axes[0].grid(alpha=0.3); axes[0].legend()
            axes[1].set_xlabel('η / positive floor'); axes[1].set_ylabel('Head count'); axes[1].set_title('Floor η distribution'); axes[1].grid(alpha=0.3); axes[1].legend()
            axes[2].set_xlabel('Layer index (depth →)'); axes[2].set_ylabel('Mean per-head value'); axes[2].set_title('Mean τ and η by depth'); axes[2].grid(alpha=0.3); axes[2].legend(fontsize=8)
            fig.suptitle('P4: Learned Kernel Parameters')
            _savefig('P4_learned_kernel_tau_eta.png',
                     'P4: Learned Kernel Parameters τ and η', 'paper_diagnostics',
                     'Distribution of learned per-head kernel temperatures τ and positive '
                     'floor parameters η after training, with layer-wise means.')
            try:
                _p4_csv = os.path.join(VIZ_DIR, 'P4_learned_kernel_tau_eta.csv')
                df_p4.to_csv(_p4_csv, index=False)
                _register_viz('P4_learned_kernel_tau_eta.csv',
                              'P4 Learned Kernel τ/η CSV', 'paper_diagnostics',
                              'Raw per-layer, per-head query/key τ and η/floor values.')
            except Exception as _csv_e:
                print(f"  ⚠ P4 CSV skipped: {_csv_e}")
        else:
            print("  ⚠ P4 found no kernels exposing τ/η-style parameters.")
except Exception as _e:
    print(f"  ⚠ P4 failed: {_e}")


# ════════════════════════════════════════════════════════════════════════
# EXTRA InLine-DS diagnostics (architecture-agnostic)
# ════════════════════════════════════════════════════════════════════════
print("\n[Extra 1] Effective attended tokens (perplexity) by depth …")
try:
    if _diag_imgs is None or not _ds_attn_modules(_viz_model):
        print("  ⚠ Extra-1 skipped: no model / attention modules.")
    else:
        rows_list, names = _collect_generic_attention_rows(_viz_model, _diag_imgs, normalization='subtractive', feature_map='mmsk')
        ppl, ks = [], []
        for r in rows_list:
            p = _rows_to_entropy_prob(r)
            ent_nat = -(p * torch.log(p.clamp_min(1e-12))).sum(-1)
            ppl.append(float(torch.exp(ent_nat).mean().item()))
            ks.append(int(r.shape[-1]))
        if ppl:
            xs = list(range(len(ppl)))
            fig, ax = plt.subplots(figsize=(9, 5))
            ax.plot(xs, ppl, 'o-', lw=2, color='#8064a2', label='effective tokens (exp H)')
            ax.plot(xs, ks, 'k:', lw=1, label='available keys N')
            ax.set_xlabel('Attention layer (depth →)'); ax.set_ylabel('Effective number of attended tokens')
            ax.grid(alpha=0.3); ax.legend()
            ax.set_title('Extra: Effective Attended Tokens by Depth')
            _savefig('E1_effective_attended_tokens_by_depth.png',
                     'Extra: Effective Attended Tokens by Depth', 'paper_diagnostics',
                     'Perplexity exp(H) of the per-query attention distribution averaged '
                     'per layer (how many keys each query effectively attends to), with '
                     'the available key count N for reference.')
        else:
            print("  ⚠ Extra-1 produced no layers.")
except Exception as _e:
    print(f"  ⚠ Extra-1 failed: {_e}")


print("\n[Extra 2] Negative attention mass fraction by depth …")
try:
    if _diag_imgs is None or not _ds_attn_modules(_viz_model):
        print("  ⚠ Extra-2 skipped: no model / attention modules.")
    else:
        rows_list, names = _collect_generic_attention_rows(_viz_model, _diag_imgs, normalization='subtractive', feature_map='mmsk')
        neg_frac, neg_mass = [], []
        for r in rows_list:
            rr = r.float()
            neg_frac.append(100.0 * float((rr < 0).float().mean().item()))
            negm = rr.clamp(max=0.0).abs().sum(-1)
            totm = rr.abs().sum(-1).clamp_min(1e-12)
            neg_mass.append(100.0 * float((negm / totm).mean().item()))
        if neg_frac:
            xs = list(range(len(neg_frac)))
            fig, ax = plt.subplots(figsize=(9, 5))
            ax.plot(xs, neg_frac, 'o-', lw=2, label='% entries negative')
            ax.plot(xs, neg_mass, 's--', lw=2, color='#c0504d', label='% of |mass| that is negative')
            ax.set_xlabel('Attention layer (depth →)'); ax.set_ylabel('Percent')
            ax.grid(alpha=0.3); ax.legend()
            ax.set_title('Extra: Negative Attention Mass by Depth')
            _savefig('E2_negative_attention_mass_by_depth.png',
                     'Extra: Negative Attention Mass by Depth', 'paper_diagnostics',
                     'Subtractive InLine-DS rows can be negative. Fraction of negative '
                     'row entries and the share of absolute attention mass carried by '
                     'negative weights, per layer.')
        else:
            print("  ⚠ Extra-2 produced no layers.")
except Exception as _e:
    print(f"  ⚠ Extra-2 failed: {_e}")


print("\n[Extra 3] Attention-entropy heatmap by layer and head …")
try:
    if _diag_imgs is None or not _ds_attn_modules(_viz_model):
        print("  ⚠ Extra-3 skipped: no model / attention modules.")
    else:
        rows_list, names = _collect_generic_attention_rows(_viz_model, _diag_imgs, normalization='subtractive', feature_map='mmsk')
        per_head = []
        for r in rows_list:
            ent = _normalized_entropy_from_rows(r)
            per_head.append(ent.mean(dim=(0, 2)).cpu().numpy())
        if per_head:
            H = max(len(v) for v in per_head)
            M = np.full((len(per_head), H), np.nan)
            for _i, _v in enumerate(per_head):
                M[_i, :len(_v)] = _v
            fig, ax = plt.subplots(figsize=(min(1.2 * H + 3, 14), 0.5 * len(per_head) + 3))
            im = ax.imshow(M, aspect='auto', cmap='viridis', vmin=0, vmax=1)
            ax.set_xlabel('head'); ax.set_ylabel('layer (depth →)')
            ax.set_title('Extra: Normalized Attention Entropy (layer × head)')
            fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
            _savefig('E3_attention_entropy_layer_head_heatmap.png',
                     'Extra: Attention Entropy Heatmap (layer × head)', 'paper_diagnostics',
                     'Mean normalized attention entropy for every (layer, head) of the '
                     'InLine-DS subtractive rows. Works for all four backbones.')
        else:
            print("  ⚠ Extra-3 produced no layers.")
except Exception as _e:
    print(f"  ⚠ Extra-3 failed: {_e}")


print("\n[Extra 4] CLS→patch attention maps (DeiT only) …")
try:
    if MODEL_FAMILY != 'deit':
        print(f"  ⚠ Extra-4 skipped: CLS→patch maps are DeiT-specific (current family = {MODEL_FAMILY}).")
    elif _diag_imgs is None:
        print("  ⚠ Extra-4 skipped: no sample images.")
    else:
        rows_list, names = _collect_generic_attention_rows(_viz_model, _diag_imgs, normalization='subtractive', feature_map='mmsk')
        sq = _square_rows_only(rows_list)
        if sq:
            n_show = min(6, len(sq))
            sel = np.linspace(0, len(sq) - 1, n_show).astype(int)
            fig, axes = plt.subplots(1, n_show, figsize=(3.0 * n_show, 3.4))
            axes = np.atleast_1d(axes)
            for _ax, _li in zip(axes, sel):
                r = sq[_li][0].mean(0)
                cls_row = r[0].cpu().numpy()
                grid = _patch_map_from_cls(cls_row, prefix=1)
                _ax.imshow(grid, cmap='inferno')
                _ax.set_title(f'layer {int(_li)}'); _ax.axis('off')
            fig.suptitle('Extra: CLS→Patch Attention Maps')
            _savefig('E4_cls_to_patch_attention_maps.png',
                     'Extra: CLS→Patch Attention Maps', 'paper_diagnostics',
                     'For DeiT, the CLS query attention over patch tokens reshaped to the '
                     'patch grid, at several depths (subtractive InLine-DS rows).')
        else:
            print("  ⚠ Extra-4 skipped: no square layers found.")
except Exception as _e:
    print(f"  ⚠ Extra-4 failed: {_e}")


# ════════════════════════════════════════════════════════════════════════
# OTHER PAPER PLOTS (26–28)
# ════════════════════════════════════════════════════════════════════════
print("\n" + "="*72)
print("Attached-style InLine-DS paper plots (26–28)")
print("="*72)


def _bar_value_labels(_ax, _xs, _ys, _fmt='{:.2f}', _dy_frac=0.015):
    try:
        _ylim = _ax.get_ylim()
        _dy = max((_ylim[1] - _ylim[0]) * _dy_frac, 1e-6)
        for _x, _y in zip(_xs, _ys):
            if np.isfinite(_y):
                _ax.text(_x, _y + _dy, _fmt.format(float(_y)), ha='center', va='bottom', fontsize=10)
    except Exception:
        pass


print("\n[Plot 26] Confusion occurrence distribution (divisive vs subtractive) …")
try:
    if _diag_imgs is None or not _ds_attn_modules(_viz_model):
        print("  ⚠ Plot 26 skipped: no diagnostic images or no InLine-DS attention modules.")
    else:
        def _confusion_occurrences_by_image(_normalization, _tol=1e-3, _max_q=256):
            _rows_list, _names = _collect_generic_attention_rows(_viz_model, _diag_imgs, normalization=_normalization, feature_map='mmsk')
            _valid = [_r for _r in _rows_list if hasattr(_r, 'dim') and _r.dim() == 4]
            if not _valid:
                return {'layer_occurrences': np.array([], dtype=float), 'near_duplicate_pairs': np.array([], dtype=float), 'num_layers': 0}
            _B = int(min(_r.shape[0] for _r in _valid))
            _layer_occ = np.zeros(_B, dtype=float)
            _pair_sum = np.zeros(_B, dtype=float)
            with torch.no_grad():
                for _r in _valid:
                    _r = _r.float()
                    for _b in range(_B):
                        _rr = _r[_b].mean(0)
                        _Nq = int(_rr.shape[0])
                        if _Nq <= 1:
                            continue
                        if _Nq > _max_q:
                            _idx = torch.linspace(0, _Nq - 1, _max_q, device=_rr.device).long()
                            _rr = _rr.index_select(0, _idx)
                            _Nq = int(_rr.shape[0])
                        _d = torch.cdist(_rr.unsqueeze(0), _rr.unsqueeze(0)).squeeze(0)
                        _upper = torch.triu(torch.ones((_Nq, _Nq), dtype=torch.bool, device=_d.device), diagonal=1)
                        _pairs = int(((_d < _tol) & _upper).sum().item())
                        if _pairs > 0:
                            _layer_occ[_b] += 1.0
                            _pair_sum[_b] += float(_pairs)
            return {'layer_occurrences': _layer_occ, 'near_duplicate_pairs': _pair_sum, 'num_layers': len(_valid)}

        _tol = CONFUSION_L2_THRESHOLD
        _div = _confusion_occurrences_by_image('divisive', _tol=_tol)
        _sub = _confusion_occurrences_by_image('subtractive', _tol=_tol)
        if _div['layer_occurrences'].size == 0 and _sub['layer_occurrences'].size == 0:
            print("  ⚠ Plot 26 produced no reconstructable attention rows.")
        else:
            _max_layers = int(max(np.max(_div['layer_occurrences']) if _div['layer_occurrences'].size else 0,
                                  np.max(_sub['layer_occurrences']) if _sub['layer_occurrences'].size else 0, 1))
            _bins = np.arange(-0.5, _max_layers + 1.5, 1.0)
            fig, axes = plt.subplots(1, 2, figsize=(14, 5))
            if _div['layer_occurrences'].size:
                axes[0].hist(_div['layer_occurrences'], bins=_bins, alpha=0.65, label='Divisive normalization')
            if _sub['layer_occurrences'].size:
                axes[0].hist(_sub['layer_occurrences'], bins=_bins, alpha=0.65, label='Subtractive InLine normalization')
            axes[0].set_title('Confusion Occurrence Distribution (InLine-DS)')
            axes[0].set_xlabel(f'Number of layers with confusion (L2 < {_tol:g})')
            axes[0].set_ylabel('Number of images'); axes[0].grid(alpha=0.3); axes[0].legend()
            _norms = ['Divisive', 'Subtractive']
            _occ_means = [float(np.mean(_div['layer_occurrences'])) if _div['layer_occurrences'].size else np.nan,
                          float(np.mean(_sub['layer_occurrences'])) if _sub['layer_occurrences'].size else np.nan]
            _pair_means = [float(np.mean(_div['near_duplicate_pairs'])) if _div['near_duplicate_pairs'].size else np.nan,
                           float(np.mean(_sub['near_duplicate_pairs'])) if _sub['near_duplicate_pairs'].size else np.nan]
            _x = np.arange(len(_norms)); _w = 0.36
            axes[1].bar(_x - _w/2, _occ_means, _w, label='Layer occurrences / image')
            axes[1].bar(_x + _w/2, _pair_means, _w, label='Near-duplicate pairs / image')
            axes[1].set_title('Mean Confusion Counts (InLine-DS)'); axes[1].set_ylabel('Mean count')
            axes[1].set_xticks(_x); axes[1].set_xticklabels(_norms); axes[1].grid(axis='y', alpha=0.3); axes[1].legend()
            _bar_value_labels(axes[1], _x - _w/2, _occ_means, _fmt='{:.2f}')
            _bar_value_labels(axes[1], _x + _w/2, _pair_means, _fmt='{:.2f}')
            _savefig('26_confusion_count_divisive_vs_subtractive.png',
                     'Confusion Count: Divisive vs Subtractive', 'paper_diagnostics',
                     'Attached-style injectivity plot showing the distribution of layer-level '
                     'confusion occurrences per image and the mean number of confusion events '
                     'for divisive and subtractive InLine normalization.')
            try:
                _rows_26 = []
                for _norm_name, _pack in [('divisive', _div), ('subtractive', _sub)]:
                    for _i in range(len(_pack['layer_occurrences'])):
                        _rows_26.append({'normalization': _norm_name, 'image_index': _i,
                                         'layers_with_confusion': float(_pack['layer_occurrences'][_i]),
                                         'near_duplicate_pairs': float(_pack['near_duplicate_pairs'][_i]),
                                         'tol_l2': _tol, 'num_layers_checked': int(_pack['num_layers'])})
                _csv26 = os.path.join(VIZ_DIR, '26_confusion_count_divisive_vs_subtractive.csv')
                pd.DataFrame(_rows_26).to_csv(_csv26, index=False)
                _register_viz('26_confusion_count_divisive_vs_subtractive.csv',
                              'Confusion Count Attached-Style CSV', 'paper_diagnostics',
                              'Per-image layer occurrences and near-duplicate query-row pairs for divisive and subtractive normalization.')
            except Exception as _csv_e:
                print(f"  ⚠ Plot 26 CSV skipped: {_csv_e}")
except Exception as _e:
    print(f"  ⚠ Plot 26 failed: {_e}")


print("\n[Plot 27] Accuracy and accuracy drop under local/random masking …")
try:
    if not _model_loaded or _val_loader is None or not _ds_attn_modules(_viz_model):
        print("  ⚠ Plot 27 skipped: no model / loader / InLine-DS attention modules.")
    else:
        _P27_BATCHES = 4
        _use_existing_p3 = ('df_p3' in globals() and isinstance(df_p3, pd.DataFrame)
                            and not df_p3.empty and {'condition', 'top1_accuracy'}.issubset(df_p3.columns))
        if _use_existing_p3:
            _p3_lookup = {str(_r['condition']): float(_r['top1_accuracy']) for _, _r in df_p3.iterrows()}
            _acc_none = _p3_lookup.get('baseline', np.nan)
            _acc_local = _p3_lookup.get('mask_local_3x3', np.nan)
            _acc_random = _p3_lookup.get('mask_random_equal_count', np.nan)
        else:
            _acc_none = _eval_with_attention_mask(_viz_model, _val_loader, kind=None, max_batches=_P27_BATCHES)
            _acc_local = _eval_with_attention_mask(_viz_model, _val_loader, kind='local3x3', max_batches=_P27_BATCHES)
            _acc_random = _eval_with_attention_mask(_viz_model, _val_loader, kind='random', max_batches=_P27_BATCHES, seed=0)
        if any(math.isnan(float(_v)) for _v in [_acc_none, _acc_local, _acc_random]):
            print("  ⚠ Plot 27 skipped: at least one masking accuracy is NaN.")
        else:
            _drop_local = float(_acc_none - _acc_local)
            _drop_random = float(_acc_none - _acc_random)
            fig, axes = plt.subplots(1, 2, figsize=(13, 5))
            _acc_labels = ['none', 'local3x3_mask', 'random_equal_count_mask']
            _acc_vals = [float(_acc_none), float(_acc_local), float(_acc_random)]
            _x0 = np.arange(len(_acc_labels))
            axes[0].bar(_x0, _acc_vals)
            axes[0].set_title('Accuracy Under Attention Masking (InLine-DS)')
            axes[0].set_ylabel('Top-1 accuracy on diagnostic subset (%)')
            axes[0].set_xticks(_x0); axes[0].set_xticklabels(_acc_labels, rotation=18, ha='right'); axes[0].grid(axis='y', alpha=0.3)
            _bar_value_labels(axes[0], _x0, _acc_vals, _fmt='{:.2f}')
            _drop_labels = ['local3x3_mask', 'random_equal_count_mask']
            _drop_vals = [_drop_local, _drop_random]
            _x1 = np.arange(len(_drop_labels))
            axes[1].bar(_x1, _drop_vals)
            axes[1].set_title('Local vs Random Masking Accuracy Drop (InLine-DS)')
            axes[1].set_ylabel('Accuracy drop from no mask (%)')
            axes[1].set_xticks(_x1); axes[1].set_xticklabels(_drop_labels, rotation=18, ha='right'); axes[1].grid(axis='y', alpha=0.3)
            _bar_value_labels(axes[1], _x1, _drop_vals, _fmt='{:.2f}')
            _savefig('27_local_vs_random_attention_masking_accuracy_drop.png',
                     'Local vs Random Attention Masking Accuracy Drop', 'paper_diagnostics',
                     'Attached-style locality plot comparing unmasked accuracy, local 3x3 key masking, '
                     'and equal-count random key masking.  The masking code is rectangular-grid aware for PVT.')
            try:
                _csv27 = os.path.join(VIZ_DIR, '27_local_vs_random_attention_masking_accuracy_drop.csv')
                pd.DataFrame([
                    {'condition': 'none', 'top1_accuracy': float(_acc_none), 'accuracy_drop_from_no_mask': 0.0, 'batches': _P27_BATCHES},
                    {'condition': 'local3x3_mask', 'top1_accuracy': float(_acc_local), 'accuracy_drop_from_no_mask': _drop_local, 'batches': _P27_BATCHES},
                    {'condition': 'random_equal_count_mask', 'top1_accuracy': float(_acc_random), 'accuracy_drop_from_no_mask': _drop_random, 'batches': _P27_BATCHES},
                ]).to_csv(_csv27, index=False)
                _register_viz('27_local_vs_random_attention_masking_accuracy_drop.csv',
                              'Local vs Random Masking Attached-Style CSV', 'paper_diagnostics',
                              'Top-1 accuracy and accuracy drops for no mask, local 3x3 mask, and equal-count random mask.')
            except Exception as _csv_e:
                print(f"  ⚠ Plot 27 CSV skipped: {_csv_e}")
except Exception as _e:
    print(f"  ⚠ Plot 27 failed: {_e}")


print("\n[Plot 28] Learned kernel τ and η/floor distributions …")
try:
    _mods28 = _ds_attn_modules(_viz_model)
    if not _mods28:
        print("  ⚠ Plot 28 skipped: no InLine-DS attention modules.")
    else:
        import torch.nn.functional as _Fp28

        def _np_1d(_x):
            if _x is None:
                return None
            _arr = _x.detach().float().cpu().numpy() if hasattr(_x, 'detach') else np.asarray(_x)
            return np.asarray(_arr).reshape(-1)

        def _tau_for_plot(_ker):
            for _name in ['log_tau', 'tau_raw', 'raw_tau']:
                if hasattr(_ker, _name):
                    return _np_1d(_Fp28.softplus(getattr(_ker, _name)))
            for _name in ['tau', 'temperature', 'temperatures']:
                if hasattr(_ker, _name):
                    return _np_1d(getattr(_ker, _name))
            return _param28_by_role(_ker, 'tau')

        def _eta_for_plot(_ker):
            for _name in ['eta', 'floor', 'floor_eta', 'beta', 'eta_raw', 'raw_eta', 'floor_raw', 'log_eta']:
                if hasattr(_ker, _name):
                    return _np_1d(getattr(_ker, _name))
            return _param28_by_role(_ker, 'eta')

        def _param28_by_role(_ker, _kind):
            """Name-scan fallback so Plot 28 populates for any InLine-DS backbone."""
            _want = ('tau', 'temp') if _kind == 'tau' else ('eta', 'floor', 'beta')
            try:
                _named = list(_ker.named_parameters()) + list(_ker.named_buffers())
            except Exception:
                return None
            for _pname, _pt in _named:
                _low = _pname.lower()
                if _pt is not None and any(_w in _low for _w in _want):
                    _v = _pt.detach().float()
                    if 'log' in _low or 'raw' in _low:
                        _v = _Fp28.softplus(_v)
                    return _np_1d(_v)
            return None

        def _row28_idx(_arr, _h):
            if _arr is None or len(_arr) == 0:
                return np.nan
            return float(_arr[_h % len(_arr)])

        _rows28 = []
        for _li, (_nm, _m) in enumerate(_mods28):
            for _side, _ker in [('query', getattr(_m, 'kernel_q', None)), ('key', getattr(_m, 'kernel_k', None))]:
                if _ker is None:
                    continue
                _tau = _tau_for_plot(_ker)
                _eta = _eta_for_plot(_ker)
                _n = max(len(_tau) if _tau is not None else 0, len(_eta) if _eta is not None else 0,
                         int(getattr(_ker, 'num_heads', 0) or 0),
                         int(getattr(_m, 'num_heads', 0) or 0), 1)
                for _h in range(_n):
                    _rows28.append({'layer_index': _li, 'layer_name': _nm, 'side': _side, 'head': _h,
                                    'tau': _row28_idx(_tau, _h),
                                    'eta_floor': _row28_idx(_eta, _h)})
        _df28 = pd.DataFrame(_rows28)
        if _df28.empty:
            print("  ⚠ Plot 28 found no exposed τ or η/floor parameters.")
        else:
            fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))
            _tau_vals = _df28['tau'].dropna().values if 'tau' in _df28 else np.array([])
            _eta_vals = _df28['eta_floor'].dropna().values if 'eta_floor' in _df28 else np.array([])
            if _tau_vals.size:
                axes[0].hist(_tau_vals, bins=20, alpha=0.75)
                axes[0].axvline(float(np.mean(_tau_vals)), linestyle='--', linewidth=1.8, label=f"mean={float(np.mean(_tau_vals)):.3f}")
                axes[0].legend()
            axes[0].set_title('Distribution of τ (InLine-DS)'); axes[0].set_xlabel('Learned temperature τ'); axes[0].set_ylabel('Number of heads'); axes[0].grid(axis='y', alpha=0.3)
            if _eta_vals.size:
                axes[1].hist(_eta_vals, bins=20, alpha=0.75)
                axes[1].axvline(float(np.mean(_eta_vals)), linestyle='--', linewidth=1.8, label=f"mean={float(np.mean(_eta_vals)):.3f}")
                axes[1].legend()
            axes[1].set_title('Distribution of η / floor (InLine-DS)'); axes[1].set_xlabel('Learned floor η'); axes[1].set_ylabel('Number of heads'); axes[1].grid(axis='y', alpha=0.3)
            if _tau_vals.size:
                _d_tau = _df28.dropna(subset=['tau'])
                axes[2].scatter(_d_tau['layer_index'], _d_tau['tau'], s=18, alpha=0.75, label='τ')
            if _eta_vals.size:
                _d_eta = _df28.dropna(subset=['eta_floor'])
                axes[2].scatter(_d_eta['layer_index'], _d_eta['eta_floor'], s=18, alpha=0.75, label='η/floor')
            axes[2].set_title('Per-Head Kernel Parameters by Layer (InLine-DS)'); axes[2].set_xlabel('Layer index'); axes[2].set_ylabel('Parameter value'); axes[2].grid(alpha=0.3); axes[2].legend()
            _savefig('28_learned_kernel_tau_eta_distributions.png',
                     'Learned Kernel τ and η/floor Distributions', 'paper_diagnostics',
                     'Attached-style kernel-parameter plot showing τ distribution, η/floor distribution, '
                     'and per-head τ and η/floor values by layer after training.')
            try:
                _csv28 = os.path.join(VIZ_DIR, '28_learned_kernel_tau_eta_distributions.csv')
                _df28.to_csv(_csv28, index=False)
                _register_viz('28_learned_kernel_tau_eta_distributions.csv',
                              'Learned Kernel τ/η Attached-Style CSV', 'paper_diagnostics',
                              'Per-layer and per-head τ and raw η/floor values used in the attached-style distribution plot.')
            except Exception as _csv_e:
                print(f"  ⚠ Plot 28 CSV skipped: {_csv_e}")
except Exception as _e:
    print(f"  ⚠ Plot 28 failed: {_e}")

# ════════════════════════════════════════════════════════════════════════
# ATTRIBUTION PLOTS (Viz 11–24)
# ════════════════════════════════════════════════════════════════════════

def _attention_modules(model):
    mods = []
    if model is None:
        return mods
    for name, module in model.named_modules():
        if hasattr(module, 'qkv') and hasattr(module, 'num_heads'):
            mods.append((name, module))
    return mods


def _collect_attention_proxy(model, imgs, need_grad=False):
    """Collect attention-like rows for visualization using InLine-only attention.

    Prefers exact InLine-DS subtractive rows when the module exposes the MMSK
    kernels (reusing the adapter helpers defined earlier); otherwise falls back
    to a positive linear feature map + subtractive normalization. No softmax.
    """
    if model is None or imgs is None:
        return ([], []) if not need_grad else ([], [], None)
    attns, names, handles = [], [], []

    def _make_hook(layer_name):
        def _hook(module, inputs, output):
            x = inputs[0]
            if not (hasattr(x, 'shape') and x.dim() == 3):
                return
            b, n, c = x.shape
            h = int(getattr(module, 'num_heads', 1))
            hd = c // h
            try:
                if _is_ds_attn(module):
                    q, k, _v = _extract_qkv(module, inputs)
                    rows = _ds_rows_from_qk(module, q, k, normalization='subtractive', feature_map='mmsk')
                    a = _rows_to_entropy_prob(rows)
                else:
                    qkv = module.qkv(x).reshape(b, n, 3, h, hd).permute(2, 0, 3, 1, 4)
                    q, k = qkv[0], qkv[1]
                    phi_q = F.elu(q.float()) + 1.0
                    phi_k = F.elu(k.float()) + 1.0
                    sim = torch.matmul(phi_q, phi_k.transpose(-2, -1))
                    rows = sim - sim.mean(dim=-1, keepdim=True) + (1.0 / float(n))
                    rows = torch.clamp(rows, min=0.0)
                    a = rows / rows.sum(dim=-1, keepdim=True).clamp_min(1e-12)
                if need_grad:
                    a.retain_grad()
                    attns.append(a)
                else:
                    attns.append(a.detach().cpu())
                names.append(layer_name)
            except Exception as _e:
                print(f"  ⚠ Could not collect InLine attention rows for {layer_name}: {_e}")
        return _hook

    for n, m in _attention_modules(model):
        handles.append(m.register_forward_hook(_make_hook(n)))
    try:
        x = imgs.to(_device)
        if need_grad:
            model.zero_grad(set_to_none=True)
            logits = model(x)
            return attns, names, logits
        else:
            with torch.no_grad():
                _ = model(x)
    finally:
        for h in handles:
            h.remove()
    return attns, names


def _attention_rollout(attns, discard_ratio=0.0, head_fusion='mean'):
    """Standard rollout over proxy attentions. Returns [B,N,N] (square only)."""
    if not attns:
        return None
    result = None
    for a in attns:
        a = a.detach().to(_device) if isinstance(a, torch.Tensor) else torch.tensor(a, device=_device)
        if a.dim() != 4 or a.shape[-1] != a.shape[-2]:
            continue
        if head_fusion == 'max':
            fused = a.max(dim=1).values
        elif head_fusion == 'min':
            fused = a.min(dim=1).values
        else:
            fused = a.mean(dim=1)
        if discard_ratio > 0:
            flat = fused.view(fused.size(0), -1)
            _, idx = flat.topk(int(flat.size(1) * discard_ratio), dim=1, largest=False)
            flat.scatter_(1, idx, 0.0)
            fused = flat.view_as(fused)
        eye = torch.eye(fused.size(-1), device=fused.device).unsqueeze(0)
        fused = fused + eye
        fused = fused / fused.sum(dim=-1, keepdim=True).clamp_min(1e-12)
        result = fused if result is None else torch.bmm(fused, result)
    return result.detach().cpu() if result is not None else None


# Extract attention proxies + rollout once for the attention plots.
_attn_cpu, _attn_names, _rollout = [], [], None
if _model_loaded and _sample_imgs is not None:
    print("\n[10e] Extracting attention proxies (Code-One plots) …")
    try:
        _attn_cpu, _attn_names = _collect_attention_proxy(_viz_model, _sample_imgs[:8], need_grad=False)
        # Attention maps assume DeiT-style square CLS attention.
        if not IS_DEIT:
            _attn_cpu = _square_rows_only(_attn_cpu) if _attn_cpu else []
        _rollout = _attention_rollout(_attn_cpu, discard_ratio=0.0, head_fusion='mean')
        print(f"  ✓ Extracted {len(_attn_cpu)} layers of attention proxies")
    except Exception as _e:
        print(f"  ⚠ Attention extraction failed: {_e}")

if _attn_cpu:
    print("\n[Viz 11] Attention entropy heatmaps …")
    try:
        ent = []
        for a in _attn_cpu:
            a_np = _to_numpy(a)
            h_ent = -np.sum(a_np * np.log(a_np + 1e-12), axis=-1) / math.log(a_np.shape[-1])
            ent.append(h_ent.mean(axis=(0,2)))
        max_h = max(len(x) for x in ent)
        ent_mat = np.full((len(ent), max_h), np.nan)
        for i, row in enumerate(ent):
            ent_mat[i, :len(row)] = row
        fig, ax = plt.subplots(figsize=(max(8, max_h*0.8), max(5, len(ent)*0.45)))
        sns.heatmap(ent_mat, annot=True, fmt='.2f', cmap='mako', ax=ax, cbar_kws={'label':'Normalized entropy'})
        ax.set_xlabel('Head'); ax.set_ylabel('Layer'); ax.set_title('Attention Entropy Heatmap')
        _savefig('11_attention_entropy_heatmap.png', 'Attention Entropy Heatmaps', 'attention', 'Layer-by-head normalized attention entropy.')
    except Exception as _e:
        print(f'  ⚠ Attention entropy skipped: {_e}')

    print("\n[Viz 11b] CLS-to-patch attention mass by layer and head …")
    try:
        cls_rows = []
        for li, a in enumerate(_attn_cpu):
            a_np = _to_numpy(a)
            cls_focus = a_np[:, :, 0, 1:].sum(axis=-1).mean(axis=0)
            cls_rows.append(cls_focus)
        max_h = max(len(x) for x in cls_rows)
        cls_mat = np.full((len(cls_rows), max_h), np.nan)
        for i, row in enumerate(cls_rows):
            cls_mat[i, :len(row)] = row
        fig, ax = plt.subplots(figsize=(max(8, max_h*0.8), max(5, len(cls_rows)*0.45)))
        sns.heatmap(cls_mat, annot=True, fmt='.2f', cmap='viridis', ax=ax, cbar_kws={'label':'CLS-to-patch mass'})
        ax.set_xlabel('Head'); ax.set_ylabel('Layer')
        ax.set_title('CLS-to-Patch Attention Mass by Layer and Head')
        _savefig('11b_cls_to_patch_attention_mass_layer_head.png',
                 'CLS-to-Patch Attention Mass by Layer and Head', 'attention',
                 'Layer-by-head heatmap of the attention mass from CLS token to patch tokens.')
    except Exception as _e:
        print(f'  ⚠ CLS-to-patch mass heatmap skipped: {_e}')

    print("\n[Viz 12] Patch attention maps - last layer …")
    try:
        last_attn = _to_numpy(_attn_cpu[-1])
        img_count = min(8, last_attn.shape[0])
        fig, axes = plt.subplots(img_count, 3, figsize=(9, 3*img_count))
        if img_count == 1:
            axes = axes[None, :]
        for i in range(img_count):
            img = _unnormalize_img(_sample_imgs[i], _mean, _std)
            cls_patch = last_attn[i].mean(axis=0)[0]
            heat = _patch_map_from_cls(cls_patch)
            heat = (heat - heat.min()) / (heat.max() - heat.min() + 1e-12)
            axes[i,0].imshow(img); axes[i,0].set_title('Image'); axes[i,0].axis('off')
            axes[i,1].imshow(heat, cmap='inferno'); axes[i,1].set_title('Patch attention'); axes[i,1].axis('off')
            axes[i,2].imshow(img); axes[i,2].imshow(heat, cmap='inferno', alpha=0.55, extent=(0, img.shape[1], img.shape[0], 0)); axes[i,2].set_title('Overlay'); axes[i,2].axis('off')
        _savefig('12_patch_attention_maps_last_layer.png', 'Patch Attention Maps - Last Layer', 'attention', 'CLS-to-patch attention heatmaps from the final layer.')
    except Exception as _e:
        print(f'  ⚠ Patch attention maps skipped: {_e}')

    print("\n[Viz 12b] Mean head CLS patch attention …")
    try:
        last_attn = _to_numpy(_attn_cpu[-1])
        H = last_attn.shape[1]
        cols = min(4, H)
        rows = int(math.ceil(H / cols))
        fig, axes = plt.subplots(rows, cols, figsize=(3.2*cols, 3.0*rows))
        axes = np.asarray(axes).reshape(-1)
        for ax in axes:
            ax.axis('off')
        for hi in range(H):
            cls_patch = last_attn[:, hi, 0, :].mean(axis=0)
            heat = _patch_map_from_cls(cls_patch)
            heat = (heat - heat.min()) / (heat.max() - heat.min() + 1e-12)
            axes[hi].imshow(heat, cmap='inferno')
            axes[hi].set_title(f'Head {hi}')
            axes[hi].axis('off')
        fig.suptitle('Mean Head CLS Patch Attention')
        _savefig('12b_mean_head_cls_patch_attention.png', 'Mean Head CLS Patch Attention', 'attention',
                 'Mean CLS-to-patch attention heatmap for every head in the final attention layer.')
    except Exception as _e:
        print(f'  ⚠ Mean head CLS patch attention skipped: {_e}')

    print("\n[Viz 13] Attention Rollout: CLS Attribution …")
    try:
        roll = _to_numpy(_rollout)
        img_count = min(8, roll.shape[0])
        fig, axes = plt.subplots(img_count, 3, figsize=(9, 3*img_count))
        if img_count == 1: axes = axes[None, :]
        for i in range(img_count):
            img = _unnormalize_img(_sample_imgs[i], _mean, _std)
            heat = _patch_map_from_cls(roll[i, 0])
            heat = (heat - heat.min()) / (heat.max() - heat.min() + 1e-12)
            axes[i,0].imshow(img); axes[i,0].axis('off'); axes[i,0].set_title('Image')
            axes[i,1].imshow(heat, cmap='viridis'); axes[i,1].axis('off'); axes[i,1].set_title('Rollout')
            axes[i,2].imshow(img); axes[i,2].imshow(heat, cmap='viridis', alpha=0.55, extent=(0,img.shape[1],img.shape[0],0)); axes[i,2].axis('off'); axes[i,2].set_title('Overlay')
        _savefig('13_attention_rollout_cls_attribution.png', 'Attention Rollout: CLS Attribution', 'attention', 'Attention rollout maps accumulated across all transformer blocks.')
    except Exception as _e:
        print(f'  ⚠ Attention rollout skipped: {_e}')

    print("\n[Viz 14] Gradient-Weighted Attention Rollout …")
    try:
        x = _sample_imgs[:4].clone().to(_device)
        x.requires_grad_(True)
        _viz_model.zero_grad(set_to_none=True)
        logits = _viz_model(x)
        top = logits.max(dim=1).values.sum()
        top.backward()
        grad_importance = x.grad.detach().abs().mean(dim=1, keepdim=True)
        grad_importance = F.interpolate(grad_importance, size=(cfg['img_size'], cfg['img_size']), mode='bilinear', align_corners=False)
        roll = _to_numpy(_rollout[:4])
        fig, axes = plt.subplots(min(4, roll.shape[0]), 3, figsize=(9, 3*min(4, roll.shape[0])))
        if min(4, roll.shape[0]) == 1: axes = axes[None, :]
        for i in range(min(4, roll.shape[0])):
            img = _unnormalize_img(_sample_imgs[i], _mean, _std)
            heat = _patch_map_from_cls(roll[i,0])
            heat_t = torch.tensor(heat)[None,None].float()
            heat_t = F.interpolate(heat_t, size=(cfg['img_size'], cfg['img_size']), mode='bilinear', align_corners=False)[0,0]
            grad_t = grad_importance[i,0].detach().cpu()
            gmar = heat_t * grad_t
            gmar = (gmar - gmar.min()) / (gmar.max() - gmar.min() + 1e-12)
            axes[i,0].imshow(img); axes[i,0].axis('off'); axes[i,0].set_title('Image')
            axes[i,1].imshow(gmar, cmap='plasma'); axes[i,1].axis('off'); axes[i,1].set_title('GMAR proxy')
            axes[i,2].imshow(img); axes[i,2].imshow(gmar, cmap='plasma', alpha=0.55, extent=(0,img.shape[1],img.shape[0],0)); axes[i,2].axis('off'); axes[i,2].set_title('Overlay')
        _savefig('14_gradient_weighted_attention_rollout.png', 'Gradient-Weighted Attention Rollout', 'attribution', 'Gradient-weighted attention rollout proxy over input pixels.')
    except Exception as _e:
        print(f'  ⚠ GMAR skipped: {_e}')

    print("\n[Viz 15] Attention flow graphs …")
    try:
        G = nx.DiGraph()
        layer_head_scores = []
        for li, a in enumerate(_attn_cpu):
            a_np = _to_numpy(a)
            cls_focus = a_np[:, :, 0, 1:].mean(axis=(0,2))
            for hi, s in enumerate(cls_focus):
                node = f'L{li}H{hi}'
                G.add_node(node, layer=li, score=float(s))
                layer_head_scores.append((li, hi, float(s)))
        for li in range(len(_attn_cpu)-1):
            h1 = _to_numpy(_attn_cpu[li]).shape[1]
            h2 = _to_numpy(_attn_cpu[li+1]).shape[1]
            for hi in range(h1):
                for hj in range(h2):
                    w = (G.nodes[f'L{li}H{hi}']['score'] + G.nodes[f'L{li+1}H{hj}']['score']) / 2
                    if w >= np.percentile([x[2] for x in layer_head_scores], 65):
                        G.add_edge(f'L{li}H{hi}', f'L{li+1}H{hj}', weight=w)
        pos = {}
        for node, data in G.nodes(data=True):
            li = data['layer']; hi = int(node.split('H')[1])
            pos[node] = (li, -hi)
        fig, ax = plt.subplots(figsize=(max(12, len(_attn_cpu)*1.2), 7))
        weights = [G[u][v]['weight']*8 for u,v in G.edges]
        scores = [G.nodes[n]['score'] for n in G.nodes]
        nx.draw_networkx_edges(G, pos, ax=ax, width=weights, alpha=0.35, arrows=True, arrowsize=8)
        nx.draw_networkx_nodes(G, pos, node_size=220, node_color=scores, cmap='viridis', ax=ax)
        nx.draw_networkx_labels(G, pos, font_size=7, ax=ax)
        ax.set_title('Attention Flow Graph: CLS Patch Focus Across Layers/Heads')
        ax.axis('off')
        _savefig('15_attention_flow_graph_cls_patch_focus_layers_heads.png', 'Attention Flow Graph: CLS Patch Focus Across Layers/Heads', 'attention', 'Graph of high CLS-focus attention heads and their layer-to-layer flow.')
    except Exception as _e:
        print(f'  ⚠ Attention flow graph skipped: {_e}')

    print("\n[Viz 16] Modified parallel coordinates for multi-head attention diagnostics …")
    try:
        rows = []
        for li, a in enumerate(_attn_cpu):
            a_np = _to_numpy(a)
            B,H,N,_ = a_np.shape
            entropy_h = -np.sum(a_np * np.log(a_np + 1e-12), axis=-1).mean(axis=(0,2)) / math.log(N)
            cls_focus_h = a_np[:, :, 0, 1:].sum(axis=-1).mean(axis=0)
            prune_drop_h = (1.0 - entropy_h) * cls_focus_h
            for hi in range(H):
                rows.append({'head_id': f'L{li}H{hi}', 'layer': float(li), 'head': float(hi),
                             'entropy': float(entropy_h[hi]), 'cls_focus': float(cls_focus_h[hi]),
                             'prune_drop': float(prune_drop_h[hi])})
        df = pd.DataFrame(rows)
        dims = ['layer', 'head', 'entropy', 'cls_focus', 'prune_drop']
        df_norm = df.copy()
        for col in dims:
            mn, mx = df_norm[col].min(), df_norm[col].max()
            df_norm[col] = (df_norm[col] - mn) / (mx - mn + 1e-12)
        fig, ax = plt.subplots(figsize=(13, 7))
        x = np.arange(len(dims))
        for ridx, row in df_norm.iterrows():
            _label = row['head_id'] if ridx < 20 else None
            ax.plot(x, row[dims].values.astype(float), alpha=0.35, lw=1.2, label=_label)
        ax.set_xticks(x); ax.set_xticklabels(dims)
        ax.set_ylabel('Normalized value')
        ax.set_xlabel('layer, head, entropy, cls_focus, prune_drop')
        ax.set_title('Parallel Coordinates: Multi-Head Attention Diagnostics')
        if len(df_norm) <= 20:
            ax.legend(fontsize=7, bbox_to_anchor=(1.02, 1), loc='upper left', title='Head')
        else:
            ax.legend(fontsize=6, bbox_to_anchor=(1.02, 1), loc='upper left', title='First 20 heads')
        ax.grid(alpha=0.3)
        _savefig('16_parallel_coordinates_multihead_attention_diagnostics.png',
                 'Parallel Coordinates: Multi-Head Attention Diagnostics', 'attention',
                 'Normalized layer, head, entropy, CLS-focus, and prune_drop diagnostic values plotted against diagnostic dimension.')
        df.to_csv(os.path.join(VIZ_DIR, '16_parallel_coordinates_multihead_attention_diagnostics.csv'), index=False)
        _register_viz('16_parallel_coordinates_multihead_attention_diagnostics.csv',
                      _skd_title('Parallel Coordinates Multi-Head Attention Diagnostics CSV'),
                      'attention-data', 'Raw per-head attention diagnostics used in the modified parallel-coordinates plot.')
    except Exception as _e:
        print(f'  ⚠ Parallel coordinates skipped: {_e}')

    print("\n[Viz 17] Attention head redundancy / similarity matrix …")
    try:
        head_vecs, labels_h = [], []
        for li, a in enumerate(_attn_cpu):
            a_np = _to_numpy(a).mean(axis=0)
            for hi in range(a_np.shape[0]):
                head_vecs.append(a_np[hi].reshape(-1))
                labels_h.append(f'L{li}H{hi}')
        V = np.vstack(head_vecs)
        V = V - V.mean(axis=1, keepdims=True)
        V = V / (np.linalg.norm(V, axis=1, keepdims=True) + 1e-12)
        sim = V @ V.T
        fig, ax = plt.subplots(figsize=(12, 10))
        sns.heatmap(sim, cmap='coolwarm', center=0, xticklabels=labels_h, yticklabels=labels_h, ax=ax)
        ax.set_title('Attention Head Redundancy / Similarity Matrix')
        plt.xticks(rotation=90, fontsize=5); plt.yticks(rotation=0, fontsize=5)
        _savefig('17_attention_head_redundancy_similarity.png', 'Attention Head Redundancy / Similarity Matrix', 'attention', 'Cosine similarity between flattened attention maps of all heads.')
    except Exception as _e:
        print(f'  ⚠ Head redundancy skipped: {_e}')

    print("\n[Viz 18] Attention distance / locality heatmap …")
    try:
        locality = []
        for a in _attn_cpu:
            a_np = _to_numpy(a)
            B,H,N,_ = a_np.shape
            g = _grid_size_from_tokens(N, prefix=1)
            coords = np.array([(i//g, i%g) for i in range(g*g)], dtype=float)
            dist = np.sqrt(((coords[:,None,:] - coords[None,:,:])**2).sum(axis=-1))
            patch_attn = a_np[:, :, 1:1+g*g, 1:1+g*g]
            d = (patch_attn * dist[None,None]).sum(axis=(-1,-2)) / (patch_attn.sum(axis=(-1,-2)) + 1e-12)
            locality.append(d.mean(axis=0))
        max_h = max(len(x) for x in locality)
        loc_mat = np.full((len(locality), max_h), np.nan)
        for i, row in enumerate(locality): loc_mat[i, :len(row)] = row
        fig, ax = plt.subplots(figsize=(max(8, max_h*0.8), max(5, len(locality)*0.45)))
        sns.heatmap(loc_mat, annot=True, fmt='.2f', cmap='rocket_r', ax=ax, cbar_kws={'label':'Mean attention distance'})
        ax.set_title('Attention Distance / Locality Heatmap'); ax.set_xlabel('Head'); ax.set_ylabel('Layer')
        _savefig('18_attention_distance_locality_heatmap.png', 'Attention Distance / Locality Heatmap', 'attention', 'Average spatial distance covered by patch-to-patch attention for each layer/head.')
    except Exception as _e:
        print(f'  ⚠ Attention distance/locality skipped: {_e}')

# ── Intervention-based visualizations: pruning, layer attribution, dropout ──
if _model_loaded and _val_loader is not None:
    print("\n[Viz 19] Head pruning impact heatmap …")
    try:
        base_acc = _evaluate_subset(_viz_model, _val_loader, max_batches=3)
        impacts = []
        modules = _attention_modules(_viz_model)
        max_layers_for_speed = min(len(modules), 12)
        for li, (lname, mod) in enumerate(modules[:max_layers_for_speed]):
            H = int(mod.num_heads)
            C = int(getattr(mod, 'dim', getattr(mod, 'num_heads', 1) * 0)) or int(mod.num_heads)
            # Derive per-head width from qkv if dim attr is missing.
            try:
                C = int(mod.qkv.in_features)
            except Exception:
                pass
            hd = C // H
            for hi in range(H):
                def _zero_head_hook(module, inputs, output, hi=hi, hd=hd):
                    y = output.clone()
                    y[:, :, hi*hd:(hi+1)*hd] = 0
                    return y
                handle = mod.register_forward_hook(_zero_head_hook)
                acc = _evaluate_subset(_viz_model, _val_loader, max_batches=3)
                handle.remove()
                impacts.append({'layer': li, 'head': hi, 'impact': base_acc - acc, 'ablated_acc': acc})
        df_imp = pd.DataFrame(impacts)
        pivot = df_imp.pivot(index='layer', columns='head', values='impact')
        fig, axes = plt.subplots(1, 2, figsize=(16, 6))
        sns.heatmap(pivot, annot=True, fmt='.3f', cmap='vlag', center=0, ax=axes[0])
        axes[0].set_title('Head Pruning Impact Heatmap')
        df_imp['rank'] = np.arange(len(df_imp))
        df_sorted = df_imp.sort_values('impact', ascending=False).reset_index(drop=True)
        axes[1].plot(np.arange(len(df_sorted)), df_sorted['impact'], marker='o')
        axes[1].set_title('Head Pruning Impact Curve'); axes[1].set_xlabel('Heads ranked by impact'); axes[1].set_ylabel('Baseline acc - pruned acc')
        axes[1].grid(alpha=0.3)
        _savefig('19_head_pruning_impact_heatmap_and_curve.png', 'Head Pruning Impact Heatmap', 'intervention', 'Accuracy drop when individual attention heads are zeroed on a validation subset.')
        df_imp.to_csv(os.path.join(VIZ_DIR, '19_head_pruning_impact_values.csv'), index=False)
        _register_viz('19_head_pruning_impact_values.csv', 'Head Pruning Impact Values', 'intervention-data', 'CSV with layer/head pruning impact values.')
    except Exception as _e:
        print(f'  ⚠ Head pruning impact skipped: {_e}')

    print("\n[Viz 20] Layer-wise performance attribution …")
    try:
        base_acc = _evaluate_subset(_viz_model, _val_loader, max_batches=3)
        blocks = list(getattr(_viz_model, 'blocks', []))
        layer_rows = []
        for li, block in enumerate(blocks):
            def _bypass_hook(module, inputs, output):
                return inputs[0]
            handle = block.register_forward_hook(_bypass_hook)
            acc = _evaluate_subset(_viz_model, _val_loader, max_batches=3)
            handle.remove()
            layer_rows.append({'layer': li, 'impact': base_acc - acc, 'bypassed_acc': acc})
        df_layer = pd.DataFrame(layer_rows)
        if not df_layer.empty:
            fig, ax = plt.subplots(figsize=(12, 5))
            ax.bar(df_layer['layer'], df_layer['impact'])
            ax.plot(df_layer['layer'], df_layer['impact'], marker='o', lw=1)
            ax.axhline(0, color='black', lw=1)
            ax.set_title('Layer-wise Performance Attribution by Block Bypass')
            ax.set_xlabel('Transformer block'); ax.set_ylabel('Baseline acc - bypassed acc')
            ax.grid(axis='y', alpha=0.3)
            _savefig('20_layer_wise_performance_attribution.png', 'Layer-wise Performance Attribution', 'intervention', 'Validation subset accuracy drop when each transformer block is bypassed.')
            df_layer.to_csv(os.path.join(VIZ_DIR, '20_layer_wise_performance_attribution.csv'), index=False)
            _register_viz('20_layer_wise_performance_attribution.csv', 'Layer-wise Performance Attribution Values', 'intervention-data', 'CSV with per-layer bypass impact values.')
        else:
            print('  ⚠ Layer-wise attribution skipped: model has no .blocks list (non-DeiT backbone).')
    except Exception as _e:
        print(f'  ⚠ Layer-wise attribution skipped: {_e}')

    print("\n[Viz 21] Attention output dropout sensitivity …")
    try:
        modules = _attention_modules(_viz_model)
        original_ps = []
        for _, m in modules:
            p = getattr(getattr(m, 'attn_drop', None), 'p', None)
            original_ps.append(p)
        rates = [0.0, 0.05, 0.10, 0.20, 0.30, 0.50]
        rows = []
        for r in rates:
            for _, m in modules:
                if hasattr(m, 'attn_drop') and hasattr(m.attn_drop, 'p'):
                    m.attn_drop.p = r
            acc = _evaluate_subset(_viz_model, _val_loader, max_batches=3)
            rows.append({'attn_dropout': r, 'subset_acc': acc})
        for (_, m), p in zip(modules, original_ps):
            if p is not None and hasattr(m, 'attn_drop') and hasattr(m.attn_drop, 'p'):
                m.attn_drop.p = p
        df_drop = pd.DataFrame(rows)
        fig, ax = plt.subplots(figsize=(8, 5))
        ax.plot(df_drop['attn_dropout'], df_drop['subset_acc'], marker='o', lw=1.8)
        ax.set_title('Attention Output Dropout Sensitivity')
        ax.set_xlabel('Attention dropout p at inference intervention'); ax.set_ylabel('Validation subset accuracy')
        ax.grid(alpha=0.3)
        _savefig('21_attention_output_dropout_sensitivity.png', 'Attention Output Dropout Sensitivity', 'intervention', 'Accuracy response to changing attention dropout probability during evaluation subset testing.')
        df_drop.to_csv(os.path.join(VIZ_DIR, '21_attention_dropout_sensitivity.csv'), index=False)
        _register_viz('21_attention_dropout_sensitivity.csv', 'Attention Dropout Sensitivity Values', 'intervention-data', 'CSV with dropout sensitivity values.')
    except Exception as _e:
        print(f'  ⚠ Attention-dropout sensitivity skipped: {_e}')

# ── Additional calibration / accuracy plots ─────────────────────────────────
if _all_probs is not None:
    print("\n[Viz 22] Calibration reliability diagram …")
    try:
        conf = _all_probs.max(axis=1)
        correct = (_all_preds == _all_labels).astype(float)
        bins = _adaptive_conf_bins(conf, n_bins=10)
        bin_ids = np.digitize(conf, bins, right=False) - 1
        bin_ids = np.clip(bin_ids, 0, len(bins) - 2)
        bin_centers = (bins[:-1] + bins[1:]) / 2
        bin_widths = np.diff(bins) * 0.9
        bin_conf, bin_acc, bin_count = [], [], []
        for b in range(len(bins)-1):
            m = bin_ids == b
            bin_conf.append(conf[m].mean() if m.any() else np.nan)
            bin_acc.append(correct[m].mean() if m.any() else np.nan)
            bin_count.append(int(m.sum()))
        fig, axes = plt.subplots(1, 2, figsize=(13, 5))
        axes[0].plot([bins[0], bins[-1]], [bins[0], bins[-1]], '--', color='gray')
        axes[0].bar(bin_centers, np.nan_to_num(bin_acc), width=bin_widths, alpha=0.8, label='Accuracy')
        axes[0].plot(bin_centers, bin_conf, marker='o', label='Mean confidence')
        axes[0].set_xlim(float(bins[0]), float(bins[-1])); axes[0].set_ylim(0, 1.05)
        axes[0].set_title('Reliability Diagram - Accuracy vs Confidence'); axes[0].set_xlabel('Confidence bin'); axes[0].set_ylabel('Accuracy / Confidence'); axes[0].legend()
        axes[1].bar(bin_centers, bin_count, width=bin_widths)
        axes[1].set_xlim(float(bins[0]), float(bins[-1]))
        axes[1].set_title('Confidence Histogram'); axes[1].set_xlabel('Confidence bin'); axes[1].set_ylabel('Samples')
        _savefig('22_reliability_diagram_accuracy_vs_confidence.png', 'Reliability Diagram - Accuracy vs Confidence', 'calibration', 'Reliability diagram and confidence histogram.')
    except Exception as _e:
        print(f'  ⚠ Calibration plot skipped: {_e}')

    print("\n[Viz 23] Top-k accuracy curve …")
    try:
        ks = list(range(1, min(10, NUM_CLASSES) + 1))
        topk = []
        for k in ks:
            topk.append(top_k_accuracy_score(_all_labels, _all_probs, k=k, labels=np.arange(NUM_CLASSES)))
        fig, ax = plt.subplots(figsize=(8, 5))
        ax.plot(ks, topk, marker='o')
        ax.set_title('Top-k Accuracy Curve'); ax.set_xlabel('k'); ax.set_ylabel('Top-k accuracy'); ax.set_ylim(0, 1.02); ax.grid(alpha=0.3)
        _savefig('23_topk_accuracy_curve.png', 'Top-k Accuracy Curve', 'classification', 'Accuracy as k increases from top-1 to top-10/classes.')
    except Exception as _e:
        print(f'  ⚠ Top-k curve skipped: {_e}')

# ── Viz 24: Full visualization manifest (JSON/CSV/TXT) ─────────────
print("\n[Viz 24] Visualization manifest …")
try:
    manifest_json = os.path.join(VIZ_DIR, '24_visualization_manifest.json')
    manifest_csv = os.path.join(VIZ_DIR, '24_visualization_manifest.csv')
    summary_txt = os.path.join(VIZ_DIR, '24_visualization_manifest.txt')
    manifest_payload = {
        'dataset': DATASET, 'model_type': MODEL_TYPE, 'num_classes': NUM_CLASSES,
        'checkpoint': _ckpt_path, 'visualization_dir': VIZ_DIR,
        'created_utc': time.strftime('%Y-%m-%d %H:%M:%S', time.gmtime()),
        'items': VIZ_MANIFEST,
    }
    with open(manifest_json, 'w') as f:
        json.dump(manifest_payload, f, indent=2)
    pd.DataFrame(VIZ_MANIFEST).to_csv(manifest_csv, index=False)
    with open(summary_txt, 'w') as f:
        f.write(f"Visualization manifest for {MODEL_TYPE} on {DATASET}\n")
        f.write(f"Checkpoint: {_ckpt_path}\n")
        f.write(f"Directory: {VIZ_DIR}\n\n")
        for i, item in enumerate(VIZ_MANIFEST, 1):
            f.write(f"{i:02d}. {item['title']}\n")
            f.write(f"    file: {item['file']}\n")
            f.write(f"    category: {item['category']}\n")
            f.write(f"    description: {item['description']}\n\n")
    _register_viz('24_visualization_manifest.json', 'Visualization Manifest JSON', 'manifest', 'Machine-readable manifest of all generated visualizations.')
    _register_viz('24_visualization_manifest.csv', 'Visualization Manifest CSV', 'manifest', 'Tabular manifest of all generated visualizations.')
    _register_viz('24_visualization_manifest.txt', 'Visualization Manifest TXT', 'manifest', 'Readable manifest summary of all generated visualizations.')
    manifest_payload['items'] = VIZ_MANIFEST
    with open(manifest_json, 'w') as f:
        json.dump(manifest_payload, f, indent=2)
    pd.DataFrame(VIZ_MANIFEST).to_csv(manifest_csv, index=False)
    print(f"  ✓ manifest JSON → {manifest_json}")
    print(f"  ✓ manifest CSV  → {manifest_csv}")
    print(f"  ✓ manifest TXT  → {summary_txt}")
except Exception as _e:
    print(f"  ⚠ Manifest writing failed: {_e}")

# ── Visualization manifest summary ────────────────────────────────
try:
    print("\n" + "="*72)
    print(f"Saved {len(VIZ_MANIFEST)} visualisations to: {VIZ_DIR}")
    print("="*72)
    _man_csv = os.path.join(VIZ_DIR, 'visualisation_manifest.csv')
    with open(_man_csv, 'w', newline='') as _f:
        _w = csv.DictWriter(_f, fieldnames=['filename', 'title', 'category', 'description', 'path'])
        _w.writeheader()
        for _row in VIZ_MANIFEST:
            _w.writerow(_row)
    print(f"  manifest written: {_man_csv}")
    _by_cat = {}
    for _row in VIZ_MANIFEST:
        _by_cat.setdefault(_row['category'], 0)
        _by_cat[_row['category']] += 1
    for _c, _n in sorted(_by_cat.items()):
        print(f"    {_c:>16}: {_n}")
except Exception as _e:
    print(f"  ⚠ Manifest summary failed: {_e}")


############### 11. Quick Reference ################
# A short operational cheat-sheet printed at the end of a run.  Covers all four
# softmax-free InLine-DS backbones supported by this notebook.
print("""
================================================================
InLine^D-S Colab — Quick Reference
================================================================
Supported backbones (all softmax-free, attn_type forced to 'DDDD'):
  DeiT  : inline_deit_ds_tiny | _small | _base        (flat ViT, CLS token)
  Swin  : inline_swin_ds_tiny | _small | _base        (shifted windows)
  PVT   : inline_pvt_ds_tiny  | _small | _medium | _large  (spatial reduction)
  CSWin : inline_cswin_ds_tiny | _small | _base       (cross-shaped windows)

Performance note (parity with "Code One"):
  - Section 1c PREFERS an uploaded build_ds.py (copied verbatim into models/)
    so DeiT patch-embed / positional-embedding construction matches Code One
    exactly. Only if no build_ds.py is uploaded does it fall back to the
    auto-generated unified builder. This is what restores Code-One accuracy on
    inline_deit_ds_tiny / TinyImageNet-200.

To switch model or dataset:
  1. Edit MODEL_TYPE and DATASET in Section 3.
  2. Re-run from Section 1b onward (the config in Section 4 is rebuilt
     automatically for the chosen dataset; ATTN_TYPE stays 'DDDD').

Required uploads (to  <Drive>/InLineDAttention/inlineds_models/):
  - the backbone file for your MODEL_TYPE, e.g. inline_cswin_ds.py
  - inline_deit_ds.py AND build_ds.py for Code-One-parity DeiT runs
  (if build_ds.py is omitted, the notebook generates a unified builder.)

Softmax guarantee:
  - The optional dot-product 'use_residual_gate' branch is deleted from every
    copied DS file (Section 1b).
  - attn_type='DDDD' is asserted to contain no 'S', so the softmax-only
    WindowAttention / LePEAttention side classes are never instantiated.
  - Section 7 traps torch.softmax / F.softmax during a forward pass and reports
    zero calls.

Visualisations (Section 10) — UNION of both notebooks:
  Viz 1–10  training curves, ROC/PR/F1, calibration, confusion, PCA, grids
  P1–P4     entropy-by-depth, confusion/injectivity, locality masking, kernel τ/η
  Extra 1–4 effective tokens, negative mass, entropy heatmap, CLS→patch (DeiT)
  26–28     attached-style confusion, masking, and kernel-parameter plots
  Viz 11–24 (from Code One) attention entropy/CLS-mass heatmaps, patch-attention
            maps, rollout, gradient-weighted rollout, attention flow graph,
            parallel coordinates, head redundancy, locality heatmap, head
            pruning, layer attribution, dropout sensitivity, reliability
            diagram, top-k accuracy, and the full JSON/CSV/TXT manifest.
            (DeiT-centric attention plots skip gracefully on Swin/PVT/CSWin.)
  All PNGs + CSV manifests are written under:
    <Drive>/InLineDAttention/output/visualisations/<dataset>_<model>/
================================================================
""")
print("Notebook finished.")


# End